# SALGINTR — Supplement S3, Python analysis notebook

**Project.** SALGINTR is a physician-based digital participatory influenza-like-illness
(ILI) surveillance system in Türkiye. This notebook holds the executed Python code for
every analysis in Supplement S3 whose estimator is not a survival or mixed-effects
model; those are estimated in the companion R notebook, `SALGINTR_S3_R.ipynb`.
Season 2025/26, ISO weeks 2025-W40 to 2026-W20 — 33 weeks.

**The dataset.** The notebook reads
`SALGINTR_Unified_Dataset_20260809.xlsx` **from the same folder as this notebook**.
Download the notebook and the workbook into one folder and the notebook runs unchanged.

```python
DATA = "SALGINTR_Unified_Dataset_20260809.xlsx"   # same folder as this notebook
```

**That single line is the only change a later release requires.** When a new dated
version of the workbook supersedes this one, edit `DATA` in the first code cell and
re-run; nothing else in the notebook refers to a file path.

**No figure is produced anywhere in this notebook.** Every analysis returns numeric
output — counts, tables, model summaries, printed statistics. `matplotlib` is not
imported and no plotting call appears in any cell. Where the thesis carries a figure
derived from an analysis, the figure exists in the thesis; its plotting code is
deliberately absent here.

**Structure.** One section per S3 group. Within each group, each analysis appears as a
numbered pair of cells: a markdown cell carrying the analysis identifier, its title and
its **English pseudocode**, then the code cell that implements it, with its executed
output embedded. Cell numbers are stated in the markdown headers so the coverage map can
point at them.

**Conventions that hold throughout.**

- Weekly incidence is computed on the **at-risk** denominator. The stored columns
  `ili_rate_per1000_pw` and `A_rate_per1000_pw` follow the filed person-week denominator
  and are never read; the rate is recomputed as
  `1000 * n_new_episodes / pw_at_risk_new`.
- The panel structure the workbook does not carry as columns — gap indicators, sequence
  numbers, forward-filled episode identifiers — is **derived here from the person-week
  sheet**, not read from any external file.
- Self-reported annual influenza-like-illness frequency is used in **four** ordered
  levels throughout.
- The six physicians who declined the health-condition question stay **missing**. They
  are not imputed, which is what fixes the individual-model sample at 242 and the
  reporting-propensity sample at 298.
- Adjustment sets are **pre-specified on epidemiological grounds**. No stepwise or
  data-driven variable selection occurs anywhere.
- Inverse-probability-of-reporting weighting is a **sensitivity analysis**. The estimates
  of record are unweighted.
- Moving Epidemic Method thresholds are **fixed a-priori inputs** estimated from ten
  historical sentinel seasons this release does not carry. They are hard-coded and are
  not derived from the season under analysis.
- Occupational analyses fit **main effects only**; no interaction term is estimable in
  the 110-physician subgroup.


## S3 Statistical methods and code — Python track

### Cell 1 — A00: Data access and shared derivations

**Pseudocode.**

1. Name the workbook. DATA is the only line to change when a later dated release
   supersedes this one; the workbook must sit in the same folder as this notebook.
2. Read the five analysis sheets: the physician registry, the person-week panel,
   the cleaned episode file, the weekly two-system file and the recoded
   individual-level file.
3. Sort the person-week panel by physician and week index, then derive the panel
   structure the notebook needs and the workbook does not carry as columns: an
   indicator for a gap immediately before each filed week, the sequence number of
   each filed week within a physician's own record, and a forward-filled episode
   identifier that carries an episode label across its continuation week.
4. Build the weekly two-system series. Weekly incidence is computed on the
   at-risk denominator throughout; the stored rate columns follow the filed
   person-week denominator and are not read.
5. Fix the ISO week calendar and the epidemic-phase windows, and hard-code the
   Moving Epidemic Method thresholds, which are a-priori inputs estimated from
   ten historical sentinel seasons that this release does not carry.
6. Define the shared helpers: causal three-week moving average, exact binomial
   interval, standardized mean difference, and the Efron bootstrap optimism
   correction that refits the model inside every replicate. A small collector holds
   the quantity each block reproduces, so the verification table can be assembled
   from the notebook's own run.


In [1]:
import numpy as np, pandas as pd
from scipy import stats
import statsmodels.api as sm, statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from statsmodels.stats.proportion import proportion_confint
from sklearn.metrics import roc_auc_score, cohen_kappa_score
from lifelines import CoxTimeVaryingFitter

# Collector for the quantities each block reproduces; the verification table is built
# from it. Every block calls record(...) at the end.
REPRODUCED = {}
def record(analysis_id, **values):
    REPRODUCED.setdefault(analysis_id, {}).update(values)

DATA = "SALGINTR_Unified_Dataset_20260809.xlsx"   # same folder as this notebook

phys      = pd.read_excel(DATA, sheet_name="01_Physicians")        # 304 x 64
personwk  = pd.read_excel(DATA, sheet_name="02_PersonWeeks")       # 4,729 x 117
episodes  = pd.read_excel(DATA, sheet_name="03_Episodes")          # 497 x 27
weeklysys = pd.read_excel(DATA, sheet_name="04_WeeklySystem")      # 33 x 26
recoded   = pd.read_excel(DATA, sheet_name="05_RecodedIndividual") # 248 x 29

# --- panel structure derived from 02_PersonWeeks, not from any external file
pw = personwk.sort_values(["participant_id", "week_idx"]).reset_index(drop=True)
pw["gap_before_derived"] = (
    pw.groupby("participant_id")["week_idx"].diff().gt(1).fillna(False).astype(int))
pw["obs_seq_derived"] = pw.groupby("participant_id")["week_idx"].rank(method="first").astype(int)
pw["episode_id_filled"] = pw.groupby("participant_id")["episode_id"].ffill()
pw["is_first_week"] = pw.groupby("participant_id")["week_idx"].transform("min").eq(pw.week_idx)

# --- weekly two-system series, at-risk denominator for incidence
wk = weeklysys.sort_values("week_idx").reset_index(drop=True)
wk["ili_rate_per1000_at_risk"] = 1000 * wk.n_new_episodes / wk.pw_at_risk_new
wk["A_rate_per1000_at_risk"]   = 1000 * wk.n_new_A        / wk.pw_at_risk_new
part_share = 100 * wk.n_new_A / wk.n_new_episodes          # participatory positivity, %
sent_share = wk.sb_positivity_pct                          # sentinel positivity, %
WEEKS = wk.week.values
N_WEEKS = len(wk)

# --- ISO calendar and epidemic-phase windows
_iso = wk.week.str.extract(r"(\d{4})-W(\d+)").astype(int)
ISO_YEAR, ISO_WEEK = _iso[0], _iso[1]
PRE   = ((ISO_YEAR == 2025) & ISO_WEEK.between(40, 49)).values
EPI   = (((ISO_YEAR == 2025) & (ISO_WEEK >= 50)) | ((ISO_YEAR == 2026) & (ISO_WEEK <= 6))).values
POST  = ((ISO_YEAR == 2026) & ISO_WEEK.between(7, 20)).values
EARLY = (PRE | EPI)                       # early-warning window of record: the
                                          # pre-epidemic and epidemic phases together,
                                          # 2025-W40 to 2026-W06, 19 weeks
EARLY_2025 = ((ISO_YEAR == 2025) & ISO_WEEK.between(40, 52)).values
                                          # narrower early-warning window: the 2025
                                          # calendar portion, 2025-W40 to 2025-W52,
                                          # 13 weeks. Scored alongside and labelled;
                                          # it carries fewer sentinel alarms and the
                                          # two windows are not interchangeable.
SEASON = np.ones(N_WEEKS, bool)

# --- MEM thresholds: fixed a-priori inputs from ten historical sentinel seasons
# that this data release does not carry. They are not derived here.
MEM_ONSET = 10.59                                  # % combined positivity
MEM_INTENSITY = (22.04, 42.56, 60.92, 71.38)       # % medium/high/very high/extraordinary

# --- shared helpers
def cma3(s):
    """Causal three-week moving average, weeks t-2 to t, partial windows retained."""
    return pd.Series(s).rolling(3, min_periods=1).mean()

def exact_ci(k, n):
    return proportion_confint(k, n, method="beta")

def smd(a, b, binary):
    a, b = pd.Series(a).dropna(), pd.Series(b).dropna()
    if binary:
        p1, p2 = a.mean(), b.mean()
        sd = np.sqrt((p1 * (1 - p1) + p2 * (1 - p2)) / 2)
        return (p1 - p2) / sd if sd > 0 else 0.0
    sd = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return (a.mean() - b.mean()) / sd

def efron_optimism(formula, data, ycol, B=1000, seed=20260707):
    """Efron bootstrap optimism correction for the AUC. The model is refitted in
    every replicate; optimism is the mean of (bootstrap AUC - original-sample AUC)."""
    rng = np.random.default_rng(seed)
    apparent = roc_auc_score(data[ycol], smf.logit(formula, data=data).fit(disp=0).predict(data))
    diffs, n = [], len(data)
    for _ in range(B):
        db = data.iloc[rng.integers(0, n, n)]
        if db[ycol].nunique() < 2:
            continue
        try:
            fb = smf.logit(formula, data=db).fit(disp=0)
        except Exception:
            continue
        diffs.append(roc_auc_score(db[ycol], fb.predict(db))
                     - roc_auc_score(data[ycol], fb.predict(data)))
    optimism = float(np.mean(diffs))
    return apparent, optimism, apparent - optimism, len(diffs)

print(f"registry {phys.shape}  panel {pw.shape}  episodes {episodes.shape} "
      f" weekly {wk.shape}  recoded {recoded.shape}")
print(f"phase weeks: pre-epidemic {PRE.sum()}, epidemic {EPI.sum()}, post-epidemic {POST.sum()}, "
      f"early-warning window {EARLY.sum()}, 2025 calendar portion {EARLY_2025.sum()}")
record("A00", n_rows_panel=len(pw), n_weeks=N_WEEKS)

registry (304, 64)  panel (4729, 121)  episodes (497, 27)  weekly (33, 28)  recoded (248, 29)
phase weeks: pre-epidemic 10, epidemic 9, post-epidemic 14, early-warning window 19, 2025 calendar portion 13


## S3.1 Dataset and descriptive analysis

### Cell 2 — A01: Cohort accrual and follow-up

**Pseudocode.**

1. Count registered physicians in the registry sheet.
2. Split them on whether at least one weekly report was filed: reporters form the
   analysis cohort, the remainder are never-reporters.
3. From the person-week panel count filed person-weeks, person-weeks at risk of a
   new episode, symptomatic person-weeks and incident episodes.
4. Continuation weeks are symptomatic weeks that are not incident onsets; they
   are the weeks removed from the at-risk denominator.
5. Report each count with its percentage of the relevant total.


In [2]:
n_reg = len(phys)
n_rep = int(phys.reported.sum())
n_never = int((phys.reported == 0).sum())
n_pw = len(pw)
n_at_risk = int(pw.at_risk_new_episode.sum())
n_sym = int(pw.symptomatic.sum())
n_new = int(pw.new_episode.sum())
n_cont = n_sym - n_new

print(f"registered physicians                  {n_reg}")
print(f"reporting (analysis) cohort            {n_rep} ({100*n_rep/n_reg:.1f}%)")
print(f"never-reporters                        {n_never} ({100*n_never/n_reg:.1f}%)")
print(f"person-weeks of follow-up              {n_pw:,}")
print(f"person-weeks at risk of a new episode  {n_at_risk:,}")
print(f"continuation (not at-risk) weeks       {n_cont}")
print(f"symptomatic (ILI) person-weeks         {n_sym}")
print(f"cleaned incident ILI episodes          {n_new}")
assert n_at_risk + n_cont == n_pw
record("A01", registered=n_reg, reporters=n_rep, never=n_never, person_weeks=n_pw,
       at_risk=n_at_risk, continuation=n_cont, symptomatic=n_sym, episodes=n_new)

registered physicians                  304
reporting (analysis) cohort            248 (81.6%)
never-reporters                        56 (18.4%)
person-weeks of follow-up              4,729
person-weeks at risk of a new episode  4,626
continuation (not at-risk) weeks       103
symptomatic (ILI) person-weeks         600
cleaned incident ILI episodes          497


### Cell 3 — A02: Weekly participation series and plateau

**Pseudocode.**

1. Take the weekly count of physicians filing any report, in ISO week order.
2. The enrolment ramp is the first eight weeks of that series.
3. The plateau is week nine onward; report its median and range, and separately
   the median over all thirty-three weeks so the two are not conflated.
4. Locate the peak week and the closing week.
5. Fit an ordinary least-squares line to the reporter count from the peak week to
   the end of the season and report its slope in reporters per week.


In [3]:
series = wk.n_reporters.values
ramp, plateau = series[:8], series[8:]
peak_i = int(np.argmax(series))

sl = np.polyfit(wk.week_idx.values[peak_i:], series[peak_i:], 1)[0]
print("weekly reporters:", ", ".join(str(x) for x in series))
print(f"opening week                      {series[0]}")
print(f"enrolment ramp, first eight weeks  {', '.join(str(x) for x in ramp)}")
print(f"plateau (week 9 onward)            median {np.median(plateau):.0f}, "
      f"range {plateau.min()}-{plateau.max()}, n = {len(plateau)}")
print(f"median over all {len(series)} weeks          {np.median(series):.0f}")
print(f"peak                               {series[peak_i]} in {WEEKS[peak_i]}")
print(f"closing week                       {series[-1]} in {WEEKS[-1]}")
print(f"post-peak trend                    {sl:+.3f} reporters per week over "
      f"{WEEKS[peak_i]} to {WEEKS[-1]}")
record("A02", opening=int(series[0]), plateau_median=float(np.median(plateau)),
       plateau_min=int(plateau.min()), plateau_max=int(plateau.max()),
       all_week_median=float(np.median(series)), peak=int(series[peak_i]),
       peak_week=str(WEEKS[peak_i]), closing=int(series[-1]), post_peak_slope=float(sl))

weekly reporters: 44, 72, 83, 84, 89, 104, 120, 133, 159, 152, 148, 147, 149, 146, 150, 150, 165, 163, 163, 163, 171, 165, 168, 168, 177, 171, 165, 164, 158, 164, 155, 162, 157
opening week                      44
enrolment ramp, first eight weeks  44, 72, 83, 84, 89, 104, 120, 133
plateau (week 9 onward)            median 163, range 146-177, n = 25
median over all 33 weeks          157
peak                               177 in 2026-W12
closing week                       157 in 2026-W20
post-peak trend                    -2.117 reporters per week over 2026-W12 to 2026-W20


### Cell 4 — A03: Reporting-frequency distribution and grid fill

**Pseudocode.**

1. For each reporting physician take the number of weeks reported.
2. Report its median, mean and range, and the count reporting all thirty-three
   weeks and twenty-five to thirty-two weeks.
3. Realised grid fill is filed person-weeks divided by a complete grid. Compute it
   twice: against the reporter grid (reporters x weeks) and against the registry
   grid (all registrants x weeks).


In [4]:
rep = phys[phys.reported == 1]
nwr = rep.n_weeks_reported
print(f"weeks reported per physician: median {nwr.median():.0f}, mean {nwr.mean():.3f}, "
      f"range {nwr.min()}-{nwr.max()}")
for lo, hi, lab in [(33, 33, "all 33 weeks"), (25, 32, "25-32 weeks")]:
    m = nwr.between(lo, hi)
    print(f"  physicians reporting {lab:12s} {int(m.sum())} ({100*m.mean():.1f}%)")
for lab, denom in [("reporter grid", len(rep) * N_WEEKS), ("registry grid", len(phys) * N_WEEKS)]:
    print(f"realised grid fill, {lab:14s} {len(pw):,} of {denom:,} ({100*len(pw)/denom:.1f}%)")
record("A03", median_weeks=float(nwr.median()), mean_weeks=float(nwr.mean()),
       all33=int((nwr == 33).sum()), fill_reporter=100*len(pw)/(len(rep)*N_WEEKS),
       fill_registry=100*len(pw)/(len(phys)*N_WEEKS))

weeks reported per physician: median 20, mean 19.069, range 1-33
  physicians reporting all 33 weeks 15 (6.0%)
  physicians reporting 25-32 weeks  86 (34.7%)
realised grid fill, reporter grid  4,729 of 8,184 (57.8%)
realised grid fill, registry grid  4,729 of 10,032 (47.1%)


### Cell 5 — A04: Reporting persistence and engagement strata membership

**Pseudocode.**

1. Classify each reporting physician by the number of weeks reported into three
   engagement strata using fixed cut-points: least engaged up to five weeks,
   moderately engaged six to twenty-three weeks, most engaged twenty-four or more.
2. Report stratum sizes, and also the count reaching twenty-four and twenty-five
   weeks, since both appear as persistence descriptors.
3. Attach each stratum's at-risk person-weeks and episode count, which the
   stratum-specific incidence block then uses.


In [5]:
rep = phys[phys.reported == 1].copy()
CUTS = [0, 5, 23, 33]                       # fixed cut-points, not data-driven
rep["stratum"] = pd.cut(rep.n_weeks_reported, bins=CUTS,
                        labels=["least engaged", "moderately engaged", "most engaged"])
per_phys = pw.groupby("participant_id").agg(at_risk=("at_risk_new_episode", "sum"),
                                            episodes=("new_episode", "sum"))
S = rep.set_index("participant_id").join(per_phys)
tab = S.groupby("stratum", observed=True).agg(physicians=("stratum", "size"),
                                              at_risk=("at_risk", "sum"),
                                              episodes=("episodes", "sum"))
print(tab.to_string())
print(f"\nphysicians reporting 24 or more weeks  {int((rep.n_weeks_reported>=24).sum())} "
      f"({100*(rep.n_weeks_reported>=24).mean():.1f}%)")
print(f"physicians reporting 25 or more weeks  {int((rep.n_weeks_reported>=25).sum())} "
      f"({100*(rep.n_weeks_reported>=25).mean():.1f}%)")
record("A04", n_most=int(tab.physicians["most engaged"]),
       n_moderate=int(tab.physicians["moderately engaged"]),
       n_least=int(tab.physicians["least engaged"]),
       ge24=int((rep.n_weeks_reported >= 24).sum()), ge25=int((rep.n_weeks_reported >= 25).sum()))
STRATA = S["stratum"]

                    physicians  at_risk  episodes
stratum                                          
least engaged               37       86        12
moderately engaged         106     1543       216
most engaged               105     2997       269

physicians reporting 24 or more weeks  105 (42.3%)
physicians reporting 25 or more weeks  101 (40.7%)


### Cell 6 — A05: Geographic distribution of the reporting cohort

**Pseudocode.**

1. Take each reporting physician's province from the person-week panel.
2. Count provinces represented out of the eighty-one in the country.
3. Rank provinces by physician count and report the three largest centres with
   their shares, and the combined share those three hold.
4. Provinces contributing fewer than five physicians are pooled for reporting.


In [6]:
prov = pw.groupby("participant_id").province_code.first()
n_prov = prov.nunique()
vc = prov.value_counts()
n_rep = len(prov)
print(f"provinces represented   {n_prov} of 81")
print(f"three largest centres   " + ", ".join(
    f"province {p} {c} ({100*c/n_rep:.1f}%)" for p, c in vc.head(3).items()))
top3 = int(vc.head(3).sum())
print(f"share in the three largest centres  {top3} of {n_rep} ({100*top3/n_rep:.1f}%)")
small = vc[vc < 5]
print(f"provinces contributing fewer than five physicians  {len(small)} "
      f"(pooled: {int(small.sum())} physicians)")
record("A05", provinces=n_prov, top3=top3, top3_pct=100*top3/n_rep)

provinces represented   46 of 81
three largest centres   province 6 72 (29.0%), province 35 30 (12.1%), province 34 29 (11.7%)
share in the three largest centres  131 of 248 (52.8%)
provinces contributing fewer than five physicians  36 (pooled: 76 physicians)


### Cell 7 — A06: Baseline numeric variables of the reporting cohort

**Pseudocode.**

1. For age and household size in the reporting cohort report median, interquartile
   range, range and mean.
2. Test each against normality with the Shapiro-Wilk test, which is what justifies
   the median-based presentation and the rank-based comparisons used later.
3. Report the count with a school-age child at home.


In [7]:
rep = phys[phys.reported == 1]
for v, lab in [("age", "age, years"), ("hh", "household size")]:
    s = rep[v].dropna()
    q1, q3 = s.quantile([.25, .75])
    W, p = stats.shapiro(s)
    print(f"{lab:16s} median {s.median():g} (IQR {q1:g}-{q3:g}; range {s.min():g}-{s.max():g}); "
          f"mean {s.mean():.3f}; Shapiro-Wilk W = {W:.4f}, p = {p:.2e}")
m = rep.school_kids_any
print(f"school-age child at home  {int(m.sum())} ({100*m.mean():.1f}%)")
record("A06", median_age=float(rep.age.median()), mean_age=float(rep.age.mean()),
       median_hh=float(rep.hh.median()), mean_hh=float(rep.hh.mean()),
       school_kids=int(m.sum()))

age, years       median 36 (IQR 32.75-48; range 25-72); mean 40.383; Shapiro-Wilk W = 0.8895, p = 1.76e-12
household size   median 2 (IQR 2-3; range 1-6); mean 2.548; Shapiro-Wilk W = 0.9047, p = 1.91e-11
school-age child at home  100 (40.3%)


### Cell 8 — A07: Demographic and professional characteristics

**Pseudocode.**

1. Over the reporting cohort tabulate sex, institution type and job title as
   counts and percentages of the valid responses for each item.
2. Report the derived professional indicators separately: university-hospital
   affiliation, academic title, general practitioner, resident, and whether the
   physician performs face-to-face patient examination.


In [8]:
rep = phys[phys.reported == 1]
for v, lab in [("female", "female"), ("university", "university hospital"),
               ("academic", "academic title"), ("gp", "general practitioner"),
               ("resident", "resident"), ("sees_pat", "performs face-to-face examination")]:
    s = rep[v].dropna()
    print(f"{lab:36s} {int(s.sum())} of {len(s)} ({100*s.mean():.1f}%)")
print("\ninstitution type")
for k, c in rep.institution_type.value_counts().items():
    print(f"  {k[:58]:60s} {c} ({100*c/len(rep):.1f}%)")
print("job title")
for k, c in rep.job_title.value_counts().items():
    print(f"  {k[:58]:60s} {c} ({100*c/len(rep):.1f}%)")
record("A07", female=int(rep.female.sum()), university=int(rep.university.sum()),
       academic=int(rep.academic.sum()), sees_pat=int(rep.sees_pat.sum()))

female                               165 of 248 (66.5%)
university hospital                  105 of 248 (42.3%)
academic title                       89 of 248 (35.9%)
general practitioner                 25 of 248 (10.1%)
resident                             44 of 248 (17.7%)
performs face-to-face examination    112 of 248 (45.2%)

institution type
  Üniversite (Kamu ya da Vakıf) - Hastane                      105 (42.3%)
  Sağlık Bakanlığı: Merkez - İl Sağlık Müdürlüğü - İlçe Sağl   61 (24.6%)
  Sağlık Bakanlığı - Hastane                                   56 (22.6%)
  Sağlık Bakanlığı: Aile Sağlığı Merkezi                       16 (6.5%)
  Özel Hastane / Klinik                                        10 (4.0%)
job title
  Uzman hekim / Yan dal uzmanı hekim                           90 (36.3%)
  Akademik ünvanı olan hekim                                   89 (35.9%)
  Araştırma görevlisi (ana dal veya yan dal) hekim             44 (17.7%)
  Pratisyen hekim                               

### Cell 9 — A08: Household, exposure and behavioural characteristics

**Pseudocode.**

1. Tabulate the household and behavioural items over the reporting cohort, each
   against its own valid-response denominator rather than a single cohort total,
   because the household-composition items were shown only to physicians reporting
   more than one household member.
2. Items: household size, a school-age child at home, a household member at risk of
   influenza complications, tobacco use, public-transport commuting time,
   respiratory allergy, physical activity, and frequent contact with children,
   adults and older people.
3. This block has no single scalar target; it produces a table of item-specific
   proportions.


In [9]:
rep = phys[phys.reported == 1]
items = [("school_kids_any", "school-age child at home", True),
         ("household_highrisk", "household member at complication risk", True),
         ("smoker_current", "current tobacco use", True),
         ("smoker_regular", "regular tobacco use", True),
         ("pub_transport_user", "uses public transport", True),
         ("allergy", "respiratory allergy", True),
         ("active", "sufficient physical activity", True),
         ("contact_children", "frequent contact with children", True),
         ("contact_elderly", "frequent contact with older people", True),
         ("hh", "household size (mean)", False),
         ("pub_transport_ord", "public-transport time (ordinal mean)", False),
         ("social_risk_score", "social-contact score (mean)", False)]
rows = []
for v, lab, binary in items:
    s = rep[v].dropna()
    val = f"{int(s.sum())} ({100*s.mean():.1f}%)" if binary else f"{s.mean():.3f}"
    rows.append(dict(item=lab, valid_n=len(s), value=val))
print(pd.DataFrame(rows).to_string(index=False))
print("\ntobacco status (categorical)")
for k, c in rep.tobacco_status.value_counts().items():
    print(f"  {k:12s} {c} ({100*c/len(rep):.1f}%)")
record("A08", n_items=len(items))

                                 item  valid_n       value
             school-age child at home      248 100 (40.3%)
household member at complication risk      205  72 (35.1%)
                  current tobacco use      248  49 (19.8%)
                  regular tobacco use      248   19 (7.7%)
                uses public transport      248  51 (20.6%)
                  respiratory allergy      248  96 (38.7%)
         sufficient physical activity      248 117 (47.2%)
       frequent contact with children      248   15 (6.0%)
   frequent contact with older people      248   20 (8.1%)
                household size (mean)      248       2.548
 public-transport time (ordinal mean)      248       0.391
          social-contact score (mean)      248       0.802

tobacco status (categorical)
  never        163 (65.7%)
  former       34 (13.7%)
  occasional   29 (11.7%)
  daily        19 (7.7%)
  unknown      3 (1.2%)


### Cell 10 — A09: Clinical, vaccination and ILI-history characteristics

**Pseudocode.**

1. Collapse the self-reported annual influenza-like-illness frequency item to its
   four ordered levels and tabulate it over the reporting cohort, then over the
   model sample that excludes physicians who declined the health-condition item.
2. Report current-season and prior-season vaccination, prior-season illness and
   prior-season care-seeking.
3. Report how many physicians declined the health-condition question. Those
   responses stay missing; they are not imputed to zero, which is what fixes the
   individual-model sample at 242.


In [10]:
rep = phys[phys.reported == 1]
lv = rep.ili_freq_ord4.value_counts().sort_index()
print("susceptibility, four ordered levels, reporting cohort  " +
      " / ".join(str(int(x)) for x in lv) + f"   (n = {int(lv.sum())})")
mod = rep[rep.comp_risk_any_nan.notna()]
lv2 = mod.ili_freq_ord4.value_counts().sort_index()
print("susceptibility, four ordered levels, model sample      " +
      " / ".join(str(int(x)) for x in lv2) + f"   (n = {int(lv2.sum())})")
for v, lab in [("vax", "current-season vaccination"), ("prev_vax", "prior-season vaccination"),
               ("had_ili_prev", "prior-season influenza-like illness"),
               ("sought_care", "prior-season care-seeking")]:
    s = rep[v].dropna()
    print(f"{lab:36s} {int(s.sum())} ({100*s.mean():.1f}%)")
nd = int(rep.comp_risk_any_nan.isna().sum())
print(f"{'declined the health-condition item':36s} {nd} ({100*nd/len(rep):.1f}%) — kept missing")
print(f"{'any health condition (valid n = ' + str(len(rep)-nd) + ')':36s} "
      f"{int(rep.comp_risk_any_nan.sum())} ({100*rep.comp_risk_any_nan.mean():.1f}%)")
record("A09", levels_248=[int(x) for x in lv], levels_242=[int(x) for x in lv2],
       vax=int(rep.vax.sum()), prev_vax=int(rep.prev_vax.sum()), declined=nd)

susceptibility, four ordered levels, reporting cohort  8 / 156 / 70 / 14   (n = 248)
susceptibility, four ordered levels, model sample      8 / 153 / 67 / 14   (n = 242)
current-season vaccination           91 (36.7%)
prior-season vaccination             99 (39.9%)
prior-season influenza-like illness  201 (81.0%)
prior-season care-seeking            55 (22.2%)
declined the health-condition item   6 (2.4%) — kept missing
any health condition (valid n = 242) 53 (21.9%)


### Cell 11 — A10: Occupational characteristics of patient-facing physicians

**Pseudocode.**

1. Restrict to physicians who perform face-to-face patient examination.
2. Tabulate high-risk unit, primary patient group, aerosol-generating procedure
   share, mask use and monthly on-call shifts as counts and percentages, and daily
   patient volume as a median with interquartile range.
3. Report how many of the patient-facing physicians have a daily patient-volume
   value, which fixes the complete-case occupational analysis set.


In [11]:
occ = phys[phys.cohort_112_primary == 1]
print(f"patient-facing physicians                      {len(occ)}")
print(f"with a daily patient-volume value              {int(occ.pat_per_day.notna().sum())} of {len(occ)}")
s = occ.pat_per_day.dropna()
q1, q3 = s.quantile([.25, .75])
print(f"daily patient volume    median {s.median():g} (IQR {q1:g}-{q3:g}; range {s.min():g}-{s.max():g})")
s = occ.nobet if "nobet" in occ else occ.oncall_any
for v, lab in [("high_risk_unit", "works in a high-risk unit"), ("mask", "routine mask use"),
               ("oncall_any", "any monthly on-call duty")]:
    x = occ[v].dropna()
    print(f"{lab:36s} {int(x.sum())} of {len(x)} ({100*x.mean():.1f}%)")
print("aerosol-generating procedure share (ordinal)")
for k, c in occ.aerosol_ord.value_counts().sort_index().items():
    print(f"  level {int(k)}  {c} ({100*c/occ.aerosol_ord.notna().sum():.1f}%)")
print("primary patient group")
for k, c in occ.primary_patient_group.value_counts().items():
    print(f"  {k:20s} {c}")
record("A10", n_patient_facing=len(occ), n_complete=int(occ.pat_per_day.notna().sum()))

patient-facing physicians                      112
with a daily patient-volume value              110 of 112
daily patient volume    median 20 (IQR 10-40; range 1-100)
works in a high-risk unit            76 of 112 (67.9%)
routine mask use                     43 of 112 (38.4%)
any monthly on-call duty             28 of 112 (25.0%)
aerosol-generating procedure share (ordinal)
  level 0  68 (60.7%)
  level 1  27 (24.1%)
  level 2  7 (6.2%)
  level 3  10 (8.9%)
primary patient group
  Erişkin              87
  Çocuk                20
  65 yaş ve üzeri      4
  İmmünsüprese         1


### Cell 12 — A11: Episode burden, cumulative incidence and recurrence

**Pseudocode.**

1. Cumulative season attack rate is the share of reporting physicians with at
   least one incident episode.
2. Recurrence is the share with two or more; also report the maximum episode count
   in any one physician.
3. Report symptom-free reporters, that is reporters with no episode, and keep them
   distinct from the never-reporters, who filed nothing at all.


In [12]:
rep = phys[phys.reported == 1]
any_ep = (rep.n_episodes > 0)
rec = (rep.n_episodes >= 2)
print(f"cumulative season attack rate   {int(any_ep.sum())} of {len(rep)} ({100*any_ep.mean():.1f}%)")
print(f"two or more episodes            {int(rec.sum())} ({100*rec.mean():.1f}%)")
print(f"maximum episodes in a physician {int(rep.n_episodes.max())}")
print(f"symptom-free reporters          {int((rep.n_episodes==0).sum())}  "
      f"(distinct from the {int((phys.reported==0).sum())} never-reporters)")
print("\nepisode-count distribution")
print(rep.n_episodes.value_counts().sort_index().to_string())
record("A11", attack_rate=100*any_ep.mean(), n_any=int(any_ep.sum()),
       recurrent=int(rec.sum()), max_ep=int(rep.n_episodes.max()),
       symptom_free=int((rep.n_episodes == 0).sum()))

cumulative season attack rate   192 of 248 (77.4%)
two or more episodes            115 (46.4%)
maximum episodes in a physician 12
symptom-free reporters          56  (distinct from the 56 never-reporters)

episode-count distribution
n_episodes
0     56
1     77
2     41
3     26
4     16
5     15
6      9
7      4
8      2
11     1
12     1


### Cell 13 — A12: Agent-cluster distribution of incident episodes

**Pseudocode.**

1. Tabulate the cleaned incident episodes by the physician's self-assigned agent
   cluster: A viral-systemic (the influenza-like proxy), B viral-local, C
   bacterial, D indeterminate.
2. Report counts and percentages of all incident episodes. The A-cluster share is
   the participatory positivity used in every system-level comparison.


In [13]:
vc = episodes.cluster_letter.value_counts().reindex(list("ABCD"))
labels = {"A": "A viral-systemic (influenza-like)", "B": "B viral-local",
          "C": "C bacterial", "D": "D indeterminate"}
for k, c in vc.items():
    print(f"{labels[k]:36s} {int(c):3d} ({100*c/len(episodes):.1f}%)")
print(f"{'total incident episodes':36s} {len(episodes)}")
print(f"\nseason participatory positivity (A-cluster share)  "
      f"{int(vc['A'])} of {len(episodes)} = {100*vc['A']/len(episodes):.4f}%")
record("A12", A=int(vc.A), B=int(vc.B), C=int(vc.C), D=int(vc.D),
       positivity=100*vc.A/len(episodes))

A viral-systemic (influenza-like)     91 (18.3%)
B viral-local                        341 (68.6%)
C bacterial                           43 (8.7%)
D indeterminate                       22 (4.4%)
total incident episodes              497

season participatory positivity (A-cluster share)  91 of 497 = 18.3099%


### Cell 14 — A13: Clinical course of influenza-like (A-cluster) episodes

**Pseudocode.**

1. Restrict to A-cluster episodes.
2. Report the median and interquartile range of the maximum reported symptom
   duration in days.
3. Report the count with any work absence, and the count whose weekly form records
   an actual diagnostic test rather than an intention to test.
4. Presenteeism is the count who reported illness but could not interrupt work;
   report it against A-cluster episodes and against all incident episodes.


In [14]:
epA = episodes[episodes.cluster_letter == "A"]
s = epA.max_symptom_days.dropna()
q1, q3 = s.quantile([.25, .75])
print(f"A-cluster episodes                     {len(epA)}")
print(f"symptom duration, days    median {s.median():g} (IQR {q1:g}-{q3:g}; range {s.min():g}-{s.max():g})")
print(f"any work absence                       {int(epA.any_work_absence.sum())} "
      f"({100*epA.any_work_absence.mean():.1f}%)")
tested = int(epA.any_test.sum())
print(f"an actual diagnostic test              {tested} ({100*tested/len(epA):.1f}% of A-cluster; "
      f"{100*tested/len(episodes):.1f}% of all {len(episodes)} episodes)")
# presenteeism: reported illness but could not interrupt work
PRES = "Evet - ancak İŞE ARA VEREMEDİM"
sym = pw[(pw.symptomatic == 1) & (pw.symptom_week_cluster == "A")]
pres = sym.groupby("episode_id_filled").daily_routine_impact.apply(lambda s: (s == PRES).any())
print(f"worked through illness (presenteeism)  {int(pres.sum())} of {len(pres)} "
      f"({100*pres.mean():.1f}%)")
print("\ndaily-routine impact, A-cluster symptomatic weeks")
for k, c in sym.daily_routine_impact.value_counts().items():
    print(f"  {k[:64]:66s} {c}")
record("A13", n_A=len(epA), median_days=float(s.median()),
       absence=int(epA.any_work_absence.sum()), tested=tested, presenteeism=int(pres.sum()))

A-cluster episodes                     91
symptom duration, days    median 4 (IQR 3-6; range 1-21)
any work absence                       29 (31.9%)
an actual diagnostic test              5 (5.5% of A-cluster; 1.0% of all 497 episodes)
worked through illness (presenteeism)  42 of 91 (46.2%)

daily-routine impact, A-cluster symptomatic weeks
  Evet - ancak İŞE ARA VEREMEDİM                                     43
  Evet - İŞE ARA VERDİM: Ayaktan izlem ve tedavi                     31
  Hayır değişmedi - günlük aktivitelerime her zamanki gibi devam e   22
  Evet - İŞE ARA VERDİM: Yatış Gerektiren izlem ve tedavi            1


### Cell 15 — A14: Attributed source of infection by clinical role

**Pseudocode.**

1. The source-of-infection question allows more than one answer, so flags are
   counted, not episodes: total flags can exceed the number of episodes.
2. Aggregate the weekly source flags to the episode through the forward-filled
   episode identifier, restricted to A-cluster episodes.
3. Report flags per category and the flag total, then the same counts split by
   whether the physician performs face-to-face examination, expressed per hundred
   episodes in each role so the two roles are comparable.


In [15]:
symA = pw[(pw.symptomatic == 1) & (pw.symptom_week_cluster == "A")]
flags = symA.groupby("episode_id_filled").agg(
    household=("source_household", "max"), social=("source_social", "max"),
    healthcare=("source_healthcare", "max"), unknown=("source_unknown", "max"),
    sees_pat=("sees_pat", "first"))
tot = flags[["household", "social", "healthcare", "unknown"]].sum()
print(f"A-cluster episodes {len(flags)}; source flags {int(tot.sum())}")
for k, v in tot.items():
    print(f"  {k:12s} {int(v)}")
print("\nby clinical role, flags per 100 episodes")
for r, lab in [(1, "face-to-face examination"), (0, "no face-to-face examination")]:
    sub = flags[flags.sees_pat == r]
    per100 = 100 * sub[["household", "social", "healthcare", "unknown"]].sum() / len(sub)
    print(f"  {lab:28s} (n = {len(sub):2d})  " +
          "  ".join(f"{k} {v:.1f}" for k, v in per100.items()))
record("A14", flag_total=int(tot.sum()), household=int(tot.household),
       social=int(tot.social), healthcare=int(tot.healthcare), unknown=int(tot.unknown))

A-cluster episodes 91; source flags 102
  household    37
  social       26
  healthcare   25
  unknown      14

by clinical role, flags per 100 episodes
  face-to-face examination     (n = 40)  household 40.0  social 20.0  healthcare 40.0  unknown 15.0
  no face-to-face examination  (n = 51)  household 41.2  social 35.3  healthcare 17.6  unknown 15.7


### Cell 16 — A15: Season incidence rate per 100 at-risk person-weeks

**Pseudocode.**

1. The numerator is incident episodes; the denominator is person-weeks at risk of
   a new episode, that is filed person-weeks less continuation weeks.
2. Express the crude rate per hundred at-risk person-weeks and attach an exact
   Poisson interval.
3. Report the filed-person-week rate alongside, and the number of weeks in which
   the two denominators differ, so that the choice of denominator is visible.


In [16]:
k, n_ar, n_pwk = int(pw.new_episode.sum()), int(pw.at_risk_new_episode.sum()), len(pw)
rate = 100 * k / n_ar
lo = 100 * stats.chi2.ppf(.025, 2*k) / 2 / n_ar
hi = 100 * stats.chi2.ppf(.975, 2*k+2) / 2 / n_ar
print(f"season incidence   {k} / {n_ar:,} at-risk person-weeks = {rate:.4f} per 100 "
      f"(95% CI {lo:.2f}-{hi:.2f})")
print(f"on filed person-weeks  {k} / {n_pwk:,} = {100*k/n_pwk:.4f} per 100")
diff = int((wk.pw_at_risk_new != wk.person_weeks).sum())
print(f"weeks where the two denominators differ  {diff} of {N_WEEKS}")
record("A15", rate=rate, ci_lo=lo, ci_hi=hi, rate_filed=100*k/n_pwk, weeks_differ=diff)

season incidence   497 / 4,626 at-risk person-weeks = 10.7436 per 100 (95% CI 9.82-11.73)
on filed person-weeks  497 / 4,729 = 10.5096 per 100
weeks where the two denominators differ  32 of 33


### Cell 17 — A16: Weekly surveillance indicator panel

**Pseudocode.**

1. Assemble the weekly panel: participatory reporters, at-risk person-weeks,
   incident episodes, A-cluster episodes, participatory positivity, sentinel
   consultations, sentinel positives and sentinel positivity.
2. Describe the participatory denominator, that is all cleaned incident episodes
   in the week, and the sentinel specimen count.
3. Apply the sparse-week rule: flag weeks whose participatory denominator falls
   below ten. Flagging is disclosure only; no week is ever excluded.


In [17]:
SPARSE_THRESHOLD = 10                # roughly two-thirds of the 33-week median
panel = pd.DataFrame({
    "week": WEEKS, "reporters": wk.n_reporters, "at_risk": wk.pw_at_risk_new,
    "episodes": wk.n_new_episodes, "A_episodes": wk.n_new_A,
    "part_positivity_pct": part_share.round(2),
    "sent_specimens": wk.sb_ili_consultations, "sent_positives": wk.sb_positive_cases,
    "sent_positivity_pct": sent_share.round(2),
    "ili_per_1000_at_risk": wk.ili_rate_per1000_at_risk.round(2)})
panel["sparse_flag"] = (wk.n_new_episodes < SPARSE_THRESHOLD).astype(int)
print(panel.to_string(index=False))
d = wk.n_new_episodes
print(f"\nparticipatory denominator  median {d.median():.0f}, range {d.min()}-{d.max()}")
s = wk.sb_ili_consultations
print(f"sentinel specimens         mean {s.mean():.3f}, median {s.median():.0f}, "
      f"range {s.min()}-{s.max()}; weeks below 100 specimens {int((s<100).sum())} of {N_WEEKS}")
record("A16", denom_median=float(d.median()), denom_min=int(d.min()), denom_max=int(d.max()),
       sent_mean=float(s.mean()), sent_median=float(s.median()), sent_below100=int((s < 100).sum()))
WEEKLY_PANEL = panel

    week  reporters  at_risk  episodes  A_episodes  part_positivity_pct  sent_specimens  sent_positives  sent_positivity_pct  ili_per_1000_at_risk  sparse_flag
2025-W40         44       44         8           2                25.00             121              13                10.74                181.82            1
2025-W41         72       69         8           3                37.50             136              22                16.18                115.94            1
2025-W42         83       81        12           1                 8.33             113              14                12.39                148.15            0
2025-W43         84       82        10           1                10.00             140               9                 6.43                121.95            0
2025-W44         89       86        12           2                16.67              76               5                 6.58                139.53            0
2025-W45        104      100         7  

### Cell 18 — A17: Count-versus-rate trend check on the weekly series

**Pseudocode.**

1. Correlate the weekly incident-episode count with the week index.
2. Correlate the weekly incidence rate on the at-risk denominator with the week
   index.
3. The two answer different questions: the count rises with the growing panel, the
   rate does not. Report both so the rising count is not read as an incidence trend.
4. Report the same correlation on the filed-person-week rate for comparison.


In [18]:
for lab, y in [("weekly episode count", wk.n_new_episodes),
               ("weekly rate per 1,000 at-risk person-weeks", wk.ili_rate_per1000_at_risk),
               ("weekly rate per 1,000 filed person-weeks", wk.ili_rate_per1000_pw)]:
    r, p = stats.pearsonr(wk.week_idx, y)
    print(f"{lab:44s} against week index  r = {r:+.5f}, p = {p:.6f}")
r_c, p_c = stats.pearsonr(wk.week_idx, wk.n_new_episodes)
r_r, p_r = stats.pearsonr(wk.week_idx, wk.ili_rate_per1000_at_risk)
record("A17", r_count=r_c, p_count=p_c, r_rate=r_r, p_rate=p_r)

weekly episode count                         against week index  r = +0.46443, p = 0.006474
weekly rate per 1,000 at-risk person-weeks   against week index  r = -0.27097, p = 0.127185
weekly rate per 1,000 filed person-weeks     against week index  r = -0.26288, p = 0.139405


### Cell 19 — A18: Dashboard reach and summary counters

**Pseudocode.**

1. The dashboard reach counters — unique visitors and page views — are
   service-reported figures held outside the analysis workbook, so they are entered
   here as declared inputs and labelled as such.
2. Report them beside the registered-physician count from the workbook, which is
   the one counter this notebook can verify.


In [19]:
# service-reported counters, not carried in the analysis workbook
DASHBOARD_UNIQUE_VISITORS = 1504
DASHBOARD_PAGE_VIEWS = 3833
print(f"registered physicians (from the workbook)   {len(phys)}")
print(f"unique visitors (service-reported)          {DASHBOARD_UNIQUE_VISITORS:,}")
print(f"page views (service-reported)               {DASHBOARD_PAGE_VIEWS:,}")
print(f"page views per unique visitor               "
      f"{DASHBOARD_PAGE_VIEWS/DASHBOARD_UNIQUE_VISITORS:.2f}")
record("A18", registered=len(phys), visitors=DASHBOARD_UNIQUE_VISITORS,
       page_views=DASHBOARD_PAGE_VIEWS)

registered physicians (from the workbook)   304
unique visitors (service-reported)          1,504
page views (service-reported)               3,833
page views per unique visitor               2.55


### Cell 20 — A19: Person-week panel structure and counting-process construction

**Pseudocode.**

1. Verify the counting-process construction: every filed week is one row with a
   half-open interval on the total-time clock, so start equals week index minus one
   and stop equals week index.
2. Confirm that the event indicator equals the incident-episode indicator and sums
   to the episode total.
3. Describe the panel: weeks per physician, physicians with at least one gap, the
   number and length of contiguous runs, and registration staggering measured as
   the distribution of first observed weeks.
4. Confirm that at-risk equals filed less continuation weeks, and that the derived
   gap indicator agrees with the stored one.


In [20]:
assert (pw.tstop - pw.tstart == 1).all()
assert (pw.tstart == pw.week_idx - 1).all()
assert (pw.event == pw.new_episode).all()
assert (pw.gap_before_derived == pw.gap_before).all()
print(f"panel rows {len(pw):,}  physicians {pw.participant_id.nunique()}  "
      f"events {int(pw.event.sum())}")
print(f"at risk {int(pw.at_risk_new_episode.sum()):,} = filed {len(pw):,} "
      f"less continuation {int(pw.symptomatic.sum()-pw.new_episode.sum())}")
wpp = pw.groupby("participant_id").size()
print(f"weeks per physician   median {wpp.median():.0f}, mean {wpp.mean():.3f}, "
      f"range {wpp.min()}-{wpp.max()}")
gaps = pw.groupby("participant_id").gap_before_derived.sum()
print(f"physicians with at least one gap  {int((gaps>0).sum())} of {len(gaps)}; "
      f"total gaps {int(gaps.sum())}")
runs = pw.groupby("participant_id").gap_before_derived.sum() + 1
print(f"contiguous runs per physician  median {runs.median():.0f}, max {int(runs.max())}")
first = pw.groupby("participant_id").week_idx.min()
print(f"registration staggering: first observed week  median {first.median():.0f}, "
      f"range {first.min()}-{first.max()}; "
      f"{int((first==1).sum())} physicians present from the opening week")
print("\nfirst observed week, distribution over the first eight weeks")
print(first.value_counts().sort_index().head(8).to_string())
record("A19", rows=len(pw), physicians=int(pw.participant_id.nunique()),
       events=int(pw.event.sum()), n_with_gap=int((gaps > 0).sum()), n_gaps=int(gaps.sum()))

panel rows 4,729  physicians 248  events 497
at risk 4,626 = filed 4,729 less continuation 103
weeks per physician   median 20, mean 19.069, range 1-33
physicians with at least one gap  169 of 248; total gaps 528
contiguous runs per physician  median 3, max 10
registration staggering: first observed week  median 7, range 1-32; 44 physicians present from the opening week

first observed week, distribution over the first eight weeks
week_idx
1    44
2    31
3    13
4     7
5     5
6    18
7    27
8    15


## S3.2 Individual-level analysis

### Cell 21 — A20: Vaccinated person-week partition under the two-week interval

**Pseudocode.**

1. Vaccination enters the incidence models as a TIME-VARYING indicator: a
   person-week counts as protected only from two weeks after the vaccination week.
2. Partition the person-weeks of vaccinated physicians into three exclusive parts:
   weeks preceding the vaccination week, weeks inside the two-week interval, and
   protected weeks.
3. Verify the three parts sum to the vaccinated person-week total, and express the
   protected weeks as a share of the whole panel, which is the exposed person-time
   share the power calculations use.


In [21]:
vaxwk = pw[pw.vax == 1]
n_vax = len(vaxwk)
n_prot = int(pw.vax_protected.sum())
n_interval = int(((vaxwk.week_idx >= vaxwk.vax_week_idx) &
                  (vaxwk.week_idx < vaxwk.vax_week_idx + 2)).sum())
n_before = int((vaxwk.week_idx < vaxwk.vax_week_idx).sum())
print(f"person-weeks of vaccinated physicians   {n_vax:,}")
print(f"  preceding the vaccination week        {n_before}")
print(f"  inside the two-week interval          {n_interval} ({100*n_interval/n_vax:.3f}% of {n_vax:,})")
print(f"  protected                             {n_prot:,}")
assert n_before + n_interval + n_prot == n_vax
print(f"protected share of the full {len(pw):,}-week panel  "
      f"{100*n_prot/len(pw):.3f}%")
record("A20", vax_weeks=n_vax, protected=n_prot, interval=n_interval, before=n_before,
       protected_share=100*n_prot/len(pw))
PROTECTED_SHARE = n_prot / len(pw)

person-weeks of vaccinated physicians   1,874
  preceding the vaccination week        178
  inside the two-week interval          83 (4.429% of 1,874)
  protected                             1,613
protected share of the full 4,729-week panel  34.109%


### Cell 22 — A21: Vaccination uptake model

**Pseudocode.**

1. The outcome is current-season influenza vaccination. The sample is reporting
   physicians with a health-condition response: 242, of whom 90 were vaccinated.
2. The adjustment set is pre-specified on epidemiological grounds: prior-season
   vaccination, age in ten-year units, sex, university affiliation, face-to-face
   patient care, any health condition conferring complication risk, household
   school-age children and susceptibility in four ordered levels. No variable is
   selected on the data.
3. Fit the logistic model and report odds ratios with Wald intervals and p-values.


In [22]:
d1 = phys[(phys.cohort_248_reporter == 1)].dropna(subset=["comp_risk_any_nan"])
F1 = ("vax ~ age10 + female + comp_risk_any_nan + ili_freq_ord4 + sees_pat "
      "+ prev_vax + school_kids_any + university")
fit1 = smf.logit(F1, data=d1).fit(disp=0)
print(f"sample {len(d1)} physicians, {int(d1.vax.sum())} vaccinated")
print(pd.DataFrame({"OR": np.exp(fit1.params), "lo": np.exp(fit1.conf_int()[0]),
                    "hi": np.exp(fit1.conf_int()[1]), "p": fit1.pvalues}).round(5).to_string())
record("A21", n=len(d1), n_vax=int(d1.vax.sum()),
       or_prev_vax=float(np.exp(fit1.params["prev_vax"])),
       or_university=float(np.exp(fit1.params["university"])),
       p_university=float(fit1.pvalues["university"]),
       or_school_kids=float(np.exp(fit1.params["school_kids_any"])),
       p_school_kids=float(fit1.pvalues["school_kids_any"]))
D1, FIT1, F1_FORMULA = d1, fit1, F1

sample 242 physicians, 90 vaccinated
                         OR       lo        hi        p
Intercept           0.19453  0.03503   1.08037  0.06126
age10               0.98954  0.70504   1.38885  0.95153
female              0.80788  0.39464   1.65383  0.55946
comp_risk_any_nan   1.50324  0.65811   3.43369  0.33344
ili_freq_ord4       1.06539  0.62006   1.83058  0.81858
sees_pat            0.82770  0.41517   1.65014  0.59115
prev_vax           14.69884  7.41620  29.13297  0.00000
school_kids_any     1.94784  0.95866   3.95771  0.06529
university          0.42787  0.20663   0.88602  0.02226


### Cell 23 — A22: Vaccination-uptake calibration and discrimination

**Pseudocode.**

1. Take the apparent area under the curve on the fitting sample.
2. Correct it for optimism with the Efron bootstrap, refitting the model in every one
   of the thousand replicates.
3. Assess calibration with the Hosmer-Lemeshow test on deciles of fitted risk and
   with the calibration slope, that is the coefficient from regressing the outcome on
   the linear predictor.


In [23]:
app, opt, corr, nrep = efron_optimism(F1_FORMULA, D1, "vax", B=1000, seed=20260707)
print(f"apparent AUC             {app:.5f}")
print(f"Efron optimism           {opt:.5f}   ({nrep} replicates, seed 20260707)")
print(f"optimism-corrected AUC   {corr:.5f}")
p_hat = FIT1.predict(D1)
grp = pd.qcut(p_hat, 10, labels=False, duplicates="drop")
obs = D1.vax.groupby(grp).sum(); exp = p_hat.groupby(grp).sum(); n_g = p_hat.groupby(grp).size()
hl = (((obs - exp) ** 2) / (exp * (1 - exp / n_g))).sum()
df_hl = n_g.size - 2
print(f"\nHosmer-Lemeshow  chi-square {hl:.4f}, df {df_hl}, p = {stats.chi2.sf(hl, df_hl):.4f} "
      f"({n_g.size} groups of fitted risk)")
# The calibration slope on the fitting sample is one by construction, so it is
# estimated out of sample: fit in each bootstrap replicate, then regress the
# original outcome on the linear predictor that replicate produces.
rng = np.random.default_rng(20260707)
slopes = []
for _ in range(1000):
    db = D1.iloc[rng.integers(0, len(D1), len(D1))]
    if db.vax.nunique() < 2:
        continue
    try:
        fb = smf.logit(F1_FORMULA, data=db).fit(disp=0)
    except Exception:
        continue
    p_b = np.clip(fb.predict(D1), 1e-9, 1 - 1e-9)
    lp_b = np.log(p_b / (1 - p_b))
    try:
        slopes.append(smf.logit("vax ~ lp_b", data=D1.assign(lp_b=lp_b)).fit(disp=0).params["lp_b"])
    except Exception:
        continue
cal_slope = float(np.mean(slopes))
print(f"bootstrap-validated calibration slope  {cal_slope:.5f} "
      f"(1.0 indicates no overfitting of the risk scale; {len(slopes)} replicates)")
record("A22", apparent=app, optimism=opt, corrected=corr, hl_chi2=float(hl),
       hl_df=int(df_hl), hl_p=float(stats.chi2.sf(hl, df_hl)), cal_slope=cal_slope)

apparent AUC             0.84477
Efron optimism           0.02589   (1000 replicates, seed 20260707)
optimism-corrected AUC   0.81888

Hosmer-Lemeshow  chi-square 1.5328, df 8, p = 0.9921 (10 groups of fitted risk)
bootstrap-validated calibration slope  0.89293 (1.0 indicates no overfitting of the risk scale; 1000 replicates)


### Cell 24 — A23: Susceptibility (annual ILI frequency) model

**Pseudocode.**

1. The outcome is the self-reported annual influenza-like-illness frequency,
   collapsed to FOUR ordered levels. It is never treated as five levels.
2. Fit an ordinal proportional-odds logistic regression on the 242 physicians with a
   health-condition response.
3. The pre-specified adjustment set is age, sex, any health condition, household
   school-age children, household size, respiratory allergy, current tobacco use and
   face-to-face patient care.
4. Report cumulative odds ratios with intervals, and the estimated cut-points.


In [24]:
d2 = phys[(phys.cohort_248_reporter == 1)].dropna(subset=["comp_risk_any_nan"])
V2 = ["age10", "female", "comp_risk_any_nan", "school_kids_any", "hh",
      "allergy", "smoker_current", "sees_pat"]
lv = d2.ili_freq_ord4.value_counts().sort_index()
print(f"sample {len(d2)} physicians; four ordered outcome levels " +
      " / ".join(str(int(x)) for x in lv))
om = OrderedModel(d2.ili_freq_ord4, d2[V2], distr="logit").fit(method="bfgs", disp=0, maxiter=2000)
ci = om.conf_int()
print(pd.DataFrame({"cumulative_OR": np.exp(om.params[V2]),
                    "lo": np.exp(ci[0][V2]), "hi": np.exp(ci[1][V2]),
                    "p": om.pvalues[V2]}).round(5).to_string())
raw = om.params[len(V2):].values                # first cut-point, then log increments
cuts = np.cumsum(np.r_[raw[0], np.exp(raw[1:])])
print("\nestimated cut-points on the latent scale  " +
      ", ".join(f"{c:.5f}" for c in cuts))
record("A23", n=len(d2), levels=[int(x) for x in lv],
       or_age=float(np.exp(om.params["age10"])), p_age=float(om.pvalues["age10"]),
       or_school=float(np.exp(om.params["school_kids_any"])),
       p_school=float(om.pvalues["school_kids_any"]),
       or_comp=float(np.exp(om.params["comp_risk_any_nan"])),
       p_comp=float(om.pvalues["comp_risk_any_nan"]))
D2, OM2, V2_TERMS = d2, om, V2

sample 242 physicians; four ordered outcome levels 8 / 153 / 67 / 14
                   cumulative_OR       lo       hi        p
age10                    0.69853  0.52978  0.92105  0.01099
female                   0.95935  0.53814  1.71023  0.88811
comp_risk_any_nan        1.80055  0.94212  3.44118  0.07515
school_kids_any          2.02481  0.99979  4.10068  0.05007
hh                       1.20274  0.88241  1.63934  0.24270
allergy                  1.13439  0.65873  1.95352  0.64934
smoker_current           0.56870  0.28457  1.13649  0.11010
sees_pat                 0.84168  0.49317  1.43647  0.52741

estimated cut-points on the latent scale  -4.32801, 0.01979, 2.23613


### Cell 25 — A24: Proportional-odds assumption test

**Pseudocode.**

1. The proportional-odds model constrains each covariate to one coefficient shared
   across all cut-points. The unconstrained alternative is a multinomial model, which
   lets every coefficient vary by level.
2. Fit both on the same sample and compare them with a likelihood-ratio test. The
   degrees of freedom are the number of extra parameters the multinomial model
   spends.
3. A non-significant test means the shared-coefficient constraint is not contradicted
   by these data.


In [25]:
y = D2.ili_freq_ord4.values
X = sm.add_constant(D2[V2_TERMS].astype(float).values)
mn = sm.MNLogit(y, X).fit(disp=0, maxiter=2000)
ll_po, ll_mn = OM2.llf, mn.llf
k_po = len(OM2.params); k_mn = int(np.prod(mn.params.shape))
lr = 2 * (ll_mn - ll_po); df = k_mn - k_po
print(f"proportional-odds log-likelihood      {ll_po:.5f}  ({k_po} parameters)")
print(f"unconstrained multinomial             {ll_mn:.5f}  ({k_mn} parameters)")
print(f"likelihood-ratio test  chi-square {lr:.4f}, df {df}, p = {stats.chi2.sf(lr, df):.5f}")
print("a non-significant test leaves the proportional-odds constraint tenable")
record("A24", lr=float(lr), df=int(df), p=float(stats.chi2.sf(lr, df)))

proportional-odds log-likelihood      -211.45502  (11 parameters)
unconstrained multinomial             -201.55105  (27 parameters)
likelihood-ratio test  chi-square 19.8079, df 16, p = 0.22900
a non-significant test leaves the proportional-odds constraint tenable


### Cell 26 — A25: Susceptibility model fit and joint hypothesis test

**Pseudocode.**

1. McFadden's pseudo R-squared compares the fitted log-likelihood with that of an
   intercept-only ordinal model.
2. Ordinal concordance is the probability that a randomly chosen higher-level
   physician has a higher predicted latent score than a randomly chosen lower-level
   one, computed over all discordant pairs.
3. The hypothesis is a JOINT statement about the covariate block, so test it with a
   Wald test on the household, health-condition, contact and behavioural terms
   simultaneously rather than reading individual p-values.


In [26]:
om0 = OrderedModel(D2.ili_freq_ord4, np.zeros((len(D2), 0)), distr="logit").fit(disp=0)
mcf = 1 - OM2.llf / om0.llf
print(f"McFadden pseudo R-squared  {mcf:.5f}")
lin = (D2[V2_TERMS].values * OM2.params[V2_TERMS].values).sum(axis=1)
yv = D2.ili_freq_ord4.values
conc = disc = ties = 0
for i in range(len(yv)):
    for j in range(i + 1, len(yv)):
        if yv[i] == yv[j]:
            continue
        hi_, lo_ = (i, j) if yv[i] > yv[j] else (j, i)
        if lin[hi_] > lin[lo_]:
            conc += 1
        elif lin[hi_] < lin[lo_]:
            disc += 1
        else:
            ties += 1
c_index = (conc + 0.5 * ties) / (conc + disc + ties)
print(f"ordinal concordance        {c_index:.5f}  "
      f"({conc:,} concordant, {disc:,} discordant, {ties} tied pairs)")
JOINT = ["comp_risk_any_nan", "school_kids_any", "hh", "allergy"]
b = OM2.params[JOINT].values
V = OM2.cov_params().loc[JOINT, JOINT].values
W = float(b @ np.linalg.solve(V, b))
print(f"\njoint Wald test on {len(JOINT)} terms  chi-square {W:.4f}, df {len(JOINT)}, "
      f"p = {stats.chi2.sf(W, len(JOINT)):.5f}")
record("A25", mcfadden=float(mcf), concordance=float(c_index), joint_chi2=float(W),
       joint_df=len(JOINT), joint_p=float(stats.chi2.sf(W, len(JOINT))))

McFadden pseudo R-squared  0.05334
ordinal concordance        0.66240  (10,066 concordant, 5,128 discordant, 9 tied pairs)

joint Wald test on 4 terms  chi-square 15.7462, df 4, p = 0.00338


### Cell 27 — A26: Prior-season care-seeking model

**Pseudocode.**

1. The population is physicians who reported an influenza-like illness in the
   previous season, since only they could have sought care for one. Among reporting
   physicians that is 201; 195 of those answered the health-condition question.
2. The outcome is whether care was sought at a health facility.
3. The pre-specified adjustment set is age, sex, any health condition,
   susceptibility, prior-season vaccination, current tobacco use and respiratory
   allergy.


In [27]:
elig = phys[(phys.cohort_248_reporter == 1) & (phys.had_ili_prev == 1)]
d3 = elig.dropna(subset=["comp_risk_any_nan"])
F3 = ("sought_care ~ age10 + female + comp_risk_any_nan + ili_freq_ord4 + prev_vax "
      "+ smoker_current + allergy")
fit3 = smf.logit(F3, data=d3).fit(disp=0)
print(f"eligible (reported prior-season illness) {len(elig)}; "
      f"in the model {len(d3)}; events {int(d3.sought_care.sum())}")
print(pd.DataFrame({"OR": np.exp(fit3.params), "lo": np.exp(fit3.conf_int()[0]),
                    "hi": np.exp(fit3.conf_int()[1]), "p": fit3.pvalues}).round(5).to_string())
record("A26", n_eligible=len(elig), n=len(d3), events=int(d3.sought_care.sum()),
       or_susc=float(np.exp(fit3.params["ili_freq_ord4"])),
       p_susc=float(fit3.pvalues["ili_freq_ord4"]),
       or_age=float(np.exp(fit3.params["age10"])), p_age=float(fit3.pvalues["age10"]),
       or_smoke=float(np.exp(fit3.params["smoker_current"])),
       p_smoke=float(fit3.pvalues["smoker_current"]),
       or_prev_vax=float(np.exp(fit3.params["prev_vax"])),
       p_prev_vax=float(fit3.pvalues["prev_vax"]))
D3, FIT3, F3_FORMULA = d3, fit3, F3

eligible (reported prior-season illness) 201; in the model 195; events 53
                        OR       lo       hi        p
Intercept          0.01786  0.00272  0.11738  0.00003
age10              1.41559  1.01200  1.98014  0.04240
female             1.49938  0.68949  3.26058  0.30682
comp_risk_any_nan  1.91038  0.83567  4.36721  0.12493
ili_freq_ord4      2.21847  1.28055  3.84336  0.00448
prev_vax           0.27900  0.12376  0.62896  0.00208
smoker_current     2.34109  1.01002  5.42632  0.04734
allergy            1.84459  0.90964  3.74051  0.08962


### Cell 28 — A27: Care-seeking calibration, discrimination and joint test

**Pseudocode.**

1. Apparent area under the curve, then Efron optimism correction with a thousand
   refitted replicates.
2. Hosmer-Lemeshow test on groups of fitted risk.
3. Joint Wald test on the block of terms the hypothesis names, rather than reading
   any single coefficient as the answer.


In [28]:
app, opt, corr, nrep = efron_optimism(F3_FORMULA, D3, "sought_care", B=1000, seed=20260707)
print(f"apparent AUC             {app:.5f}")
print(f"Efron optimism           {opt:.5f}   ({nrep} replicates, seed 20260707)")
print(f"optimism-corrected AUC   {corr:.5f}")
p_hat = FIT3.predict(D3)
grp = pd.qcut(p_hat, 10, labels=False, duplicates="drop")
obs = D3.sought_care.groupby(grp).sum(); exp = p_hat.groupby(grp).sum(); n_g = p_hat.groupby(grp).size()
hl = (((obs - exp) ** 2) / (exp * (1 - exp / n_g))).sum(); df_hl = n_g.size - 2
print(f"\nHosmer-Lemeshow  chi-square {hl:.4f}, df {df_hl}, p = {stats.chi2.sf(hl, df_hl):.4f}")
JOINT = ["age10", "comp_risk_any_nan", "ili_freq_ord4", "prev_vax", "smoker_current"]
b = FIT3.params[JOINT].values; V = FIT3.cov_params().loc[JOINT, JOINT].values
W = float(b @ np.linalg.solve(V, b))
print(f"joint Wald test on {len(JOINT)} terms  chi-square {W:.4f}, df {len(JOINT)}, "
      f"p = {stats.chi2.sf(W, len(JOINT)):.6f}")
record("A27", apparent=app, optimism=opt, corrected=corr, joint_chi2=float(W),
       joint_df=len(JOINT), joint_p=float(stats.chi2.sf(W, len(JOINT))))

apparent AUC             0.74302
Efron optimism           0.04615   (1000 replicates, seed 20260707)
optimism-corrected AUC   0.69687

Hosmer-Lemeshow  chi-square 14.6512, df 8, p = 0.0663
joint Wald test on 5 terms  chi-square 21.1053, df 5, p = 0.000774


### Cell 29 — A33: A-versus-B cluster contrast (ratio of hazard ratios)

**Pseudocode.**

1. The A-cluster contrast estimates vaccine effectiveness against influenza-like
   episodes; the B-cluster contrast is a negative control with no influenza
   interpretation.
2. The ratio of the two hazard ratios asks whether vaccination acts differentially on
   the influenza-like cluster. Its logarithm is the difference of the two log hazard
   ratios.
3. The two estimates come from the same physicians, so their covariance is not zero;
   treating them as independent gives a conservative delta-method interval, which is
   what is reported.
4. The two cluster-specific hazard ratios and their standard errors are the estimates
   of record produced by the frailty models in the R notebook and are entered here as
   inputs.


In [29]:
# estimates of record from the shared gamma-frailty models (R notebook)
HR_A, SE_LOG_HR_A = 0.5368, 0.2772
HR_B, CI_B = 1.0783, (0.8065, 1.4417)
SE_LOG_HR_B = (np.log(CI_B[1]) - np.log(CI_B[0])) / (2 * 1.959964)
log_rhr = np.log(HR_A) - np.log(HR_B)
se_rhr = np.sqrt(SE_LOG_HR_A ** 2 + SE_LOG_HR_B ** 2)
rhr = np.exp(log_rhr)
lo, hi = np.exp(log_rhr - 1.96 * se_rhr), np.exp(log_rhr + 1.96 * se_rhr)
p = 2 * stats.norm.sf(abs(log_rhr) / se_rhr)
print(f"A-cluster hazard ratio  {HR_A:.4f}  (se of log HR {SE_LOG_HR_A:.4f})")
print(f"B-cluster hazard ratio  {HR_B:.4f}  (se of log HR {SE_LOG_HR_B:.4f})")
print(f"ratio of hazard ratios  {rhr:.4f} ({lo:.3f}-{hi:.3f}), p = {p:.4f}")
print("the interval treats the two estimates as independent, which is conservative "
      "because they are estimated on the same physicians")
record("A33", rhr=float(rhr), lo=float(lo), hi=float(hi), p=float(p))

A-cluster hazard ratio  0.5368  (se of log HR 0.2772)
B-cluster hazard ratio  1.0783  (se of log HR 0.1482)
ratio of hazard ratios  0.4978 (0.269-0.922), p = 0.0265
the interval treats the two estimates as independent, which is conservative because they are estimated on the same physicians


### Cell 30 — A34: Exposed-event accounting for the vaccine contrast

**Pseudocode.**

1. For every incident episode determine whether its onset week fell under vaccine
   protection, that is two or more weeks after the physician's vaccination week.
2. Cross-tabulate episodes by agent cluster and exposure state.
3. Report the exposed-event count in each cluster. The A-cluster exposed count is the
   quantity that governs the precision of the vaccine-effectiveness estimate, so it is
   reported with the estimate everywhere.


In [30]:
ct = pd.crosstab(episodes.cluster_letter, episodes.under_vax_protection,
                 margins=True, margins_name="all")
ct.columns = ["unexposed", "under vaccine protection", "all"][:ct.shape[1]]
print(ct.to_string())
print("\nexposed events by outcome")
print(f"  all-ILI    {int(episodes.under_vax_protection.sum())} of {len(episodes)}")
for c in "ABCD":
    sub = episodes[episodes.cluster_letter == c]
    print(f"  cluster {c}  {int(sub.under_vax_protection.sum())} of {len(sub)}")
record("A34", all_exposed=int(episodes.under_vax_protection.sum()),
       **{f"exposed_{c}": int(episodes[episodes.cluster_letter == c].under_vax_protection.sum())
          for c in "ABCD"})

                unexposed  under vaccine protection  all
cluster_letter                                          
A                      70                        21   91
B                     219                       122  341
C                      30                        13   43
D                      14                         8   22
all                   333                       164  497

exposed events by outcome
  all-ILI    164 of 497
  cluster A  21 of 91
  cluster B  122 of 341
  cluster C  13 of 43
  cluster D  8 of 22


### Cell 31 — A39: Joint test of the four occupational terms

**Pseudocode.**

1. The occupational hypothesis is a joint statement about four exposures — daily
   patient volume, monthly on-call shifts, mask use and aerosol-generating procedure
   share — so it is tested as a block on four degrees of freedom.
2. MAIN EFFECTS ONLY. No interaction term is fitted: one mask-by-aerosol cell holds a
   single physician, so no interaction is estimable.
3. Fit the count model on the occupational subgroup and take the Wald statistic for
   the four occupational coefficients jointly from the fitted covariance matrix.
4. The corresponding test on the frailty survival model is reported in the R notebook;
   both are reported so that the joint conclusion does not rest on one estimator.


In [31]:
occ_ids = phys.loc[(phys.cohort_112_primary == 1) & phys.pat_per_day.notna(), "participant_id"]
O = pw[pw.participant_id.isin(occ_ids)]
Oa = O.groupby("participant_id").agg(
    episodes=("new_episode", "sum"), at_risk=("at_risk_new_episode", "sum"),
    patients=("pat_per_day", "first"), oncall=("oncall_n", "first"),
    mask_use=("mask", "first"), aerosol=("aerosol_ord", "first"),
    age10=("age10", "first"), school_kids=("school_kids_any", "first"),
    susceptibility=("ili_freq_ord4", "first")).dropna()
TERMS = ["patients", "oncall", "mask_use", "aerosol", "age10", "school_kids", "susceptibility"]
X = sm.add_constant(Oa[TERMS])
nb = sm.NegativeBinomial(Oa.episodes, X, offset=np.log(Oa.at_risk),
                         loglike_method="nb2").fit(disp=0)
OCC4 = ["patients", "oncall", "mask_use", "aerosol"]
b = nb.params[OCC4].values; V = nb.cov_params().loc[OCC4, OCC4].values
W = float(b @ np.linalg.solve(V, b))
print(f"occupational subgroup: {len(Oa)} physicians, {int(Oa.at_risk.sum()):,} at-risk "
      f"person-weeks, {int(Oa.episodes.sum())} episodes")
print("no interaction terms are fitted: one mask-by-aerosol cell holds a single physician")
print(f"\njoint Wald test on the four occupational terms  chi-square {W:.4f}, df 4, "
      f"p = {stats.chi2.sf(W, 4):.4f}")
print("\nindividual occupational terms from the same fit (incidence-rate ratios)")
print(pd.DataFrame({"IRR": np.exp(nb.params[OCC4]), "lo": np.exp(nb.conf_int()[0][OCC4]),
                    "hi": np.exp(nb.conf_int()[1][OCC4]),
                    "p": nb.pvalues[OCC4]}).round(5).to_string())
record("A39", chi2=float(W), df=4, p=float(stats.chi2.sf(W, 4)), n=len(Oa),
       at_risk=int(Oa.at_risk.sum()), episodes=int(Oa.episodes.sum()))
OCC_LEVEL, OCC_NB, OCC_TERMS = Oa, nb, TERMS

occupational subgroup: 110 physicians, 1,724 at-risk person-weeks, 199 episodes
no interaction terms are fitted: one mask-by-aerosol cell holds a single physician

joint Wald test on the four occupational terms  chi-square 1.6343, df 4, p = 0.8026

individual occupational terms from the same fit (incidence-rate ratios)
              IRR       lo       hi        p
patients  1.00290  0.99358  1.01230  0.54333
oncall    0.99595  0.90435  1.09682  0.93423
mask_use  0.84288  0.54659  1.29977  0.43923
aerosol   1.13174  0.90030  1.42268  0.28906


### Cell 32 — A40: Recurrence count model in the occupational subgroup

**Pseudocode.**

1. Model each physician's episode count with negative-binomial regression and a log
   at-risk-week offset, on the same occupational specification, main effects only.
2. Report incidence-rate ratios for every term, and the dispersion parameter with its
   standard error; the size parameter is its reciprocal.
3. Fit the Poisson model on the same specification for comparison, and compare them
   with a likelihood-ratio test, which is the formal statement that overdispersion is
   present.


In [32]:
Oa, nb, TERMS = OCC_LEVEL, OCC_NB, OCC_TERMS
print(pd.DataFrame({"IRR": np.exp(nb.params[TERMS]), "lo": np.exp(nb.conf_int()[0][TERMS]),
                    "hi": np.exp(nb.conf_int()[1][TERMS]),
                    "p": nb.pvalues[TERMS]}).round(5).to_string())
alpha = nb.params["alpha"]
print(f"\noverdispersion parameter alpha  {alpha:.5f} (SE {nb.bse['alpha']:.5f})")
print(f"size parameter 1/alpha          {1/alpha:.4f}")
po = smf.glm("episodes ~ " + " + ".join(TERMS), data=Oa, family=sm.families.Poisson(),
             offset=np.log(Oa.at_risk)).fit()
lr = 2 * (nb.llf - po.llf)
print(f"\nPoisson comparison on the same specification: log-likelihood {po.llf:.4f} "
      f"against {nb.llf:.4f}")
print(f"likelihood-ratio test for overdispersion  chi-square {lr:.4f}, "
      f"p = {0.5*stats.chi2.sf(lr,1):.6f} (one-sided, boundary parameter)")
print("Poisson incidence-rate ratios for the same terms")
print(pd.DataFrame({"IRR": np.exp(po.params[TERMS]), "p": po.pvalues[TERMS]}).round(5).to_string())
record("A40", alpha=float(alpha), se_alpha=float(nb.bse["alpha"]), size=float(1/alpha),
       irr_school=float(np.exp(nb.params["school_kids"])),
       irr_susc=float(np.exp(nb.params["susceptibility"])), lr_overdisp=float(lr))

                    IRR       lo       hi        p
patients        1.00290  0.99358  1.01230  0.54333
oncall          0.99595  0.90435  1.09682  0.93423
mask_use        0.84288  0.54659  1.29977  0.43923
aerosol         1.13174  0.90030  1.42268  0.28906
age10           0.84494  0.69620  1.02546  0.08811
school_kids     1.89318  1.26486  2.83362  0.00192
susceptibility  1.44190  1.07913  1.92663  0.01332

overdispersion parameter alpha  0.30468 (SE 0.11657)
size parameter 1/alpha          3.2821

Poisson comparison on the same specification: log-likelihood -186.7500 against -178.7768
likelihood-ratio test for overdispersion  chi-square 15.9464, p = 0.000033 (one-sided, boundary parameter)
Poisson incidence-rate ratios for the same terms
                    IRR        p
patients        1.00314  0.37062
oncall          0.99572  0.90797
mask_use        0.84150  0.29305
aerosol         1.16637  0.09041
age10           0.83913  0.02764
school_kids     1.82750  0.00008
susceptibility  1.4364

## S3.3 System-level analysis

### Cell 33 — A41: Season-level agreement of the two positivity series

**Pseudocode.**

1. The participatory positivity is A-cluster episodes over all incident episodes,
   pooled across the season. The sentinel positivity is influenza-positive specimens
   over influenza-like-illness consultations, pooled the same way.
2. Attach an exact binomial interval to each pooled proportion.
3. The season-level difference is the participatory proportion minus the sentinel
   proportion, in percentage points. It is a pooled quantity and is not the same as
   the mean of the weekly differences, which the limits-of-agreement block reports.


In [33]:
k_p, n_p = int(wk.n_new_A.sum()), int(wk.n_new_episodes.sum())
k_s, n_s = int(wk.sb_positive_cases.sum()), int(wk.sb_ili_consultations.sum())
for lab, k, n in [("participatory", k_p, n_p), ("sentinel", k_s, n_s)]:
    lo, hi = exact_ci(k, n)
    print(f"{lab:14s} {k:5d} of {n:6,} = {100*k/n:7.4f}%  (95% CI {100*lo:.3f}-{100*hi:.3f})")
diff = 100*k_p/n_p - 100*k_s/n_s
print(f"season-level difference (participatory minus sentinel)  {diff:+.4f} percentage points")
record("A41", part_pct=100*k_p/n_p, sent_pct=100*k_s/n_s, sent_consults=n_s,
       sent_positives=k_s, diff_pp=diff)

participatory     91 of    497 = 18.3099%  (95% CI 15.006-21.996)
sentinel         739 of  3,779 = 19.5554%  (95% CI 18.302-20.857)
season-level difference (participatory minus sentinel)  -1.2456 percentage points


### Cell 34 — A42: Measurement reliability and the attenuation ceiling

**Pseudocode.**

1. Each weekly positivity is a proportion measured on a finite denominator, so its
   observed variance is signal variance plus binomial sampling error.
2. Estimate the binomial error variance as the mean over weeks of p(1-p)/n, and take
   reliability as the signal share of the observed variance. The complement is the
   noise share.
3. The attenuation ceiling on the correlation between the two series is the
   geometric mean of the two reliabilities. It is computed on the RAW weekly series,
   never on the smoothed ones, because smoothing changes the error variance.


In [34]:
def reliability(k, n):
    p = k / n
    var_obs = p.var(ddof=1)
    var_err = (p * (1 - p) / n).mean()
    rel = max(0.0, var_obs - var_err) / var_obs
    return rel, var_err / var_obs, var_obs, var_err

rel_p, noise_p, vo_p, ve_p = reliability(wk.n_new_A, wk.n_new_episodes)
rel_s, noise_s, vo_s, ve_s = reliability(wk.sb_positive_cases, wk.sb_ili_consultations)
ceiling = np.sqrt(rel_p * rel_s)
print("computed on the raw weekly series, not the smoothed ones")
print(f"participatory  reliability {rel_p:.5f}   noise share {100*noise_p:.3f}%")
print(f"sentinel       reliability {rel_s:.5f}   noise share {100*noise_s:.3f}%")
print(f"attenuation ceiling on the correlation  {ceiling:.5f}")
record("A42", rel_part=rel_p, rel_sent=rel_s, noise_part=100*noise_p,
       noise_sent=100*noise_s, ceiling=ceiling)
RELIABILITY_PART, RELIABILITY_SENT, CEILING = rel_p, rel_s, ceiling

computed on the raw weekly series, not the smoothed ones
participatory  reliability 0.29331   noise share 70.669%
sentinel       reliability 0.96197   noise share 3.803%
attenuation ceiling on the correlation  0.53118


### Cell 35 — A43: Phase-stratified concordance of the two positivity series

**Pseudocode.**

1. Smooth both series with a causal three-week moving average covering weeks t-2 to
   t. Partial windows at the season's opening are retained rather than dropped;
   retaining them is part of the specification because it changes the full-season
   coefficient.
2. Within each Moving Epidemic Method phase, and over the pre-specified
   early-warning window and the full season, compute the Pearson and Spearman
   correlation between the two smoothed series.
3. Compute the same Pearson correlations on the raw weekly series and report them
   beside the smoothed ones. The smoothed coefficients are the estimates of record;
   the raw ones are reported so the effect of smoothing is visible.
4. Attach a moving-block bootstrap interval to the early-warning estimate.
5. Recompute the post-epidemic coefficient with the flagged closing week omitted,
   as a disclosure of that week's influence.


In [35]:
p_sm, s_sm = cma3(part_share), cma3(sent_share)
windows = [("pre-epidemic", PRE), ("epidemic", EPI), ("post-epidemic", POST),
           ("early-warning window", EARLY), ("full season", SEASON)]
rows = []
for lab, m in windows:
    r_s, p_s = stats.pearsonr(p_sm[m], s_sm[m])
    rho, p_rho = stats.spearmanr(p_sm[m], s_sm[m])
    r_r, p_r = stats.pearsonr(part_share[m], sent_share[m])
    rows.append(dict(window=lab, n=int(m.sum()), pearson_smoothed=round(r_s, 5),
                     p_smoothed=round(p_s, 6), spearman_smoothed=round(rho, 5),
                     pearson_raw=round(r_r, 5), p_raw=round(p_r, 6)))
print(pd.DataFrame(rows).to_string(index=False))

def block_bootstrap_r(x, y, B=2000, block=4, seed=20260707):
    """Moving-block bootstrap of the correlation; blocks are NON-circular."""
    rng = np.random.default_rng(seed)
    x, y, n = np.asarray(x), np.asarray(y), len(x)
    out = []
    for _ in range(B):
        idx = []
        while len(idx) < n:
            st = rng.integers(0, max(1, n - block + 1))
            idx.extend(range(st, min(st + block, n)))
        idx = np.array(idx[:n])
        xx, yy = x[idx], y[idx]
        if xx.std() > 0 and yy.std() > 0:
            out.append(np.corrcoef(xx, yy)[0, 1])
    return np.percentile(out, [2.5, 97.5])
lo, hi = block_bootstrap_r(p_sm[EARLY], s_sm[EARLY])
r_ew = stats.pearsonr(p_sm[EARLY], s_sm[EARLY])[0]
print(f"\nearly-warning window (primary): r = {r_ew:+.5f}  "
      f"moving-block bootstrap 95% CI {lo:+.3f} to {hi:+.3f} (block length 4, non-circular, B = 2000)")
m2 = POST & (ISO_WEEK.values < 20)
print(f"post-epidemic excluding the flagged closing week: r = "
      f"{stats.pearsonr(p_sm[m2], s_sm[m2])[0]:+.5f}  (n = {int(m2.sum())})")
record("A43", r_pre=rows[0]["pearson_smoothed"], r_epi=rows[1]["pearson_smoothed"],
       r_post=rows[2]["pearson_smoothed"], r_ew=r_ew, r_season=rows[4]["pearson_smoothed"],
       raw_pre=rows[0]["pearson_raw"], raw_epi=rows[1]["pearson_raw"],
       raw_post=rows[2]["pearson_raw"], raw_season=rows[4]["pearson_raw"],
       r_post_excl=float(stats.pearsonr(p_sm[m2], s_sm[m2])[0]))
PART_SM, SENT_SM = p_sm, s_sm

              window  n  pearson_smoothed  p_smoothed  spearman_smoothed  pearson_raw    p_raw
        pre-epidemic 10           0.88609    0.000641            0.87879      0.52384 0.120158
            epidemic  9           0.75414    0.018893            0.63333      0.43690 0.239659
       post-epidemic 14           0.04696    0.873347            0.03297     -0.22590 0.437424
early-warning window 19           0.64196    0.003044            0.75439      0.48907 0.033585
         full season 33           0.53804    0.001240            0.63937      0.24258 0.173770

early-warning window (primary): r = +0.64196  moving-block bootstrap 95% CI +0.351 to +0.913 (block length 4, non-circular, B = 2000)
post-epidemic excluding the flagged closing week: r = +0.28460  (n = 13)


### Cell 36 — A46: Lead-lag cross-correlation profile

**Pseudocode.**

1. Fix the sign convention: the participatory series is entered first, so a
   positive lag means the participatory series LEADS the sentinel.
2. For each lag from minus four to plus four weeks, pair the two smoothed series at
   that lag and compute the Pearson correlation on the overlapping weeks only.
3. Report the whole profile and locate its point maximum. The point maximum is not
   by itself an identified lead time; the bootstrap block settles identifiability.


In [36]:
def lag_pairs(a, b, L):
    """Positive L = participatory leads: participatory at t against sentinel at t+L."""
    a, b = np.asarray(a), np.asarray(b)
    if L >= 0:
        return a[:len(a) - L], b[L:]
    return a[-L:], b[:len(b) + L]

LAGS = range(-4, 5)
prof = {}
for L in LAGS:
    x, y = lag_pairs(PART_SM, SENT_SM, L)
    prof[L] = stats.pearsonr(x, y)[0]
for L, r in prof.items():
    mark = "  <- point maximum" if r == max(prof.values()) else ""
    print(f"lag {L:+d}  n = {len(lag_pairs(PART_SM, SENT_SM, L)[0]):2d}  r = {r:+.5f}{mark}")
peak = max(prof, key=prof.get)
near = {L: prof[L] for L in prof if abs(prof[L] - prof[peak]) < 0.06}
print(f"\npoint profile peaks at lag {peak:+d} (r = {prof[peak]:.5f}); the neighbouring lags "
      f"{sorted(near)} run {min(near.values()):.3f} to {max(near.values()):.3f}, "
      "so the profile is flat across the central lags")
record("A46", profile={int(k): round(v, 5) for k, v in prof.items()}, peak_lag=int(peak),
       peak_r=prof[peak])
LAG_PROFILE = prof

lag -4  n = 29  r = +0.36569
lag -3  n = 30  r = +0.52844
lag -2  n = 31  r = +0.58738  <- point maximum
lag -1  n = 32  r = +0.55777
lag +0  n = 33  r = +0.53804
lag +1  n = 32  r = +0.57464
lag +2  n = 31  r = +0.42356
lag +3  n = 30  r = +0.22912
lag +4  n = 29  r = -0.01188

point profile peaks at lag -2 (r = 0.58738); the neighbouring lags [-3, -2, -1, 0, 1] run 0.528 to 0.587, so the profile is flat across the central lags


### Cell 37 — A47: Peak-lag identifiability under the lag-preserving bootstrap

**Pseudocode.**

1. The quantity resampled is the whole lag procedure, not the correlation at one
   lag, so the bootstrap must preserve which sentinel week each participatory week
   is paired with.
2. Therefore: form the lagged pairs FIRST, one pair series per lag, then resample
   PAIRS within each lag in non-circular blocks of four weeks. Non-circularity is
   load-bearing — a circular-wrap variant moves the modal lag.
3. In each replicate recompute the whole profile and record which lag attains the
   maximum. The bootstrap distribution of that argmax is the answer.
4. Report the modal lag, its share, the share at lag zero and the 95% interval, and
   repeat across several seeds so that seed sensitivity is visible rather than hidden.


In [37]:
def block_index(rng, n, block, circular=False):
    idx = []
    while len(idx) < n:
        if circular:
            st = rng.integers(0, n)
            idx.extend([(st + j) % n for j in range(block)])
        else:
            st = rng.integers(0, max(1, n - block + 1))
            idx.extend(range(st, min(st + block, n)))
    return np.array(idx[:n])

def peak_lag_bootstrap(a, b, lags=LAGS, B=2000, block=4, seed=20260707,
                       circular=False, preserve_pairing=True):
    rng = np.random.default_rng(seed)
    pairs = {L: lag_pairs(a, b, L) for L in lags}
    modal = []
    for _ in range(B):
        rs = {}
        if preserve_pairing:
            for L in lags:                       # resample PAIRS within each lag
                x, y = pairs[L]
                idx = block_index(rng, len(x), block, circular)
                xx, yy = x[idx], y[idx]
                if xx.std() > 0 and yy.std() > 0:
                    rs[L] = np.corrcoef(xx, yy)[0, 1]
        else:                                     # comparator: resample then re-pair
            idx = block_index(rng, len(a), block, circular)
            aa, bb = pd.Series(np.asarray(a)[idx]), pd.Series(np.asarray(b)[idx])
            for L in lags:
                x, y = lag_pairs(aa, bb, L)
                if x.std() > 0 and y.std() > 0:
                    rs[L] = np.corrcoef(x, y)[0, 1]
        if rs:
            modal.append(max(rs, key=rs.get))
    return pd.Series(modal)

print("procedure of record: lagged pairs formed first, then PAIRS resampled in "
      "NON-CIRCULAR blocks of 4 weeks, B = 2000")
SEEDS = (7, 42, 2026, 20260707)
modes, shares = {}, {}
for seed in SEEDS:
    s = peak_lag_bootstrap(PART_SM, SENT_SM, seed=seed)
    vc = s.value_counts(normalize=True)
    mode = int(s.mode().iloc[0])
    modes[seed], shares[seed] = mode, 100 * vc[mode]
    print(f"  seed {seed:>8}  modal lag {mode:+d} in {100*vc[mode]:.1f}% of resamples; "
          f"lag 0 in {100*vc.get(0,0):.1f}%; 95% interval "
          f"[{int(s.quantile(.025)):+d}, {int(s.quantile(.975)):+d}]")
s = peak_lag_bootstrap(PART_SM, SENT_SM, seed=20260707)
vc = s.value_counts(normalize=True)
mode = int(s.mode().iloc[0])
tally = pd.Series(list(modes.values())).value_counts()
print(f"\nthe modal lag is NOT stable across seeds: " +
      ", ".join(f"lag {int(L):+d} in {int(c)} of {len(SEEDS)} seeds" for L, c in tally.items()) +
      f". No modal lag holds more than {max(shares.values()):.0f}% of resamples, the "
      "runner-up lags are within a few percentage points of it, and each 95% interval "
      "spans several weeks. NO LEAD TIME IS IDENTIFIABLE from these data, and the "
      "instability across seeds is itself part of that conclusion rather than a "
      "nuisance to be averaged away.")
sc = peak_lag_bootstrap(PART_SM, SENT_SM, seed=20260707, circular=True)
print(f"circular-wrap variant, for contrast: modal lag "
      f"{int(sc.mode().iloc[0]):+d} in {100*sc.value_counts(normalize=True).iloc[0]:.1f}% "
      "— the block rule changes the answer, which is why non-circular blocks are specified")
record("A47", modal_lag=mode, modal_share=100*vc[mode], lag0_share=100*vc.get(0, 0),
       ci=[int(s.quantile(.025)), int(s.quantile(.975))],
       circular_modal=int(sc.mode().iloc[0]),
       modes_by_seed={int(k): int(v) for k, v in modes.items()},
       shares_by_seed={int(k): round(v, 1) for k, v in shares.items()})

procedure of record: lagged pairs formed first, then PAIRS resampled in NON-CIRCULAR blocks of 4 weeks, B = 2000
  seed        7  modal lag -1 in 28.2% of resamples; lag 0 in 19.1%; 95% interval [-3, +1]
  seed       42  modal lag -1 in 27.5% of resamples; lag 0 in 18.4%; 95% interval [-3, +1]
  seed     2026  modal lag -2 in 27.6% of resamples; lag 0 in 16.8%; 95% interval [-3, +2]
  seed 20260707  modal lag -1 in 27.7% of resamples; lag 0 in 18.4%; 95% interval [-3, +2]

the modal lag is NOT stable across seeds: lag -1 in 3 of 4 seeds, lag -2 in 1 of 4 seeds. No modal lag holds more than 28% of resamples, the runner-up lags are within a few percentage points of it, and each 95% interval spans several weeks. NO LEAD TIME IS IDENTIFIABLE from these data, and the instability across seeds is itself part of that conclusion rather than a nuisance to be averaged away.
circular-wrap variant, for contrast: modal lag -2 in 21.8% — the block rule changes the answer, which is why non-circular bl

### Cell 38 — A49: Timing landmarks within the wave

**Pseudocode.**

1. Restrict attention to the wave, that is the pre-epidemic and epidemic weeks, so
   that a late off-season excursion in the participatory series cannot be read as
   the wave peak.
2. In each smoothed series locate the peak week within the epidemic window.
3. The half-maximum week is the first week in the epidemic window at which the
   smoothed series reaches half its within-window maximum.
4. Report the peak-to-peak difference in weeks and note that it is a landmark
   comparison, not an estimated lead time.


In [38]:
rows = []
for lab, s in [("participatory", PART_SM), ("sentinel", SENT_SM)]:
    v = np.asarray(s)[EPI]
    wks_epi = WEEKS[EPI]
    pk = int(np.argmax(v))
    half = v.max() / 2
    hm = int(np.argmax(v >= half))
    rows.append(dict(series=lab, peak_week=wks_epi[pk], peak_value=round(v[pk], 3),
                     half_max_week=wks_epi[hm], half_max_value=round(half, 3)))
print(pd.DataFrame(rows).to_string(index=False))
pk_p = WEEKS[EPI][int(np.argmax(np.asarray(PART_SM)[EPI]))]
pk_s = WEEKS[EPI][int(np.argmax(np.asarray(SENT_SM)[EPI]))]
i_p, i_s = list(WEEKS).index(pk_p), list(WEEKS).index(pk_s)
print(f"\npeak-to-peak difference  {i_p - i_s:+d} weeks (participatory {pk_p} against "
      f"sentinel {pk_s}); a landmark comparison, not an estimated lead time")
record("A49", peak_part=str(pk_p), peak_sent=str(pk_s),
       half_part=rows[0]["half_max_week"], half_sent=rows[1]["half_max_week"])

       series peak_week  peak_value half_max_week  half_max_value
participatory  2026-W05      28.497      2025-W51          14.248
     sentinel  2026-W03      55.547      2026-W01          27.773

peak-to-peak difference  +2 weeks (participatory 2026-W05 against sentinel 2026-W03); a landmark comparison, not an estimated lead time


### Cell 39 — A50: Bland-Altman limits of agreement

**Pseudocode.**

1. Take the weekly difference between the two positivity series, participatory
   minus sentinel, on the raw weekly values.
2. The mean difference is the bias and the limits of agreement are that mean plus
   and minus 1.96 standard deviations of the differences.
3. Report the mean weekly difference and the pooled season-level difference side by
   side; they are different quantities and are not interchangeable.


In [39]:
d = np.asarray(part_share) - np.asarray(sent_share)
mean_d, sd_d = d.mean(), d.std(ddof=1)
lo, hi = mean_d - 1.96*sd_d, mean_d + 1.96*sd_d
print(f"mean weekly difference   {mean_d:+.4f} percentage points (SD {sd_d:.4f})")
print(f"limits of agreement      {lo:+.3f} to {hi:+.3f} percentage points")
pooled = 100*wk.n_new_A.sum()/wk.n_new_episodes.sum() - 100*wk.sb_positive_cases.sum()/wk.sb_ili_consultations.sum()
print(f"pooled season difference {pooled:+.4f} percentage points  "
      "(a different quantity from the mean weekly difference)")
record("A50", mean_diff=mean_d, sd_diff=sd_d, loa_lo=lo, loa_hi=hi, pooled=pooled)

mean weekly difference   +2.7572 percentage points (SD 17.8052)
limits of agreement      -32.141 to +37.655 percentage points
pooled season difference -1.2456 percentage points  (a different quantity from the mean weekly difference)


### Cell 40 — A51: Calibration regression of one series on the other

**Pseudocode.**

1. Regress the sentinel positivity on the participatory positivity by ordinary least
   squares, so that the slope answers how many percentage points of sentinel
   positivity accompany one percentage point of participatory positivity.
2. Fit the same regression on the smoothed and on the raw series.
3. A slope far from one indicates the two series are not on a common scale; report
   the intercept with it.


In [40]:
for lab, x, y in [("smoothed (causal MA-3)", PART_SM, SENT_SM),
                  ("raw weekly", part_share, sent_share)]:
    r = stats.linregress(np.asarray(x), np.asarray(y))
    print(f"{lab:24s} slope {r.slope:+.5f}  intercept {r.intercept:+.5f}  "
          f"R-squared {r.rvalue**2:.4f}  p {r.pvalue:.6f}")
rs = stats.linregress(np.asarray(PART_SM), np.asarray(SENT_SM))
rr = stats.linregress(np.asarray(part_share), np.asarray(sent_share))
record("A51", slope_smoothed=rs.slope, intercept_smoothed=rs.intercept, slope_raw=rr.slope)

smoothed (causal MA-3)   slope +1.26556  intercept -7.01189  R-squared 0.2895  p 0.001240
raw weekly               slope +0.31725  intercept +10.45003  R-squared 0.0588  p 0.173770


### Cell 41 — A52: MEM epidemic-phase and intensity thresholds

**Pseudocode.**

1. The Moving Epidemic Method onset and intensity thresholds are fixed a-priori
   inputs estimated from ten historical sentinel seasons that this data release does
   not carry. They are stated as constants and are NOT derived from the season
   under analysis.
2. Compute the combined weekly positivity to which the onset threshold applies, and
   report every week that reaches it.
3. State the resulting phase calendar and confirm the phase week counts, which every
   phase-stratified analysis then uses without re-deriving them.


In [41]:
print(f"onset threshold      {MEM_ONSET}% combined positivity   (fixed a-priori input)")
print("intensity thresholds " + " / ".join(f"{t}%" for t in MEM_INTENSITY) +
      "   (medium / high / very high / extraordinary; fixed a-priori inputs)")
combined = 100 * (wk.n_new_A + wk.sb_positive_cases) / (wk.n_new_episodes + wk.sb_ili_consultations)
above = wk.week[combined >= MEM_ONSET].tolist()
print(f"\nweeks reaching the onset threshold  {len(above)} of {N_WEEKS}")
print("  " + ", ".join(above))
print("\nepidemic-phase calendar of record (fixed, not re-derived per analysis)")
print(f"  pre-epidemic   2025-W40 to 2025-W49   {int(PRE.sum())} weeks")
print(f"  epidemic       2025-W50 to 2026-W06   {int(EPI.sum())} weeks")
print(f"  post-epidemic  2026-W07 to 2026-W20   {int(POST.sum())} weeks")
print(f"  early-warning window (pre-epidemic and epidemic)  {int(EARLY.sum())} weeks")
print("\nmaximum combined positivity in the season "
      f"{combined.max():.2f}% in {wk.week[combined.idxmax()]}, which reaches the "
      f"{'high' if combined.max() >= MEM_INTENSITY[1] else 'medium'} intensity level")
record("A52", onset=MEM_ONSET, intensity=list(MEM_INTENSITY), n_pre=int(PRE.sum()),
       n_epi=int(EPI.sum()), n_post=int(POST.sum()), max_combined=float(combined.max()))

onset threshold      10.59% combined positivity   (fixed a-priori input)
intensity thresholds 22.04% / 42.56% / 60.92% / 71.38%   (medium / high / very high / extraordinary; fixed a-priori inputs)

weeks reaching the onset threshold  21 of 33
  2025-W40, 2025-W41, 2025-W42, 2025-W49, 2025-W50, 2025-W51, 2025-W52, 2026-W01, 2026-W02, 2026-W03, 2026-W04, 2026-W05, 2026-W06, 2026-W07, 2026-W08, 2026-W09, 2026-W11, 2026-W13, 2026-W15, 2026-W18, 2026-W20

epidemic-phase calendar of record (fixed, not re-derived per analysis)
  pre-epidemic   2025-W40 to 2025-W49   10 weeks
  epidemic       2025-W50 to 2026-W06   9 weeks
  post-epidemic  2026-W07 to 2026-W20   14 weeks
  early-warning window (pre-epidemic and epidemic)  19 weeks

maximum combined positivity in the season 55.13% in 2026-W02, which reaches the high intensity level


### Cell 42 — A53: Epidemic-week discrimination (AUC)

**Pseudocode.**

1. The label is membership of the Moving Epidemic Method epidemic window: nine of
   the thirty-three weeks.
2. Score four candidate indicators against that label: the smoothed and raw
   participatory positivity, the all-episode incidence rate on the AT-RISK
   denominator, and the smoothed sentinel positivity.
3. Attach a bootstrap interval by resampling weeks.
4. The sentinel row is a construction check, not external validation: the epidemic
   window was itself defined by thresholding the sentinel series, so its value is
   near-tautological and is labelled as such.
5. The incidence row is computed on the at-risk denominator and on the smoothed
   series; report the filed-denominator value beside it so the two are not confused.


In [42]:
label = EPI.astype(int)
rate_at_risk = wk.ili_rate_per1000_at_risk
rate_filed = wk.ili_rate_per1000_pw

def auc_ci(y, score, B=2000, seed=20260707):
    rng = np.random.default_rng(seed)
    y, score, n = np.asarray(y), np.asarray(score), len(y)
    out = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        if len(set(y[idx])) < 2:
            continue
        out.append(roc_auc_score(y[idx], score[idx]))
    return np.percentile(out, [2.5, 97.5])

rows = []
for lab, sc, note in [
        ("participatory positivity, smoothed", PART_SM, "indicator of record"),
        ("participatory positivity, raw", part_share, ""),
        ("all-episode ILI incidence per 1,000 at-risk person-weeks, smoothed",
         cma3(rate_at_risk), "at-risk denominator"),
        ("sentinel positivity, smoothed", SENT_SM, "construction check, not validation")]:
    a = roc_auc_score(label, np.asarray(sc))
    lo, hi = auc_ci(label, sc)
    rows.append(dict(indicator=lab, AUC=round(a, 5), ci_lo=round(lo, 3), ci_hi=round(hi, 3),
                     note=note))
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nfor comparison, the same incidence indicator on the FILED person-week denominator: "
      f"AUC {roc_auc_score(label, cma3(rate_filed)):.5f} smoothed, "
      f"{roc_auc_score(label, rate_filed):.5f} raw")
record("A53", auc_part_sm=rows[0]["AUC"], auc_part_raw=rows[1]["AUC"],
       auc_incidence_at_risk_sm=rows[2]["AUC"], auc_sent_sm=rows[3]["AUC"],
       auc_incidence_filed_sm=float(roc_auc_score(label, cma3(rate_filed))))

                                                         indicator     AUC  ci_lo  ci_hi                               note
                                participatory positivity, smoothed 0.77315  0.583  0.938                indicator of record
                                     participatory positivity, raw 0.69907  0.486  0.896                                   
all-episode ILI incidence per 1,000 at-risk person-weeks, smoothed 0.40741  0.206  0.624                at-risk denominator
                                     sentinel positivity, smoothed 0.98148  0.933  1.000 construction check, not validation

for comparison, the same incidence indicator on the FILED person-week denominator: AUC 0.39815 smoothed, 0.44213 raw


### Cell 43 — A54: Calendar-window discrimination check

**Pseudocode.**

1. Because the epidemic window is defined from the sentinel series, score the same
   sentinel indicator against a window that takes no sentinel input: a calendar
   window fixed on the northern-hemisphere influenza season.
2. Report the AUC against that calendar window beside the value against the Moving
   Epidemic Method window, so the circularity is visible where the number is read.
3. Also shift the locked epidemic window by one week in each direction, which shows
   how sensitive the near-tautological value is to the window's exact placement.


In [43]:
label_mem = EPI.astype(int)
cal = (((ISO_YEAR == 2025) & (ISO_WEEK >= 49)) | ((ISO_YEAR == 2026) & (ISO_WEEK <= 7))).values
print(f"MEM epidemic window        {int(label_mem.sum())} weeks   sentinel AUC "
      f"{roc_auc_score(label_mem, SENT_SM):.5f}   (construction check)")
print(f"calendar window W49-W07    {int(cal.sum())} weeks   sentinel AUC "
      f"{roc_auc_score(cal.astype(int), SENT_SM):.5f}   (no sentinel input)")
print("\nsensitivity of the MEM-window value to shifting the window")
for sh in (-1, 0, 1):
    lo_, hi_ = 50 + sh, 6 + sh
    m = (((ISO_YEAR == 2025) & (ISO_WEEK >= lo_)) | ((ISO_YEAR == 2026) & (ISO_WEEK <= hi_))).values
    print(f"  window shifted {sh:+d} week(s): {int(m.sum())} weeks, sentinel AUC "
          f"{roc_auc_score(m.astype(int), SENT_SM):.5f}, participatory AUC "
          f"{roc_auc_score(m.astype(int), PART_SM):.5f}")
record("A54", auc_calendar=float(roc_auc_score(cal.astype(int), SENT_SM)),
       auc_mem=float(roc_auc_score(label_mem, SENT_SM)))

MEM epidemic window        9 weeks   sentinel AUC 0.98148   (construction check)
calendar window W49-W07    11 weeks   sentinel AUC 0.91322   (no sentinel input)

sensitivity of the MEM-window value to shifting the window
  window shifted -1 week(s): 9 weeks, sentinel AUC 0.87037, participatory AUC 0.71296
  window shifted +0 week(s): 9 weeks, sentinel AUC 0.98148, participatory AUC 0.77315
  window shifted +1 week(s): 9 weeks, sentinel AUC 1.00000, participatory AUC 0.83333


### Cell 44 — A55: Six-detector aberration panel under the locked specification

**Pseudocode.**

1. Six detectors run on each of the two series: three EARS control charts, a
   negative-binomial CUSUM, an exponentially weighted moving average and a
   nonparametric percentile threshold. Farrington Flexible is not part of the bank: it
   requires several years of historical baseline and one season of data cannot supply
   it.
2. Both series enter the detection block as causal three-week moving averages, weeks
   t-2 to t. The rationale is the participatory denominator: fourteen to seventeen
   incident episodes in a typical week, so a single episode moves the A-cluster share
   by about six percentage points and the raw weekly series is dominated by sampling
   noise rather than by epidemic signal. Averaging over three weeks reduces that noise
   without looking ahead, which a detector intended for prospective use may not do.
3. EARS C1 standardizes the current value against the mean and standard deviation of
   the seven preceding weeks and alarms when the statistic exceeds three.
4. EARS C2 is the same statistic on a baseline shifted back: its seven baseline weeks
   END TWO WEEKS BEFORE the week being tested. The two intervening weeks form a guard
   band, so a rise already under way does not inflate the baseline it is judged
   against. C2 alarms when its statistic exceeds three.
5. EARS C3 accumulates, over the week being tested and the two weeks before it, the
   part of the C2 statistic that lies ABOVE ONE standard deviation, counting only
   positive excess, and alarms when that three-week sum exceeds two. C3 INHERITS C2'S
   TWO-WEEK GUARD BAND: the statistic it accumulates is the guarded one. Without the
   guard band the detector alarms at the wrong weeks in both series.
6. The count detector is a negative-binomial likelihood-ratio CUSUM on the weekly
   COUNTS of each stream — incident A-cluster episodes on the participatory side,
   positive specimens on the sentinel side — not on the smoothed share. Its in-control
   mean is the mean of the seven preceding counts, its dispersion a method-of-moments
   estimate on the same window, and its out-of-control mean twice the in-control mean.
   The statistic accumulates the negative-binomial log-likelihood ratio of the two
   means, is floored at zero, and RESETS THE ACCUMULATOR TO ZERO after an alarm, so one
   excursion raises one alarm.
7. The CUSUM decision intervals are MONTE-CARLO CALIBRATED and SERIES-SPECIFIC: each
   was fixed by simulation from that series' own fitted in-control negative binomial so
   that both streams carry a common false-alarm rate. Because the two count series
   differ in level and dispersion, one shared interval would not do this; the
   participatory interval is 1.40 and the sentinel interval 2.52, and they are not
   interchangeable.
8. The exponentially weighted moving average carries a smoothing weight of 0.4 and
   alarms when its statistic exceeds the baseline mean by three standard deviations of
   the statistic, that deviation being the seven-week baseline deviation scaled by the
   square root of the weight divided by two minus the weight. THE CONTROL LIMIT
   RE-ARMS TO THE BASELINE MEAN AFTER AN ALARM: on alarming, the statistic is set back
   to the baseline mean. Because the statistic carries its own history forward, an
   unre-armed chart stays above the limit and alarms in consecutive weeks after a
   single excursion; the re-arm is what makes one excursion raise one alarm.
9. The percentile detector alarms when the current value exceeds the ninety-fifth
   percentile of the PRIOR TEN WEEKS, a window that slides with the week being tested
   and does not expand over the season to date, so the reference distribution stays
   local to the recent past.
10. Every detector observes a seven-week warm-up in which no alarm can be raised, so
    the season's first seven weeks cannot alarm under any detector.
11. The consensus rule of record is at least one detector of six. A stricter rule of at
    least two is carried alongside and reported separately. Report the per-detector
    matrix, the per-detector alarm weeks and the consensus alarm weeks for both series.

In [44]:
# ============================================================================
# Aberration detector bank — the locked specification (Table 2.2).
# Six detectors, all on the CAUSAL three-week moving average of the monitored
# series, a seven-week rolling baseline and a seven-week warm-up. Every constant
# carries the source of its value; no literal appears inline below.
# ============================================================================
BASELINE_WEEKS  = 7      # rolling baseline of the EARS charts, the EWMA control
                         # limit and the NB-CUSUM in-control mean
WARMUP_WEEKS    = 7      # no detector may alarm in the season's first seven weeks
SD_THRESHOLD    = 3.0    # EARS C1 and C2 alarm above three standard deviations
GUARD_WEEKS     = 2      # C2's guard band: the baseline ends two weeks before t.
                         # C3 INHERITS THIS GUARD BAND.
C3_EXCESS_SD    = 1.0    # C3 accumulates the C2 excess measured ABOVE one SD
C3_THRESHOLD    = 2.0    # ...and alarms when the three-week sum exceeds two
C3_WINDOW       = 3      # the current week and the prior two
EWMA_LAMBDA     = 0.4    # exponential smoothing weight
EWMA_L          = 3.0    # EWMA control limit in standard deviations
PCT_Q           = 95     # empirical percentile detector
PCT_HIST        = 10     # ...over the PRIOR TEN WEEKS, a window that slides
NBCUSUM_MULT    = 2.0    # NB-CUSUM out-of-control mean = twice the baseline mean
# Decision intervals are MONTE-CARLO CALIBRATED and SERIES-SPECIFIC: each was set
# by simulation from the series' own fitted in-control negative binomial to hold a
# common false-alarm rate. They are not interchangeable between the two streams.
NBCUSUM_H_PARTICIPATORY = 1.40
NBCUSUM_H_SENTINEL      = 2.52
# Bank order follows Table 2.2 and is the column order of the alarm panel.
DETECTOR_NAMES = ["C1", "C2", "C3", "NBCUSUM", "EWMA", "P95"]


def ears_c1_c2(x, guard=0, base=BASELINE_WEEKS, thr=SD_THRESHOLD, warm=WARMUP_WEEKS):
    """EARS C1 and C2.

    C1 (guard 0):  (x_t - mu7) / sigma7 over the seven weeks preceding t; alarm
                   when the statistic exceeds three.
    C2 (guard 2):  the same statistic, but the seven baseline weeks END TWO WEEKS
                   BEFORE t. The two intervening weeks form a guard band, so a
                   rise already under way does not inflate its own baseline.
    """
    x = np.asarray(x, float)
    alarm = np.zeros(len(x), int)
    for t in range(base + guard, len(x)):
        window = x[t - base - guard:t - guard]
        sd = np.nanstd(window, ddof=1)
        if sd > 0 and (x[t] - np.nanmean(window)) / sd > thr:
            alarm[t] = 1
    alarm[:warm] = 0
    return alarm


def ears_c3(x, base=BASELINE_WEEKS, guard=GUARD_WEEKS, excess_sd=C3_EXCESS_SD,
            thr=C3_THRESHOLD, span=C3_WINDOW, warm=WARMUP_WEEKS):
    """EARS C3: the cumulative sum of POSITIVE C2 EXCESS over the current and the
    prior two weeks, where excess is measured ABOVE ONE standard deviation; alarm
    when the cumulative statistic exceeds two.

    C3 INHERITS C2'S TWO-WEEK GUARD BAND: the C2 statistic it accumulates is
    standardized against the seven weeks ending two weeks before t, exactly as C2
    itself is. This clause is load-bearing. Without the guard band the statistic
    is standardized against weeks that already contain the rise, and the detector
    alarms at the wrong weeks in both streams.
    """
    x = np.asarray(x, float)
    n = len(x)
    c2_stat = np.full(n, np.nan)
    for t in range(base + guard, n):
        window = x[t - base - guard:t - guard]        # the inherited guard band
        sd = np.nanstd(window, ddof=1)
        if sd > 0:
            c2_stat[t] = (x[t] - np.nanmean(window)) / sd
    excess = np.clip(c2_stat - excess_sd, 0, None)     # positive excess above 1 SD
    alarm = np.zeros(n, int)
    for t in range(span - 1, n):
        block = excess[t - span + 1:t + 1]
        if np.all(np.isfinite(block)) and block.sum() > thr:
            alarm[t] = 1
    alarm[:warm] = 0
    return alarm


def nb_cusum(counts, h, base=BASELINE_WEEKS, mult=NBCUSUM_MULT, warm=WARMUP_WEEKS):
    """Negative-binomial likelihood-ratio CUSUM on weekly COUNTS:

        S_t = max(0, S_{t-1} + log[ f_NB(y_t; mu1, alpha) / f_NB(y_t; mu0, alpha) ])

    In each week the in-control mean mu0 is the mean of the seven preceding counts
    and the dispersion alpha is a method-of-moments estimate on the same window;
    the out-of-control mean is mu1 = 2 x mu0. The ACCUMULATOR IS RESET TO ZERO
    AFTER AN ALARM, so one excursion produces one alarm. The decision interval h
    is Monte-Carlo calibrated and passed in per series.
    """
    y = np.asarray(counts, float)
    n = len(y)
    alarm = np.zeros(n, int)
    S = 0.0
    for t in range(warm, n):
        window = y[t - base:t]
        mu0 = np.nanmean(window)
        var = np.nanvar(window, ddof=1)
        if not np.isfinite(mu0) or mu0 <= 0:
            continue
        alpha = max((var - mu0) / mu0 ** 2, 1e-8)      # method-of-moments dispersion
        size = 1.0 / alpha                             # negative-binomial size r
        mu1 = mult * mu0
        S = max(0.0, S + (stats.nbinom.logpmf(y[t], size, size / (size + mu1))
                          - stats.nbinom.logpmf(y[t], size, size / (size + mu0))))
        if S > h:
            alarm[t] = 1
            S = 0.0                                    # reset after an alarm
    return alarm


def ewma_detector(x, lam=EWMA_LAMBDA, L=EWMA_L, base=BASELINE_WEEKS,
                  warm=WARMUP_WEEKS):
    """EWMA chart:

        Z_t = lambda x_t + (1 - lambda) Z_{t-1},
        alarm when Z_t > mu7 + L sigma_Z,  sigma_Z = sd7 sqrt(lambda / (2 - lambda))

    with mu7 and sd7 the mean and standard deviation of the seven weeks preceding t.

    THE CONTROL LIMIT RE-ARMS TO THE BASELINE MEAN AFTER AN ALARM: on alarming,
    Z is set to mu7. This clause is load-bearing. Without it the statistic latches
    above the limit and the sentinel chart alarms in three consecutive weeks
    instead of one, because an EWMA carries its own history forward.
    """
    x = np.asarray(x, float)
    n = len(x)
    alarm = np.zeros(n, int)
    z = np.nanmean(x[:warm])
    for t in range(n):
        z = lam * x[t] + (1 - lam) * z
        if t >= warm:
            window = x[t - base:t]
            sd = np.nanstd(window, ddof=1)
            mu = np.nanmean(window)
            if sd > 0:
                sigma_z = sd * np.sqrt(lam / (2 - lam))
                if z > mu + L * sigma_z:
                    alarm[t] = 1
                    z = mu                             # re-arm to the baseline mean
    return alarm


def percentile_detector(x, q=PCT_Q, hist=PCT_HIST, warm=WARMUP_WEEKS):
    """Nonparametric rule: alarm when the current value exceeds the q-th percentile
    of the PRIOR TEN WEEKS. The window SLIDES; it does not expand over the season
    to date, so the reference distribution stays local to the recent past.
    """
    x = np.asarray(x, float)
    alarm = np.zeros(len(x), int)
    for t in range(hist, len(x)):
        if x[t] > np.nanpercentile(x[t - hist:t], q):
            alarm[t] = 1
    alarm[:warm] = 0
    return alarm


def detector_panel(rate_series, count_series, h_cusum, base=BASELINE_WEEKS,
                   thr=SD_THRESHOLD, warm=WARMUP_WEEKS):
    """The six-detector bank. The five chart detectors read the smoothed rate
    series; the NB-CUSUM reads the raw weekly counts of the same stream. Consensus
    is at least one of six; a stricter at-least-two-of-six variant is carried
    alongside and reported separately.
    """
    d = {"C1":      ears_c1_c2(rate_series, guard=0, base=base, thr=thr, warm=warm),
         "C2":      ears_c1_c2(rate_series, guard=GUARD_WEEKS, base=base, thr=thr,
                               warm=warm),
         "C3":      ears_c3(rate_series, base=base, warm=warm),
         "NBCUSUM": nb_cusum(count_series, h_cusum, base=base, warm=warm),
         "EWMA":    ewma_detector(rate_series, base=base, warm=warm),
         "P95":     percentile_detector(rate_series, warm=warm)}
    m = np.sum([d[k] for k in DETECTOR_NAMES], axis=0)
    d["n_detectors"] = m
    d["any"] = (m >= 1).astype(int)     # consensus rule of record
    d["two"] = (m >= 2).astype(int)     # reported separately
    return d


# --- the two streams -------------------------------------------------------
# Chart detectors read the smoothed rate series; the NB-CUSUM reads raw counts.
PART_COUNTS = wk.n_new_A.values.astype(float)            # incident A-cluster episodes
SENT_COUNTS = wk.sb_positive_cases.values.astype(float)  # positive specimens

D_PART = detector_panel(PART_SM, PART_COUNTS, NBCUSUM_H_PARTICIPATORY)
D_SENT = detector_panel(SENT_SM, SENT_COUNTS, NBCUSUM_H_SENTINEL)

for lab, d in [("participatory", D_PART), ("sentinel", D_SENT)]:
    t = pd.DataFrame({k: d[k] for k in DETECTOR_NAMES + ["n_detectors", "any"]},
                     index=WEEKS)
    al = t.index[t["any"] == 1].tolist()
    print(f"--- {lab}: {len(al)} consensus alarm weeks (rule: at least one of six)")
    print(t[t["any"] == 1].to_string())
    print(f"    alarm weeks: {', '.join(str(x) for x in al)}")
    print(f"    under the stricter rule of at least two of six: "
          f"{', '.join(str(x) for x in WEEKS[d['two'] == 1])}\n")

print("per-detector alarm weeks")
for lab, d in [("participatory", D_PART), ("sentinel", D_SENT)]:
    for k in DETECTOR_NAMES:
        print(f"  {lab:14s} {k:8s} {', '.join(str(x) for x in WEEKS[d[k] == 1]) or 'none'}")

print(f"\nweeks that cannot alarm under any detector (warm-up): "
      f"{WEEKS[0]} to {WEEKS[WARMUP_WEEKS-1]} — {WARMUP_WEEKS} weeks")
print(f"decision intervals in force: participatory {NBCUSUM_H_PARTICIPATORY}, "
      f"sentinel {NBCUSUM_H_SENTINEL} (Monte-Carlo calibrated, series-specific)")
print(f"C3 guard band: {GUARD_WEEKS} weeks, inherited from C2")
print(f"EWMA: lambda {EWMA_LAMBDA}, control limit re-arms to the baseline mean "
      f"after an alarm")
print(f"percentile detector: {PCT_Q}th percentile of the prior {PCT_HIST} weeks, "
      f"sliding window")

# the participatory denominator that motivates the causal three-week average
den = wk.n_new_episodes
print(f"\nparticipatory weekly denominator: median {den.median():.0f} incident episodes "
      f"(interquartile range {den.quantile(.25):.0f} to {den.quantile(.75):.0f}); "
      f"one episode moves the share by about {100/den.median():.1f} percentage points")

record("A55", part_alarms=[str(x) for x in WEEKS[D_PART["any"] == 1]],
       sent_alarms=[str(x) for x in WEEKS[D_SENT["any"] == 1]],
       part_alarms_two=[str(x) for x in WEEKS[D_PART["two"] == 1]],
       sent_alarms_two=[str(x) for x in WEEKS[D_SENT["two"] == 1]],
       warmup=WARMUP_WEEKS, h_part=NBCUSUM_H_PARTICIPATORY, h_sent=NBCUSUM_H_SENTINEL,
       c3_guard=GUARD_WEEKS, ewma_rearm=True)

--- participatory: 8 consensus alarm weeks (rule: at least one of six)
          C1  C2  C3  NBCUSUM  EWMA  P95  n_detectors  any
2025-W51   0   0   0        1     0    0            1    1
2025-W52   0   0   0        1     0    1            2    1
2026-W01   0   1   1        0     0    1            3    1
2026-W02   0   1   1        0     0    0            2    1
2026-W03   0   0   1        0     0    0            1    1
2026-W04   0   0   1        0     0    1            2    1
2026-W05   0   0   1        0     0    1            2    1
2026-W20   1   1   1        0     1    1            5    1
    alarm weeks: 2025-W51, 2025-W52, 2026-W01, 2026-W02, 2026-W03, 2026-W04, 2026-W05, 2026-W20
    under the stricter rule of at least two of six: 2025-W52, 2026-W01, 2026-W02, 2026-W04, 2026-W05, 2026-W20

--- sentinel: 8 consensus alarm weeks (rule: at least one of six)
          C1  C2  C3  NBCUSUM  EWMA  P95  n_detectors  any
2025-W50   0   0   0        1     0    0            1    1
2025-W

### Cell 45 — A56: Alarm concordance by epidemic phase

**Pseudocode.**

1. Treat the sentinel consensus alarm as the reference and the participatory
   consensus alarm as the test, week by week.
2. Within each window compute agreement, Cohen's kappa, sensitivity, positive
   predictive value and the phi coefficient.
3. Report the same statistics over the full season, the early-warning window and the
   epidemic phase, and separately over the full season with the flagged closing week
   omitted, since the sparse-week rule flags that week.
4. The early-warning window of record is the nineteen weeks of the pre-epidemic and
   epidemic phases together, 2025-W40 to 2026-W06, which is the window fixed with the
   epidemic calendar. The thirteen weeks of the 2025 calendar portion, 2025-W40 to
   2025-W52, are a narrower window scored alongside it and labelled as such; the two
   windows carry different numbers of sentinel alarms and are not interchangeable.


In [45]:
a, b = D_PART["any"], D_SENT["any"]

def concord(x, y):
    tp = int(((x == 1) & (y == 1)).sum()); fp = int(((x == 1) & (y == 0)).sum())
    fn = int(((x == 0) & (y == 1)).sum()); tn = int(((x == 0) & (y == 0)).sum())
    phi = stats.pearsonr(x, y)[0] if x.std() > 0 and y.std() > 0 else np.nan
    kap = cohen_kappa_score(x, y) if (x.std() > 0 or y.std() > 0) else np.nan
    return dict(n=len(x), agreement=round(float((x == y).mean()), 5),
                kappa=round(float(kap), 5) if np.isfinite(kap) else None,
                sensitivity=round(tp / (tp + fn), 4) if tp + fn else None,
                ppv=round(tp / (tp + fp), 4) if tp + fp else None,
                phi=round(float(phi), 5) if np.isfinite(phi) else None,
                tp=tp, fp=fp, fn=fn, tn=tn)

excl_closing = SEASON.copy(); excl_closing[-1] = False
WINDOWS = [("full season", SEASON),
           ("early-warning window (pre-epidemic and epidemic, 19 weeks)", EARLY),
           ("epidemic phase", EPI),
           ("full season excluding the flagged closing week", excl_closing),
           ("2025 calendar portion (13 weeks, narrower early-warning window)", EARLY_2025)]
rows = {lab: concord(a[m], b[m]) for lab, m in WINDOWS}
print(pd.DataFrame(rows).T.to_string())

print("\nunder the stricter consensus rule of at least two detectors of six")
rows2 = {lab: concord(D_PART["two"][m], D_SENT["two"][m]) for lab, m in WINDOWS}
print(pd.DataFrame(rows2).T.to_string())

k_full = rows["full season"]["kappa"]
n_weeks_needed = int(np.ceil(((1.96 + 0.8416) / np.arctanh(max(k_full, 1e-6))) ** 2 + 3))
print(f"\nweeks required to establish an agreement of this size at 80% power: {n_weeks_needed}")

record("A56", kappa_full=rows["full season"]["kappa"],
       kappa_ew=rows["early-warning window (pre-epidemic and epidemic, 19 weeks)"]["kappa"],
       kappa_epi=rows["epidemic phase"]["kappa"],
       kappa_excl=rows["full season excluding the flagged closing week"]["kappa"],
       sens_ew=rows["early-warning window (pre-epidemic and epidemic, 19 weeks)"]["sensitivity"],
       ppv_ew=rows["early-warning window (pre-epidemic and epidemic, 19 weeks)"]["ppv"],
       kappa_ew13=rows["2025 calendar portion (13 weeks, narrower early-warning window)"]["kappa"],
       sens_ew13=rows["2025 calendar portion (13 weeks, narrower early-warning window)"]["sensitivity"],
       ppv_ew13=rows["2025 calendar portion (13 weeks, narrower early-warning window)"]["ppv"],
       kappa_ew13_two=rows2["2025 calendar portion (13 weeks, narrower early-warning window)"]["kappa"])
CONCORD_ROWS, CONCORD_ROWS_TWO = rows, rows2

                                                                    n  agreement    kappa  sensitivity    ppv      phi   tp   fp   fn    tn
full season                                                      33.0    0.93939  0.83500       0.8750  0.875  0.83500  7.0  1.0  1.0  24.0
early-warning window (pre-epidemic and epidemic, 19 weeks)       19.0    0.94737  0.89017       0.8750  1.000  0.89559  7.0  0.0  1.0  11.0
epidemic phase                                                    9.0    0.88889  0.60870       0.8750  1.000  0.66144  7.0  0.0  1.0   1.0
full season excluding the flagged closing week                   32.0    0.96875  0.91304       0.8750  1.000  0.91652  7.0  0.0  1.0  24.0
2025 calendar portion (13 weeks, narrower early-warning window)  13.0    0.92308  0.75472       0.6667  1.000  0.77850  2.0  0.0  1.0  10.0

under the stricter consensus rule of at least two detectors of six
                                                                    n  agreement    kappa  s

## S3.4 Sampling, selection and panel-validity analysis

### Cell 46 — A60: Sparse-week flagging rule

**Pseudocode.**

1. The participatory denominator in a week is all cleaned incident episodes filed
   that week.
2. Flag a week when that denominator falls below ten, roughly two-thirds of the
   thirty-three-week median.
3. Flagging is DISCLOSURE ONLY. No flagged week is excluded from any analysis and no
   reported coefficient changes because a week is flagged.
4. Report which weeks are flagged and which epidemic phase each falls in, so it is
   visible that the flagged weeks cluster at the season's opening and close.


In [46]:
SPARSE_THRESHOLD = 10
d = wk.n_new_episodes
flag = d < SPARSE_THRESHOLD
phase = np.where(PRE, "pre-epidemic", np.where(EPI, "epidemic", "post-epidemic"))
print(f"threshold: participatory denominator below {SPARSE_THRESHOLD} "
      f"(median over {N_WEEKS} weeks = {d.median():.0f})")
print(f"flagged weeks: {int(flag.sum())} of {N_WEEKS}")
print(pd.DataFrame({"week": WEEKS[flag], "denominator": d[flag].values,
                    "phase": phase[flag]}).to_string(index=False))
print("\nflagged weeks per phase")
for lab, m in [("pre-epidemic", PRE), ("epidemic", EPI), ("post-epidemic", POST)]:
    print(f"  {lab:14s} {int((flag.values & m).sum())} of {int(m.sum())}")
print("\nflagging is disclosure, never exclusion: no analysis in this supplement drops "
      "a flagged week, so no reported value changes.")
record("A60", n_flagged=int(flag.sum()), flagged=[str(x) for x in WEEKS[flag]])

threshold: participatory denominator below 10 (median over 33 weeks = 15)
flagged weeks: 5 of 33
    week  denominator         phase
2025-W40            8  pre-epidemic
2025-W41            8  pre-epidemic
2025-W45            7  pre-epidemic
2026-W19            8 post-epidemic
2026-W20            8 post-epidemic

flagged weeks per phase
  pre-epidemic   3 of 10
  epidemic       0 of 9
  post-epidemic  2 of 14

flagging is disclosure, never exclusion: no analysis in this supplement drops a flagged week, so no reported value changes.


### Cell 47 — A61: Baseline comparison of reporters and never-reporters

**Pseudocode.**

1. Compare the 248 physicians who filed at least one weekly report against the 56
   who registered and filed none, over the sixteen baseline characteristics reported
   in the thesis table.
2. For each characteristic compute the standardized mean difference using the pooled
   standard deviation, with the binary form for proportions and the continuous form
   for means.
3. The health-condition row rests on 242 reporters, because the six physicians who
   declined that question stay missing rather than being imputed to zero.
4. Test proportions with Fisher's exact test and continuous variables with the
   Mann-Whitney U test.
5. Count how many characteristics exceed absolute differences of 0.10 and 0.25, and
   report them ordered by magnitude.


In [47]:
CHARACTERISTICS = [
    ("age", "age, years (mean)", False), ("female", "female", True),
    ("sees_pat", "performs face-to-face examination", True),
    ("university", "university hospital", True), ("academic", "academic title", True),
    ("gp", "general practitioner", True), ("resident", "resident", True),
    ("comp_risk_any_nan", "any health condition conferring complication risk", True),
    ("school_kids_any", "household school-age child", True),
    ("hh", "household size (mean)", False), ("smoker_current", "current tobacco use", True),
    ("allergy", "respiratory allergy", True), ("active", "sufficient physical activity", True),
    ("ili_freq_ord4", "susceptibility (four-level ordinal mean)", False),
    ("prev_vax", "prior-season vaccination", True),
    ("vax", "current-season vaccination", True)]
R = phys[phys.reported == 1]
NR = phys[phys.reported == 0]
rows = []
for v, lab, binary in CHARACTERISTICS:
    a, b = R[v].dropna(), NR[v].dropna()
    s = smd(a, b, binary)
    if binary:
        ct = [[int(a.sum()), len(a) - int(a.sum())], [int(b.sum()), len(b) - int(b.sum())]]
        p = stats.fisher_exact(ct)[1]
        va, vb = f"{100*a.mean():.1f}%", f"{100*b.mean():.1f}%"
    else:
        p = stats.mannwhitneyu(a, b).pvalue
        va, vb = f"{a.mean():.2f}", f"{b.mean():.2f}"
    rows.append(dict(characteristic=lab, n_reporters=len(a), n_never=len(b),
                     reporters=va, never_reporters=vb, SMD=round(s, 5), p=round(p, 4)))
B = pd.DataFrame(rows).sort_values("SMD", key=lambda s: s.abs(), ascending=False)
print(B.to_string(index=False))
n10 = int((B.SMD.abs() > 0.10).sum()); n25 = int((B.SMD.abs() > 0.25).sum())
print(f"\ncharacteristics with |SMD| > 0.10   {n10} of {len(B)}")
print(f"characteristics with |SMD| > 0.25   {n25} of {len(B)}")
print(f"largest imbalance: {B.iloc[0].characteristic} at {B.iloc[0].SMD:+.5f}")
record("A61", n_gt10=n10, n_gt25=n25,
       **{f"smd_{v}": float(B.set_index('characteristic').loc[lab, 'SMD'])
          for v, lab, _ in CHARACTERISTICS[:1]})
SMD_TABLE = B

                                   characteristic  n_reporters  n_never reporters never_reporters      SMD      p
                                   academic title          248       56     35.9%            7.1%  0.74666 0.0000
                performs face-to-face examination          248       56     45.2%           67.9% -0.47030 0.0029
                              university hospital          248       56     42.3%           21.4%  0.46043 0.0037
                       current-season vaccination          248       56     36.7%           17.9%  0.43272 0.0073
                       household school-age child          248       56     40.3%           21.4%  0.41781 0.0089
                            household size (mean)          248       56      2.55            2.09  0.40581 0.0040
                             general practitioner          248       56     10.1%           23.2% -0.35818 0.0125
                     sufficient physical activity          248       56     47.2%       

### Cell 48 — A62: Reporting-propensity model

**Pseudocode.**

1. The outcome is whether a registrant filed at least one weekly report. The sample
   is every registrant with a health-condition response: 298 of the 304, of whom 242
   are reporters.
2. The adjustment set is the pre-specified baseline set of eight terms: age in
   ten-year units, sex, face-to-face patient care, university affiliation, any health
   condition conferring complication risk, susceptibility in four ordered levels,
   household school-age children and prior-season vaccination. It is fixed on
   epidemiological grounds; no variable is selected on the data.
3. Fit the logistic model, report coefficients as odds ratios, and take the apparent
   area under the curve on the fitting sample.
4. Correct for optimism with the Efron bootstrap: one thousand replicates, seed
   20260707, REFITTING the model in every replicate and scoring each refit on both
   the replicate and the original sample.
5. Report the fitted propensity distribution, which the weighting block then uses.


In [48]:
d4 = phys.dropna(subset=["comp_risk_any_nan"]).copy()
F4 = ("reported ~ age10 + female + comp_risk_any_nan + ili_freq_ord4 + sees_pat "
      "+ prev_vax + school_kids_any + university")
fit4 = smf.logit(F4, data=d4).fit(disp=0)
print(f"sample {len(d4)} registrants, of whom {int(d4.reported.sum())} are reporters; "
      f"{len(phys)-len(d4)} declined the health-condition item and stay missing")
print(pd.DataFrame({"OR": np.exp(fit4.params),
                    "lo": np.exp(fit4.conf_int()[0]), "hi": np.exp(fit4.conf_int()[1]),
                    "p": fit4.pvalues}).round(5).to_string())
app, opt, corr, nrep = efron_optimism(F4, d4, "reported", B=1000, seed=20260707)
print(f"\napparent AUC             {app:.5f}   (eight-term model of record)")
print(f"Efron optimism           {opt:.5f}   ({nrep} usable replicates of 1000, seed 20260707)")
print(f"optimism-corrected AUC   {corr:.5f}")
F7 = F4.replace(" + university", "")
app7 = roc_auc_score(d4.reported, smf.logit(F7, data=d4).fit(disp=0).predict(d4))
print(f"for contrast, dropping university affiliation gives an apparent AUC of {app7:.5f}; "
      "the eight-term fit above is the model of record")
PI = fit4.predict(d4)
print(f"\nfitted reporting propensity  min {PI.min():.5f}, median {PI.median():.5f}, "
      f"max {PI.max():.5f}")
record("A62", n=len(d4), n_reporters=int(d4.reported.sum()), apparent=app, optimism=opt,
       corrected=corr, apparent_7term=app7, pi_min=float(PI.min()),
       pi_median=float(PI.median()), pi_max=float(PI.max()))
D4, FIT4, PROPENSITY = d4, fit4, PI

sample 298 registrants, of whom 242 are reporters; 6 declined the health-condition item and stay missing
                        OR       lo       hi        p
Intercept          1.99318  0.43577  9.11676  0.37392
age10              1.15247  0.84583  1.57028  0.36860
female             1.59977  0.84282  3.03655  0.15072
comp_risk_any_nan  0.57380  0.26928  1.22269  0.15013
ili_freq_ord4      0.87891  0.54754  1.41083  0.59296
sees_pat           0.47282  0.24523  0.91162  0.02534
prev_vax           1.70807  0.86197  3.38466  0.12496
school_kids_any    2.15768  1.02902  4.52429  0.04178
university         2.19596  1.05662  4.56382  0.03507

apparent AUC             0.71499   (eight-term model of record)
Efron optimism           0.04998   (1000 usable replicates of 1000, seed 20260707)
optimism-corrected AUC   0.66500
for contrast, dropping university affiliation gives an apparent AUC of 0.69772; the eight-term fit above is the model of record

fitted reporting propensity  min 0.41423, med

### Cell 49 — A63: Stabilized reporting weights, Kish effective sample and design effect

**Pseudocode.**

1. The stabilized weight for a reporter is the marginal reporting probability divided
   by that physician's fitted propensity. Stabilization is what keeps the weights
   centred near one.
2. Weights are formed for reporters only, since they are the physicians who
   contribute person-time.
3. Report the weight range, mean and standard deviation.
4. The Kish effective sample size is the squared sum of the weights over the sum of
   their squares; the design effect is the nominal sample divided by it.


In [49]:
p_marginal = D4.reported.mean()
w = (p_marginal / PROPENSITY)[D4.reported == 1]
kish = w.sum() ** 2 / (w ** 2).sum()
deff = len(w) / kish
print(f"marginal reporting probability P(R = 1)  {p_marginal:.6f}")
print(f"stabilized weights, n = {len(w)}")
print(f"  range  {w.min():.5f} to {w.max():.5f}")
print(f"  mean   {w.mean():.5f}")
print(f"  SD     {w.std(ddof=1):.5f}")
print(f"Kish effective sample size  {kish:.4f} of {len(w)}")
print(f"design effect               {deff:.5f}")
record("A63", w_min=float(w.min()), w_max=float(w.max()), w_mean=float(w.mean()),
       w_sd=float(w.std(ddof=1)), kish=float(kish), deff=float(deff))
STAB_WEIGHTS = pd.Series(w.values, index=D4.loc[D4.reported == 1, "participant_id"].values)

marginal reporting probability P(R = 1)  0.812081
stabilized weights, n = 242
  range  0.82920 to 1.96047
  mean   1.00083
  SD     0.16801
Kish effective sample size  235.3936 of 242
design effect               1.02807


### Cell 50 — A65: Association of reporting propensity with the episode rate

**Pseudocode.**

1. If reporting propensity were associated with the episode rate, weighting would
   move the incidence estimate. Test that association directly.
2. Regress each reporter's episode count on the fitted propensity with a log at-risk
   week offset, Poisson family and robust variance, and report the rate ratio per unit
   of propensity.
3. Split reporters into propensity tertiles and report the crude rate per hundred
   at-risk person-weeks in each.
4. Correlate the stabilized weight with the episode count.


In [50]:
per_phys = pw.groupby("participant_id").agg(episodes=("new_episode", "sum"),
                                            at_risk=("at_risk_new_episode", "sum"))
d = pd.DataFrame({"participant_id": D4.loc[D4.reported == 1, "participant_id"].values,
                  "propensity": PROPENSITY[D4.reported == 1].values,
                  "weight": STAB_WEIGHTS.values}).set_index("participant_id").join(per_phys)
m = smf.glm("episodes ~ propensity", data=d, family=sm.families.Poisson(),
            offset=np.log(d.at_risk)).fit(cov_type="HC0")
rr = np.exp(m.params["propensity"]); ci = np.exp(m.conf_int().loc["propensity"])
print(f"rate ratio per unit of fitted propensity  {rr:.4f} ({ci[0]:.3f}-{ci[1]:.3f}), "
      f"p = {m.pvalues['propensity']:.4f}")
d["tertile"] = pd.qcut(d.propensity, 3, labels=["T1 lowest", "T2", "T3 highest"])
G = d.groupby("tertile", observed=True).agg(physicians=("episodes", "size"),
                                           episodes=("episodes", "sum"),
                                           at_risk=("at_risk", "sum"))
G["rate_per_100"] = (100 * G.episodes / G.at_risk).round(4)
print("\ncrude rates across propensity tertiles")
print(G.to_string())
print(f"\ncorrelation of the stabilized weight with the episode count  "
      f"Pearson {d.weight.corr(d.episodes):+.4f}, Spearman "
      f"{d.weight.corr(d.episodes, method='spearman'):+.4f}")
record("A65", rr=float(rr), rr_lo=float(ci[0]), rr_hi=float(ci[1]),
       p=float(m.pvalues["propensity"]), tertile_rates=[float(x) for x in G.rate_per_100],
       corr_weight_count=float(d.weight.corr(d.episodes)))

rate ratio per unit of fitted propensity  0.7327 (0.240-2.239), p = 0.5854

crude rates across propensity tertiles
            physicians  episodes  at_risk  rate_per_100
tertile                                                
T1 lowest           81       146     1391       10.4960
T2                  80       162     1477       10.9682
T3 highest          81       181     1697       10.6659

correlation of the stabilized weight with the episode count  Pearson -0.0169, Spearman -0.0680


### Cell 51 — A66: Engagement-stratum exposure profile comparison

**Pseudocode.**

1. Compare the most engaged stratum (twenty-four or more weeks reported) with the
   least engaged (up to five weeks) on exposure-relevant characteristics.
2. Institution type is a five-level categorical variable and is tested with a
   chi-square test on the contingency table.
3. Face-to-face patient care and on-call duty are binary and are tested with a
   chi-square test with continuity correction.
4. Public-transport commuting time is ordinal and is tested with the Mann-Whitney U
   test.


In [51]:
rep = phys[phys.reported == 1]
hi_ = rep[rep.n_weeks_reported >= 24]
lo_ = rep[rep.n_weeks_reported <= 5]
grp = ["most engaged"] * len(hi_) + ["least engaged"] * len(lo_)
print(f"most engaged {len(hi_)} physicians; least engaged {len(lo_)} physicians")
ct = pd.crosstab(pd.concat([hi_.institution_type, lo_.institution_type]), grp)
chi2, p, dof, _ = stats.chi2_contingency(ct, correction=False)
print(f"\ninstitution type      chi-square {chi2:.4f}, df {dof}, p = {p:.6f}")
print(ct.to_string())
for v, lab in [("sees_pat", "performs face-to-face examination"),
               ("oncall_any", "any monthly on-call duty")]:
    t = pd.crosstab(pd.concat([hi_[v], lo_[v]]), grp)
    chi2, p, dof, _ = stats.chi2_contingency(t, correction=True)
    print(f"{lab:36s} {100*hi_[v].mean():.1f}% against {100*lo_[v].mean():.1f}%; "
          f"chi-square {chi2:.4f}, df {dof}, p = {p:.6f}")
u = stats.mannwhitneyu(hi_.pub_transport_ord, lo_.pub_transport_ord)
print(f"{'public-transport commuting time':36s} Mann-Whitney U = {u.statistic:.1f}, "
      f"p = {u.pvalue:.6f}")
record("A66", inst_chi2=float(stats.chi2_contingency(ct, correction=False)[0]),
       sees_pat_hi=100*hi_.sees_pat.mean(), sees_pat_lo=100*lo_.sees_pat.mean(),
       transport_p=float(u.pvalue))

most engaged 105 physicians; least engaged 37 physicians

institution type      chi-square 22.2825, df 4, p = 0.000176
col_0                                                                                               least engaged  most engaged
institution_type                                                                                                               
Sağlık Bakanlığı - Hastane                                                                                     16            15
Sağlık Bakanlığı: Aile Sağlığı Merkezi                                                                          5             4
Sağlık Bakanlığı: Merkez - İl Sağlık Müdürlüğü - İlçe Sağlık Müdürlüğü - 112 Komuta Kontrol Birimi              7            27
Özel Hastane / Klinik                                                                                           2             4
Üniversite (Kamu ya da Vakıf) - Hastane                                                                         7

### Cell 52 — A67: Reporting-intensity gradient in incidence

**Pseudocode.**

1. Aggregate the panel to one row per reporting physician: episode count and at-risk
   person-weeks.
2. Model the episode count with Poisson regression, a log at-risk-week offset and
   robust sandwich variance, so that overdispersion does not distort the interval.
3. The exposure of interest is the number of weeks reported, entered in units of ten
   weeks. The pre-specified adjustment set is age, household school-age children and
   susceptibility in four levels.
4. Report the adjusted incidence-rate ratio per ten additional reported weeks, and
   the unadjusted estimate beside it.


In [52]:
D = pw.groupby("participant_id").agg(episodes=("new_episode", "sum"),
                                     at_risk=("at_risk_new_episode", "sum"))
D = D.join(phys.set_index("participant_id")[["n_weeks_reported", "age10",
                                             "school_kids_any", "ili_freq_ord4"]])
D["weeks10"] = D.n_weeks_reported / 10
print(f"{len(D)} physicians, {int(D.at_risk.sum()):,} at-risk person-weeks, "
      f"{int(D.episodes.sum())} episodes")
FULL = "episodes ~ weeks10 + age10 + school_kids_any + ili_freq_ord4"
fit = smf.glm(FULL, data=D, family=sm.families.Poisson(),
              offset=np.log(D.at_risk)).fit(cov_type="HC0")
print("\nadjusted offset Poisson with robust variance")
print(pd.DataFrame({"IRR": np.exp(fit.params), "lo": np.exp(fit.conf_int()[0]),
                    "hi": np.exp(fit.conf_int()[1]), "p": fit.pvalues}).round(6).to_string())
un = smf.glm("episodes ~ weeks10", data=D, family=sm.families.Poisson(),
             offset=np.log(D.at_risk)).fit(cov_type="HC0")
ci = np.exp(un.conf_int().loc["weeks10"])
print(f"\nunadjusted IRR per ten more reported weeks  {np.exp(un.params['weeks10']):.5f} "
      f"({ci[0]:.5f}-{ci[1]:.5f}), p = {un.pvalues['weeks10']:.6f}")
ciA = np.exp(fit.conf_int().loc["weeks10"])
record("A67", irr=float(np.exp(fit.params["weeks10"])), lo=float(ciA[0]), hi=float(ciA[1]),
       p=float(fit.pvalues["weeks10"]), irr_unadj=float(np.exp(un.params["weeks10"])),
       n=len(D), at_risk=int(D.at_risk.sum()))
PHYS_LEVEL, GRADIENT_FORMULA = D, FULL

248 physicians, 4,626 at-risk person-weeks, 497 episodes

adjusted offset Poisson with robust variance
                      IRR        lo        hi         p
Intercept        0.221369  0.123558  0.396611  0.000000
weeks10          0.804553  0.709541  0.912288  0.000695
age10            0.855737  0.761155  0.962072  0.009134
school_kids_any  1.282328  1.017404  1.616235  0.035196
ili_freq_ord4    1.244626  1.054306  1.469300  0.009751

unadjusted IRR per ten more reported weeks  0.79121 (0.69567-0.89986), p = 0.000361


### Cell 53 — A68: Stratum-specific incidence rates

**Pseudocode.**

1. Within each engagement stratum sum episodes and at-risk person-weeks.
2. Report the crude rate per hundred at-risk person-weeks with an exact Poisson
   interval derived from the chi-square distribution.
3. The gradient runs against intuition: the most engaged stratum has the LOWEST rate,
   which is what the gradient model quantifies.


In [53]:
rep = phys[phys.reported == 1].copy()
rep["stratum"] = pd.cut(rep.n_weeks_reported, bins=[0, 5, 23, 33],
                        labels=["least engaged (1-5 weeks)", "moderately engaged (6-23 weeks)",
                                "most engaged (24+ weeks)"])
per_phys = pw.groupby("participant_id").agg(episodes=("new_episode", "sum"),
                                            at_risk=("at_risk_new_episode", "sum"))
S = rep.set_index("participant_id").join(per_phys)
rows = []
for lab, g in S.groupby("stratum", observed=True):
    k, n = int(g.episodes.sum()), int(g.at_risk.sum())
    lo = 100 * stats.chi2.ppf(.025, 2*k) / 2 / n
    hi = 100 * stats.chi2.ppf(.975, 2*k+2) / 2 / n
    rows.append(dict(stratum=lab, physicians=len(g), episodes=k, at_risk=n,
                     rate_per_100=round(100*k/n, 5), ci_lo=round(lo, 2), ci_hi=round(hi, 2)))
print(pd.DataFrame(rows).to_string(index=False))
record("A68", rates=[r["rate_per_100"] for r in rows],
       strata=[str(r["stratum"]) for r in rows])

                        stratum  physicians  episodes  at_risk  rate_per_100  ci_lo  ci_hi
      least engaged (1-5 weeks)          37        12       86      13.95349   7.21  24.37
moderately engaged (6-23 weeks)         106       216     1543      13.99870  12.19  16.00
       most engaged (24+ weeks)         105       269     2997       8.97564   7.93  10.11


### Cell 54 — A73: Episodes attributable to the coding rule

**Pseudocode.**

1. The episode-coding rule treats a symptomatic week as a NEW episode when the
   preceding week in the physician's own record was not filed, because continuity
   cannot be established across a gap.
2. Enumerate incident onsets that fall in a week preceded by a gap, and among those,
   the ones where the last week actually filed before the gap was itself symptomatic.
   Those are the onsets the rule creates: continuity would have been plausible.
3. Report that count against all incident episodes, and separately the subset where
   the gap was a single week, where the attribution is strongest.


In [54]:
gap_onsets = pw[(pw.new_episode == 1) & (pw.gap_before_derived == 1)]
rows = []
for pid, k in zip(gap_onsets.participant_id, gap_onsets.week_idx):
    prior = pw[(pw.participant_id == pid) & (pw.week_idx < k)]
    if len(prior) == 0:
        continue
    last = prior.loc[prior.week_idx.idxmax()]
    rows.append(dict(gap_length=int(k - last.week_idx - 1), prior_symptomatic=int(last.symptomatic)))
G = pd.DataFrame(rows)
n_ep = int(pw.new_episode.sum())
attrib = int(G.prior_symptomatic.sum())
one_wk = int(((G.gap_length == 1) & (G.prior_symptomatic == 1)).sum())
print(f"incident onsets in a week preceded by a gap   {len(G)} of {n_ep} "
      f"({100*len(G)/n_ep:.2f}%)")
print(f"of those, the last week filed before the gap was symptomatic  {attrib} "
      f"({100*attrib/n_ep:.2f}% of all episodes) — these are the onsets the rule creates")
print(f"restricted to a single-week gap                {one_wk} ({100*one_wk/n_ep:.2f}%)")
print("\ngap length by whether the preceding filed week was symptomatic")
print(G.groupby(["prior_symptomatic", "gap_length"]).size().to_string())
record("A73", n_gap_onsets=len(G), attributable=attrib, one_week=one_wk,
       pct=100*attrib/n_ep)

incident onsets in a week preceded by a gap   83 of 497 (16.70%)
of those, the last week filed before the gap was symptomatic  22 (4.43% of all episodes) — these are the onsets the rule creates
restricted to a single-week gap                14 (2.82%)

gap length by whether the preceding filed week was symptomatic
prior_symptomatic  gap_length
0                  1             40
                   2              8
                   3              9
                   4              2
                   7              1
                   10             1
1                  1             14
                   2              5
                   3              2
                   12             1


### Cell 55 — A75: Differential ascertainment by agent cluster

**Pseudocode.**

1. If more engaged physicians ascertained episodes differently, the CLUSTER MIX of
   their episodes would differ, not just the episode count.
2. Cross-tabulate incident episodes by engagement stratum and agent cluster and test
   the composition with a chi-square test.
3. Report the A-cluster share within each stratum, which is the quantity the
   system-level analysis depends on.
4. Refit the reporting-intensity gradient on the B-cluster episodes alone, as a
   negative control: B-cluster episodes have no influenza interpretation, so a
   gradient there indicates a reporting artefact rather than an incidence difference.


In [55]:
rep = phys[phys.reported == 1].copy()
rep["stratum"] = pd.cut(rep.n_weeks_reported, bins=[0, 5, 23, 33],
                        labels=["least engaged", "moderately engaged", "most engaged"])
smap = rep.set_index("participant_id").stratum
ep = episodes.assign(stratum=episodes.participant_id.map(smap))
ct = pd.crosstab(ep.stratum, ep.cluster_letter)
chi2, p, dof, _ = stats.chi2_contingency(ct)
print("incident episodes by engagement stratum and agent cluster")
print(ct.to_string())
print(f"composition chi-square {chi2:.5f}, df {dof}, p = {p:.5f}")
share = (100 * ct["A"] / ct.sum(axis=1)).round(2)
print(f"\nA-cluster share within each stratum  " +
      ", ".join(f"{k} {v}%" for k, v in share.items()))
# negative-control re-fit on B-cluster episodes only
epB = pw.assign(newB=((pw.new_episode == 1) & (pw.symptom_week_cluster == "B")).astype(int))
DB = epB.groupby("participant_id").agg(episodes=("newB", "sum"),
                                       at_risk=("at_risk_new_episode", "sum"))
DB = DB.join(phys.set_index("participant_id")[["n_weeks_reported", "age10",
                                               "school_kids_any", "ili_freq_ord4"]])
DB["weeks10"] = DB.n_weeks_reported / 10
fb = smf.glm("episodes ~ weeks10 + age10 + school_kids_any + ili_freq_ord4", data=DB,
             family=sm.families.Poisson(), offset=np.log(DB.at_risk)).fit(cov_type="HC0")
cib = np.exp(fb.conf_int().loc["weeks10"])
print(f"\nnegative-control gradient on B-cluster episodes only: IRR "
      f"{np.exp(fb.params['weeks10']):.5f} ({cib[0]:.3f}-{cib[1]:.3f}), "
      f"p = {fb.pvalues['weeks10']:.5f}, {int(DB.episodes.sum())} episodes")
print("a gradient of similar size in the negative control points to reporting intensity "
      "rather than to a difference in influenza-like illness")
record("A75", chi2=float(chi2), p=float(p), A_shares=[float(v) for v in share],
       irr_B=float(np.exp(fb.params["weeks10"])))

incident episodes by engagement stratum and agent cluster
cluster_letter       A    B   C   D
stratum                            
least engaged        6    5   1   0
moderately engaged  39  148  21   8
most engaged        46  188  21  14
composition chi-square 9.80176, df 6, p = 0.13325

A-cluster share within each stratum  least engaged 50.0%, moderately engaged 18.06%, most engaged 17.1%

negative-control gradient on B-cluster episodes only: IRR 0.83338 (0.717-0.968), p = 0.01733, 341 episodes
a gradient of similar size in the negative control points to reporting intensity rather than to a difference in influenza-like illness


### Cell 56 — A76: Panel effective sample size and design effect

**Pseudocode.**

1. Person-weeks within a physician are correlated, so the nominal 4,729 rows carry
   less information than 4,729 independent observations.
2. Estimate the intracluster correlation of the event indicator with the one-way
   analysis-of-variance estimator on the observed panel.
3. The design effect is one plus the mean cluster size minus one, times the
   intracluster correlation; the effective sample is the nominal panel divided by it.
4. Report the ACHIEVED value from these data, and separately the planning value that
   follows from the intracluster correlation assumed at the design stage. The two are
   different quantities and the planning one is labelled as such.


In [56]:
y = pw.new_episode.values
g = pw.participant_id.values
grand = y.mean()
sizes = pd.Series(y).groupby(g).size()
means = pd.Series(y).groupby(g).mean()
k = len(sizes)
msb = float((sizes * (means - grand) ** 2).sum() / (k - 1))
msw = float(((y - pd.Series(y).groupby(g).transform("mean")) ** 2).sum() / (len(y) - k))
mbar = len(y) / k
icc = (msb - msw) / (msb + (mbar - 1) * msw)
deff = 1 + (mbar - 1) * icc
print(f"clusters (physicians)          {k}")
print(f"mean cluster size              {mbar:.4f} weeks per physician")
print(f"ACHIEVED intracluster correlation of the event indicator  {icc:.5f}")
print(f"ACHIEVED design effect         {deff:.5f}")
print(f"ACHIEVED effective sample      {len(y)/deff:.1f} of {len(y):,} person-weeks")
ICC_PLANNING = 0.127          # value assumed at the design stage, not estimated here
deff_p = 1 + (mbar - 1) * ICC_PLANNING
print(f"\nPLANNING intracluster correlation {ICC_PLANNING} (design-stage assumption)")
print(f"PLANNING design effect         {deff_p:.4f}")
print(f"PLANNING effective sample      {len(y)/deff_p:.1f} person-weeks")
record("A76", icc=float(icc), deff=float(deff), n_eff=float(len(y)/deff),
       mbar=float(mbar), deff_planning=float(deff_p), n_eff_planning=float(len(y)/deff_p))
ICC_ACHIEVED, DEFF_ACHIEVED, DEFF_PLANNING, MEAN_CLUSTER = icc, deff, deff_p, mbar

clusters (physicians)          248
mean cluster size              19.0685 weeks per physician
ACHIEVED intracluster correlation of the event indicator  0.05299
ACHIEVED design effect         1.95744
ACHIEVED effective sample      2415.9 of 4,729 person-weeks

PLANNING intracluster correlation 0.127 (design-stage assumption)
PLANNING design effect         3.2947
PLANNING effective sample      1435.3 person-weeks


## S3.5 Sensitivity analyses and other analyses

### Cell 57 — A31: Estimator-family comparison for the incidence model

**Pseudocode.**

1. The primary incidence estimate comes from one recurrent-event family. Refit the
   SAME pre-specified specification under several defensible families and read across
   them, so that no substantive claim rests on one estimator's assumptions.
2. Families fitted here: Andersen-Gill on the counting-process panel without a
   frailty term, offset Poisson at the physician level with robust variance, and
   negative-binomial at the physician level with the same offset.
3. The Prentice-Williams-Peterson stratification and the shared gamma-frailty fit are
   estimated in the R notebook; they belong to the same comparison and are read
   alongside these.
4. Report the four pre-specified effects under each family, not a single headline
   number.


In [57]:
COV = ["age10", "school_kids_any", "ili_freq_ord4", "vax_protected"]
ag = CoxTimeVaryingFitter().fit(pw[["participant_id", "tstart", "tstop", "event"] + COV],
                               id_col="participant_id", event_col="event",
                               start_col="tstart", stop_col="tstop", show_progress=False)
D = pw.groupby("participant_id").agg(episodes=("new_episode", "sum"),
                                     at_risk=("at_risk_new_episode", "sum"),
                                     age10=("age10", "first"),
                                     school_kids_any=("school_kids_any", "first"),
                                     ili_freq_ord4=("ili_freq_ord4", "first"))
# time-varying vaccination collapses at the physician level to the protected share
D["vax_protected"] = pw.groupby("participant_id").vax_protected.mean()
FML = "episodes ~ age10 + school_kids_any + ili_freq_ord4 + vax_protected"
po = smf.glm(FML, data=D, family=sm.families.Poisson(),
             offset=np.log(D.at_risk)).fit(cov_type="HC0")
nbf = sm.NegativeBinomial(D.episodes, sm.add_constant(D[COV]),
                          offset=np.log(D.at_risk), loglike_method="nb2").fit(disp=0)
out = pd.DataFrame({
    "Andersen-Gill (no frailty)": np.exp(ag.params_[COV]),
    "offset Poisson, robust": np.exp(po.params[COV]),
    "offset negative-binomial": np.exp(nbf.params[COV])}).round(4)
print("ratio-scale effects for the same pre-specified specification")
print(out.to_string())
print("\nthe shared gamma-frailty Andersen-Gill fit of record and the "
      "Prentice-Williams-Peterson stratified fit are estimated in the R notebook and "
      "are read alongside this table")
record("A31", ag={k: float(v) for k, v in np.exp(ag.params_[COV]).items()},
       poisson={k: float(v) for k, v in np.exp(po.params[COV]).items()},
       nb={k: float(v) for k, v in np.exp(nbf.params[COV]).items()})
AG_UNWEIGHTED, AG_COV = ag, COV

ratio-scale effects for the same pre-specified specification
                 Andersen-Gill (no frailty)  offset Poisson, robust  offset negative-binomial
age10                                0.8513                  0.8580                    0.8611
school_kids_any                      1.2917                  1.2969                    1.3704
ili_freq_ord4                        1.3098                  1.2723                    1.2950
vax_protected                        0.9541                  0.9420                    0.9245

the shared gamma-frailty Andersen-Gill fit of record and the Prentice-Williams-Peterson stratified fit are estimated in the R notebook and are read alongside this table


### Cell 58 — A35: Minimum detectable vaccine effect and achieved power

**Pseudocode.**

1. Under Schoenfeld's formula the standard error of a log hazard ratio is one over
   the square root of the event count times the exposed share times one minus the
   exposed share. The exposed share here is the protected person-time share.
2. THREE DISTINCT QUANTITIES follow and must not be confused:
   the all-ILI DESIGN boundary on all 497 events;
   the A-cluster DESIGN boundary on the 91 A-cluster events, which is the boundary of
   record for the vaccine contrast;
   the effect implied by the A-cluster fit's OWN observed standard error, which is not
   a design boundary at all.
3. A detectable effect at 80% power uses the sum of the two-sided critical value and
   the one-sided power quantile, not the critical value alone.
4. Also report the power the study had at an assumed true effect, computed from the
   observed standard error.


In [58]:
Z = stats.norm.ppf(0.975) + stats.norm.ppf(0.80)      # 80% power, two-sided 5%
p_exp = PROTECTED_SHARE
def schoenfeld_se(events, p=p_exp):
    return np.sqrt(1 / (events * p * (1 - p)))
print(f"protected person-time share  {p_exp:.4f}")
rows = []
for lab, ev in [("all-ILI design boundary", int(pw.new_episode.sum())),
                ("A-cluster design boundary (boundary of record)",
                 int((episodes.cluster_letter == 'A').sum()))]:
    se = schoenfeld_se(ev); hr = np.exp(-Z * se)
    rows.append(dict(quantity=lab, events=ev, se_log_HR=round(se, 5),
                     HR=round(hr, 5), VE_pct=round(100*(1-hr), 3)))
SE_OBSERVED_A = 0.2772                # observed se of the A-cluster log hazard ratio
hr = np.exp(-Z * SE_OBSERVED_A)
rows.append(dict(quantity="effect implied by the A-cluster fit's own observed standard error",
                 events=int((episodes.cluster_letter == 'A').sum()),
                 se_log_HR=SE_OBSERVED_A, HR=round(hr, 5), VE_pct=round(100*(1-hr), 3)))
print(pd.DataFrame(rows).to_string(index=False))
print("\nthe observed A-cluster effectiveness of 46.3% sits ON the A-cluster design "
      "boundary; the third row is a different quantity and is never attached to the "
      "primary contrast")
for ve in (0.30,):
    b = abs(np.log(1 - ve))
    power = stats.norm.cdf(b / SE_OBSERVED_A - stats.norm.ppf(0.975))
    print(f"power at a true effectiveness of {100*ve:.0f}%  {100*power:.1f}% "
          "(from the observed standard error)")
    # Winner's curse: the expected |log HR| among estimates that reach significance,
    # given the true effect, against the true |log HR| itself.
    c = stats.norm.ppf(0.975) * SE_OBSERVED_A          # significance boundary on |log HR|
    mu, sg = b, SE_OBSERVED_A
    alpha_ = (c - mu) / sg
    e_trunc = mu + sg * stats.norm.pdf(alpha_) / (1 - stats.norm.cdf(alpha_))
    print(f"  among estimates that reach significance the expected effect on the log scale "
          f"is {e_trunc:.4f} against a true {b:.4f}, so a significant result overstates the "
          f"effect by about {e_trunc/b:.1f}-fold at this power")
record("A35", boundary_all=rows[0], boundary_A=rows[1], implied_obs=rows[2],
       power30=float(stats.norm.cdf(abs(np.log(0.7))/SE_OBSERVED_A - stats.norm.ppf(0.975))))

protected person-time share  0.3411
                                                         quantity  events  se_log_HR      HR  VE_pct
                                          all-ILI design boundary     497    0.09462 0.76714  23.286
                   A-cluster design boundary (boundary of record)      91    0.22112 0.53822  46.178
effect implied by the A-cluster fit's own observed standard error      91    0.27720 0.45997  54.003

the observed A-cluster effectiveness of 46.3% sits ON the A-cluster design boundary; the third row is a different quantity and is never attached to the primary contrast
power at a true effectiveness of 30%  25.0% (from the observed standard error)
  among estimates that reach significance the expected effect on the log scale is 0.7088 against a true 0.3567, so a significant result overstates the effect by about 2.0-fold at this power


### Cell 59 — A36: Naive ever-vaccinated comparison (two-week interval sensitivity)

**Pseudocode.**

1. The exposure coding of record makes a person-week protected only two weeks after
   vaccination. A naive alternative treats every person-week of a vaccinated physician
   as exposed, so weeks before vaccination and weeks inside the interval count as
   protected.
2. Refit the incidence model under the naive coding, unadjusted and adjusted, and put
   both beside the time-varying estimate.
3. The naive coding attributes to vaccination the person-time before the vaccine could
   act, so it inflates apparent effectiveness. Report all three to make that visible.


In [59]:
P = pw.assign(vax_ever=pw.vax)
ag_naive_un = CoxTimeVaryingFitter().fit(
    P[["participant_id", "tstart", "tstop", "event", "vax_ever"]], id_col="participant_id",
    event_col="event", start_col="tstart", stop_col="tstop", show_progress=False)
ag_naive_adj = CoxTimeVaryingFitter().fit(
    P[["participant_id", "tstart", "tstop", "event", "vax_ever", "age10",
       "school_kids_any", "ili_freq_ord4"]], id_col="participant_id",
    event_col="event", start_col="tstart", stop_col="tstop", show_progress=False)
rows = [("naive ever-vaccinated, unadjusted", float(np.exp(ag_naive_un.params_["vax_ever"]))),
        ("naive ever-vaccinated, adjusted", float(np.exp(ag_naive_adj.params_["vax_ever"]))),
        ("time-varying coding of record, adjusted",
         float(np.exp(AG_UNWEIGHTED.params_["vax_protected"])))]
print(pd.DataFrame([dict(coding=k, HR=round(v, 4), VE_pct=round(100*(1-v), 2))
                    for k, v in rows]).to_string(index=False))
print(f"\nthe naive coding counts {int((pw.vax==1).sum()) - int(pw.vax_protected.sum())} "
      "person-weeks as protected that precede protection, which inflates apparent "
      "effectiveness")
record("A36", naive_unadj=rows[0][1], naive_adj=rows[1][1], time_varying=rows[2][1])

                                 coding     HR  VE_pct
      naive ever-vaccinated, unadjusted 0.9565    4.35
        naive ever-vaccinated, adjusted 0.9302    6.98
time-varying coding of record, adjusted 0.9541    4.59

the naive coding counts 261 person-weeks as protected that precede protection, which inflates apparent effectiveness


### Cell 60 — A37: Vaccinated-versus-unvaccinated covariate balance

**Pseudocode.**

1. Vaccination was not randomised, so the vaccinated and unvaccinated groups may
   differ on the covariates the model adjusts for.
2. Compare the two groups over the pre-specified adjustment set and the main baseline
   characteristics, as proportions or means with standardized mean differences.
3. Any imbalance the model adjusts for is handled; an imbalance on a variable the
   model does not carry is a residual-confounding concern and is named as such.


In [60]:
rep = phys[phys.reported == 1]
V, U = rep[rep.vax == 1], rep[rep.vax == 0]
ITEMS = [("comp_risk_any_nan", "any health condition conferring complication risk", True),
         ("school_kids_any", "household school-age child", True),
         ("age", "age, years (mean)", False), ("female", "female", True),
         ("university", "university hospital", True),
         ("sees_pat", "performs face-to-face examination", True),
         ("ili_freq_ord4", "susceptibility (ordinal mean)", False),
         ("prev_vax", "prior-season vaccination", True),
         ("hh", "household size (mean)", False)]
rows = []
for v, lab, binary in ITEMS:
    a, b = V[v].dropna(), U[v].dropna()
    fmt = (lambda s: f"{100*s.mean():.1f}%") if binary else (lambda s: f"{s.mean():.2f}")
    rows.append(dict(characteristic=lab, vaccinated=fmt(a), unvaccinated=fmt(b),
                     SMD=round(smd(a, b, binary), 4),
                     in_adjustment_set=v in ("age", "school_kids_any", "ili_freq_ord4")))
print(f"vaccinated {len(V)}, unvaccinated {len(U)}")
print(pd.DataFrame(rows).to_string(index=False))
resid = [r["characteristic"] for r in rows
         if abs(r["SMD"]) > 0.10 and not r["in_adjustment_set"]]
print("\nimbalanced (|SMD| > 0.10) and NOT in the adjustment set — residual-confounding "
      "concerns: " + ("; ".join(resid) if resid else "none"))
record("A37", comorbidity_vax=100*V.comp_risk_any_nan.mean(),
       comorbidity_unvax=100*U.comp_risk_any_nan.mean(),
       children_vax=100*V.school_kids_any.mean(), children_unvax=100*U.school_kids_any.mean(),
       n_residual=len(resid))

vaccinated 91, unvaccinated 157
                                   characteristic vaccinated unvaccinated     SMD  in_adjustment_set
any health condition conferring complication risk      31.1%        16.4%  0.3497              False
                       household school-age child      49.5%        35.0%  0.2951               True
                                age, years (mean)      41.53        39.72  0.1674               True
                                           female      64.8%        67.5% -0.0567              False
                              university hospital      30.8%        49.0% -0.3799              False
                performs face-to-face examination      47.3%        43.9%  0.0664              False
                    susceptibility (ordinal mean)       1.42         1.33  0.1336               True
                         prior-season vaccination      78.0%        17.8%  1.5094              False
                            household size (mean)       2.6

### Cell 61 — A44: Detrended and first-difference concordance checks

**Pseudocode.**

1. Both series rise and fall over the season, so part of their correlation may be a
   shared trend rather than week-to-week co-movement.
2. Remove a linear trend from each series within the window under test and correlate
   the residuals.
3. Separately correlate the first differences, which is the strictest form of the
   question: do the two series move together from one week to the next?
4. Report both beside the level correlation. A first-difference coefficient much
   smaller than the level coefficient means the agreement is largely in the trend.


In [61]:
t = np.arange(N_WEEKS)
def detrended(s, m):
    x, y = t[m], np.asarray(s)[m]
    r = stats.linregress(x, y)
    return y - (r.intercept + r.slope * x)
rows = []
for lab, m in [("pre-epidemic", PRE), ("epidemic", EPI),
               ("early-warning window", EARLY), ("full season", SEASON)]:
    lvl = stats.pearsonr(np.asarray(PART_SM)[m], np.asarray(SENT_SM)[m])[0]
    det = stats.pearsonr(detrended(PART_SM, m), detrended(SENT_SM, m))[0]
    dp, ds = np.diff(np.asarray(PART_SM)), np.diff(np.asarray(SENT_SM))
    md = m[1:]
    fd = stats.pearsonr(dp[md], ds[md])[0]
    rows.append(dict(window=lab, n=int(m.sum()), level=round(lvl, 5),
                     detrended=round(det, 5), first_difference=round(fd, 5)))
print(pd.DataFrame(rows).to_string(index=False))
print("\nwhere the first-difference coefficient falls well below the level coefficient, "
      "the agreement rests mainly on the shared seasonal trend")
record("A44", pre_detrended=rows[0]["detrended"], ew_detrended=rows[2]["detrended"],
       ew_first_diff=rows[2]["first_difference"])

              window  n   level  detrended  first_difference
        pre-epidemic 10 0.88609    0.65126           0.66852
            epidemic  9 0.75414    0.55618           0.21579
early-warning window 19 0.64196    0.71564           0.26618
         full season 33 0.53804    0.53699           0.16712

where the first-difference coefficient falls well below the level coefficient, the agreement rests mainly on the shared seasonal trend


### Cell 62 — A45: Complete-window smoothing sensitivity

**Pseudocode.**

1. The causal three-week average retains partial windows in the season's first two
   weeks, where fewer than three weeks are available. That retention is part of the
   specification.
2. Refit the full-season correlation using complete three-week windows only, which
   drops those two weeks.
3. Report both, since dropping partial windows moves the coefficient materially and
   the reader must see which convention produced which number.


In [62]:
p_partial, s_partial = cma3(part_share), cma3(sent_share)
p_complete = pd.Series(part_share).rolling(3).mean()
s_complete = pd.Series(sent_share).rolling(3).mean()
m = p_complete.notna().values
r_part = stats.pearsonr(p_partial, s_partial)
r_comp = stats.pearsonr(p_complete[m], s_complete[m])
print(f"partial windows retained (specification of record)  r = {r_part[0]:+.5f}, "
      f"p = {r_part[1]:.6f}, n = {N_WEEKS}")
print(f"complete windows only                               r = {r_comp[0]:+.5f}, "
      f"p = {r_comp[1]:.6f}, n = {int(m.sum())}")
print(f"difference  {r_comp[0]-r_part[0]:+.5f} — the convention is stated wherever the "
      "coefficient is reported")
record("A45", r_partial=float(r_part[0]), r_complete=float(r_comp[0]), n_complete=int(m.sum()))

partial windows retained (specification of record)  r = +0.53804, p = 0.001240, n = 33
complete windows only                               r = +0.61849, p = 0.000209, n = 31
difference  +0.08044 — the convention is stated wherever the coefficient is reported


### Cell 63 — A48: Pairing-broken resampling comparison

**Pseudocode.**

1. An alternative bootstrap resamples the week INDEX first and only then forms the
   lagged pairs. This looks similar but destroys the pairing that the lag procedure is
   about: after resampling, a week's neighbour at lag L is no longer the week that was
   actually L weeks away.
2. Run it and report the result, then state plainly that the concentration it produces
   at lag zero is an artefact of the procedure and not a finding.
3. The lag-preserving procedure — pairs formed first, then pairs resampled — is the
   procedure of record.


In [63]:
s_broken = peak_lag_bootstrap(PART_SM, SENT_SM, seed=20260707, preserve_pairing=False)
vc_b = s_broken.value_counts(normalize=True)
s_ok = peak_lag_bootstrap(PART_SM, SENT_SM, seed=20260707, preserve_pairing=True)
vc_o = s_ok.value_counts(normalize=True)
print("pairing-broken procedure (resample the index, then form pairs)")
print(f"  modal lag {int(s_broken.mode().iloc[0]):+d} in "
      f"{100*vc_b.iloc[0]:.1f}% of resamples; lag 0 in {100*vc_b.get(0,0):.1f}%")
print("procedure of record (form pairs, then resample pairs)")
print(f"  modal lag {int(s_ok.mode().iloc[0]):+d} in "
      f"{100*vc_o.iloc[0]:.1f}% of resamples; lag 0 in {100*vc_o.get(0,0):.1f}%")
print("\nthe pairing-broken procedure concentrates the peak at lag 0 in the large "
      "majority of resamples. That concentration is an artefact of destroying the "
      "lag pairing, not evidence of synchrony; the lag-preserving procedure is the "
      "procedure of record.")
record("A48", broken_modal=int(s_broken.mode().iloc[0]), broken_lag0=100*vc_b.get(0, 0),
       ok_modal=int(s_ok.mode().iloc[0]), ok_lag0=100*vc_o.get(0, 0))

pairing-broken procedure (resample the index, then form pairs)
  modal lag +0 in 91.3% of resamples; lag 0 in 91.3%
procedure of record (form pairs, then resample pairs)
  modal lag -1 in 27.7% of resamples; lag 0 in 18.4%

the pairing-broken procedure concentrates the peak at lag 0 in the large majority of resamples. That concentration is an artefact of destroying the lag pairing, not evidence of synchrony; the lag-preserving procedure is the procedure of record.


### Cell 64 — A57: CUSUM reset-rule equivalence check

**Pseudocode.**

1. THE RULE OF RECORD IS THE RESET: the negative-binomial CUSUM zeroes its accumulator
   after an alarm, so one excursion above the decision interval raises one alarm. This
   block exists to show what that clause is worth, by scoring the alternatives beside it.
2. The alternatives are the persistent accumulator, which keeps the statistic running and
   alarms in every week it stays above the interval, and the first-upcrossing rule, which
   keeps it running but alarms only in the week it first crosses. If two alarms never fall
   in consecutive weeks the three rules can coincide; if the accumulator stays above the
   interval for several weeks they do not.
3. Compare the alarm sets under all three rules across a grid of decision intervals around
   each series' own calibrated value, on both series.
4. Report whether the sets are identical in every combination, and whether any alarm falls
   in consecutive weeks, since that is the condition under which the rules can agree. THE
   RESET IS LOAD-BEARING AT THE CALIBRATED INTERVALS: the accumulated log-likelihood ratio
   stays above the interval for several weeks on both count series, so the persistent
   accumulator raises many more alarms than the reset rule does and the rules are not
   interchangeable.

In [64]:
def nb_cusum_rule(counts, h, rule="reset", base=BASELINE_WEEKS,
                  mult=NBCUSUM_MULT, warm=WARMUP_WEEKS):
    """rule: 'reset' zeroes the accumulator after an alarm (the rule of record);
    'persist' leaves it running and alarms in every week it stays above h;
    'upcross' leaves it running but alarms only when it first crosses h."""
    y = np.asarray(counts, float); n = len(y); a = np.zeros(n, int); S = prev = 0.0
    for t in range(warm, n):
        b = y[t - base:t]
        mu0 = np.nanmean(b); v = np.nanvar(b, ddof=1)
        if not np.isfinite(mu0) or mu0 <= 0:
            continue
        size = 1.0 / max((v - mu0) / mu0 ** 2, 1e-8)
        mu1 = mult * mu0
        S = max(0.0, S + (stats.nbinom.logpmf(y[t], size, size / (size + mu1))
                          - stats.nbinom.logpmf(y[t], size, size / (size + mu0))))
        if rule == "reset":
            if S > h:
                a[t] = 1; S = 0.0
        elif rule == "persist":
            if S > h:
                a[t] = 1
        else:
            if S > h and prev <= h:
                a[t] = 1
        prev = S
    return a

rows = []
for series_lab, y, h0 in [("participatory A-cluster count", PART_COUNTS,
                           NBCUSUM_H_PARTICIPATORY),
                          ("sentinel positive count", SENT_COUNTS,
                           NBCUSUM_H_SENTINEL)]:
    for f in (1.0, 1.5, 2.0):
        h = round(h0 * f, 4)
        A_r = set(WEEKS[nb_cusum_rule(y, h, "reset") == 1])
        A_p = set(WEEKS[nb_cusum_rule(y, h, "persist") == 1])
        A_u = set(WEEKS[nb_cusum_rule(y, h, "upcross") == 1])
        idx = sorted(list(WEEKS).index(x) for x in A_r)
        rows.append(dict(series=series_lab, h=h, multiple_of_calibrated=f,
                         n_reset=len(A_r), n_persist=len(A_p), n_upcross=len(A_u),
                         reset_eq_persist=A_r == A_p, reset_eq_upcross=A_r == A_u,
                         consecutive_reset_alarms=bool(len(idx) > 1
                                                       and np.any(np.diff(idx) == 1))))
E = pd.DataFrame(rows)
print(E.to_string(index=False))
print(f"\nreset and persistent-accumulator rules give identical alarm sets in "
      f"{int(E.reset_eq_persist.sum())} of {len(E)} parameter combinations")
print(f"reset and first-upcrossing rules agree in {int(E.reset_eq_upcross.sum())} of {len(E)}")
print(f"consecutive alarm weeks occur under the reset rule in "
      f"{int(E.consecutive_reset_alarms.sum())} of {len(E)} combinations")
print("\nThe rules coincide only where no alarm falls in consecutive weeks. At the "
      "calibrated decision intervals the accumulated log-likelihood ratio stays above "
      "the interval for several weeks on both count series, so the rules diverge: the "
      "persistent accumulator keeps alarming while it remains above the interval and "
      "the first-upcrossing rule alarms once per excursion. The reset rule is the rule "
      "of record and is stated wherever an alarm set is reported.")
record("A57", n_identical_persist=int(E.reset_eq_persist.sum()),
       n_identical_upcross=int(E.reset_eq_upcross.sum()), n_combinations=len(E),
       n_consecutive=int(E.consecutive_reset_alarms.sum()))

                       series    h  multiple_of_calibrated  n_reset  n_persist  n_upcross  reset_eq_persist  reset_eq_upcross  consecutive_reset_alarms
participatory A-cluster count 1.40                     1.0        2          7          1             False             False                      True
participatory A-cluster count 2.10                     1.5        1          5          1             False              True                     False
participatory A-cluster count 2.80                     2.0        1          4          1             False              True                     False
      sentinel positive count 2.52                     1.0        2         15          1             False             False                     False
      sentinel positive count 3.78                     1.5        1         12          1             False              True                     False
      sentinel positive count 5.04                     2.0        1         12          

### Cell 65 — A58: Specification curve for alarm concordance

**Pseudocode.**

1. Enumerate every defensible specification of the alarm-concordance analysis rather
   than reporting one. The enumeration is the product set carried by the aberration
   analysis: three measured quantities, three smoothing conventions, two baseline
   lengths, two standard-deviation thresholds and two consensus rules, giving
   seventy-two specifications.
2. The three measured quantities are the A-cluster share of incident episodes against
   sentinel positivity, which is the pair of record; the all-ILI weekly count of each
   system, that is every cleaned incident episode against every positive specimen; and
   a per-denominator rate, A-cluster episodes per hundred reporters against positive
   specimens per hundred consultations.
3. The three smoothing conventions are the raw weekly series, the causal three-week
   average of record and a symmetric three-week average. Each specification is built from
   the UNSMOOTHED quantity and smoothed once, so no series is smoothed twice. The partial
   windows at the season's opening and closing are RETAINED under all three conventions,
   which is the convention the panel of record is produced under; dropping them instead
   costs the participatory chart its closing-week alarm and the sentinel chart its
   2025-W52 alarm. Complete-window smoothing is scored as its own sensitivity analysis,
   not silently inside this grid.
4. In each specification run the whole six-detector bank on both series, with the
   specification's own baseline length and standard-deviation threshold in force, and
   compute Cohen's kappa between the two consensus alarm series. The negative-binomial
   CUSUM keeps its series-specific calibrated decision intervals throughout, since
   those were calibrated on the count series and not on the quantity under variation.
5. Score every specification on the full thirty-three-week season, which is the window
   of record for this curve, and on the nineteen-week early-warning window alongside.
6. Report the median, the range, the interquartile range, the share reaching
   substantial agreement at or above 0.60 and the share below 0.40, and identify the
   specifications producing the maximum and the minimum.
7. Report the median kappa separately by measured quantity, smoothing convention and
   consensus rule; locate the specification of record on the curve by its rank; and
   name every specification that scores above it, with the margin by which it does.

In [65]:
# ============================================================================
# Specification curve for alarm concordance. The enumeration is the product set
# carried by the aberration analysis: three measured quantities x three smoothing
# conventions x two baseline lengths x two standard-deviation thresholds x two
# consensus rules = 72 specifications. Each specification re-runs the whole
# six-detector bank on both streams from the UNSMOOTHED quantity, so no series is
# smoothed twice.
# ============================================================================
QUANTITIES = {
    # the monitored quantity of record: the A-cluster share of incident episodes
    # against sentinel positivity
    "A-cluster positivity (%)":  (part_share.values, sent_share.values),
    # the all-ILI weekly count of each system: every cleaned incident episode
    # against every positive specimen
    "weekly count":              (wk.n_new_episodes.values.astype(float),
                                  wk.sb_positive_cases.values.astype(float)),
    # a per-denominator rate: A-cluster episodes per 100 reporters against
    # positives per 100 consultations
    "rate per 100":              (100 * wk.n_new_A.values / wk.n_reporters.values,
                                  100 * wk.sb_positive_cases.values
                                  / wk.sb_ili_consultations.values)}
SMOOTHINGS = ("raw", "causal MA-3", "symmetric MA-3")
BASELINES, THRESHOLDS, CONSENSUS = (5, 7), (2.0, 3.0), (1, 2)
# Scoring windows. The full season is the window of record for this curve; the
# nineteen-week early-warning window is scored alongside because the alarm
# analysis exists to describe performance BEFORE and DURING the wave.
SCORING = [("full season (33 weeks)", SEASON),
           ("early-warning window (19 weeks)", EARLY)]

def smooth_series(x, kind):
    """Apply one smoothing convention to a raw quantity. Partial windows are
    retained (min_periods = 1), the same convention as cma3() in the setup cell,
    so each specification is run on the series its own label describes."""
    x = np.asarray(x, float)
    if kind == "raw":
        return x
    return pd.Series(x).rolling(3, min_periods=1,
                                center=(kind == "symmetric MA-3")).mean().values

# Season-boundary convention. Every specification is smoothed with min_periods = 1, so
# the partial windows at the season's opening and closing are RETAINED and no smoothed
# series carries a missing value. This is the convention of the setup cell's cma3() and
# therefore the convention the panel of record is produced under: dropping the partial
# windows instead costs the participatory EWMA its 2026-W20 alarm and the sentinel EWMA
# its 2025-W52 alarm, which are two of the panel's cells. Complete-window smoothing is
# scored as its own sensitivity analysis (A45), not silently inside this grid.

def kappa_on(x, y, m):
    xs, ys = x[m], y[m]
    if xs.std() == 0 and ys.std() == 0:
        return np.nan
    return round(float(cohen_kappa_score(xs, ys)), 5)

rows = []
for q_lab, (pq, sq) in QUANTITIES.items():
    for sm_lab in SMOOTHINGS:
        for base in BASELINES:
            for thr in THRESHOLDS:
                dp = detector_panel(smooth_series(pq, sm_lab), PART_COUNTS,
                                    NBCUSUM_H_PARTICIPATORY, base=base, thr=thr)
                ds = detector_panel(smooth_series(sq, sm_lab), SENT_COUNTS,
                                    NBCUSUM_H_SENTINEL, base=base, thr=thr)
                for cons in CONSENSUS:
                    key = "any" if cons == 1 else "two"
                    x, y = dp[key], ds[key]
                    r = dict(spec_id=len(rows) + 1, measured_quantity=q_lab,
                             smoothing=sm_lab, baseline_weeks=base, sd_threshold=thr,
                             consensus_rule=f">={cons}",
                             n_part_alarms=int(x.sum()), n_sent_alarms=int(y.sum()))
                    for w_lab, m in SCORING:
                        r[f"kappa — {w_lab}"] = kappa_on(x, y, m)
                    rows.append(r)
SC = pd.DataFrame(rows)
SC["is_specification_of_record"] = ((SC.measured_quantity == "A-cluster positivity (%)")
                                    & (SC.smoothing == "causal MA-3")
                                    & (SC.baseline_weeks == 7) & (SC.sd_threshold == 3.0)
                                    & (SC.consensus_rule == ">=1"))
print(f"{len(SC)} specifications enumerated "
      f"({len(QUANTITIES)} quantities x {len(SMOOTHINGS)} smoothings x "
      f"{len(BASELINES)} baseline lengths x {len(THRESHOLDS)} thresholds x "
      f"{len(CONSENSUS)} consensus rules)")

summary = []
for w_lab, _ in SCORING:
    col = f"kappa — {w_lab}"
    k = SC[col].dropna()
    q1, q3 = k.quantile([0.25, 0.75])
    mx, mn = SC.loc[SC[col].idxmax()], SC.loc[SC[col].idxmin()]
    lock_k = float(SC.loc[SC.is_specification_of_record, col].iloc[0])
    n_above = int((k > lock_k + 1e-9).sum())
    def spec_str(r):
        return (f"{r.measured_quantity}, {r.smoothing}, {r.baseline_weeks}-week baseline, "
                f"{r.sd_threshold} SD, consensus {r.consensus_rule}")
    print(f"\n--- scored on the {w_lab}")
    print(f"    median {k.median():.4f}   range {k.min():.4f} to {k.max():.4f}   "
          f"interquartile range {q1:.4f} to {q3:.4f}")
    print(f"    substantial agreement (kappa >= 0.60)  {100*(k>=0.60).mean():.1f}%")
    print(f"    fair or poor (kappa < 0.40)            {100*(k<0.40).mean():.1f}%")
    print(f"    maximum  {mx[col]:.5f}  {spec_str(mx)}")
    print(f"    minimum  {mn[col]:.5f}  {spec_str(mn)}")
    print(f"    the specification of record sits at {lock_k:.5f}, ranking "
          f"{n_above + 1} of {len(k)}, above {100*(k<lock_k).mean():.0f}% of the "
          f"curve, tied with {int((abs(k-lock_k)<1e-9).sum())-1} other specifications")
    if n_above:
        above = SC.loc[SC[col] > lock_k + 1e-9].sort_values(col, ascending=False)
        for _, r in above.iterrows():
            print(f"      higher by {r[col]-lock_k:+.4f}: {spec_str(r)}")
    summary.append(dict(scoring_window=w_lab, n_specifications=len(k),
                        median=round(float(k.median()), 5), minimum=round(float(k.min()), 5),
                        maximum=round(float(k.max()), 5), iqr_lower=round(float(q1), 5),
                        iqr_upper=round(float(q3), 5),
                        pct_at_or_above_0_60=round(100*float((k >= 0.60).mean()), 4),
                        pct_below_0_40=round(100*float((k < 0.40).mean()), 4),
                        specification_at_maximum=spec_str(mx),
                        specification_at_minimum=spec_str(mn),
                        kappa_of_specification_of_record=lock_k,
                        rank_of_specification_of_record=n_above + 1,
                        n_above_specification_of_record=n_above,
                        n_tied_with_specification_of_record=int((abs(k-lock_k)<1e-9).sum())-1))
    for lab, c in [("measured quantity", "measured_quantity"),
                   ("smoothing convention", "smoothing"),
                   ("consensus rule", "consensus_rule")]:
        print(f"\n    median kappa by {lab}")
        print(SC.groupby(c)[col].agg(["median", "min", "max", "count"]).round(4).to_string())

SUMMARY = pd.DataFrame(summary)
print("\nInternal consistency: in each scoring window the grid row carrying the "
      "specification of record equals the value that same specification produces in the "
      "aberration panel (A56):")
print(f"  full season          grid {float(SC.loc[SC.is_specification_of_record, 'kappa — full season (33 weeks)'].iloc[0]):.5f}  "
      f"panel {CONCORD_ROWS['full season']['kappa']:.5f}")
print(f"  early-warning window grid {float(SC.loc[SC.is_specification_of_record, 'kappa — early-warning window (19 weeks)'].iloc[0]):.5f}  "
      f"panel {CONCORD_ROWS['early-warning window (pre-epidemic and epidemic, 19 weeks)']['kappa']:.5f}")

SC.to_csv("specification_curve.csv", index=False)
SUMMARY.to_csv("specification_curve_summary.csv", index=False)
record("A58", n_specs=len(SC), **{f"{k}_{r['scoring_window'][:5]}": v
       for r in summary for k, v in r.items() if isinstance(v, (int, float))})
SPEC_CURVE, SPEC_SUMMARY = SC, SUMMARY

72 specifications enumerated (3 quantities x 3 smoothings x 2 baseline lengths x 2 thresholds x 2 consensus rules)

--- scored on the full season (33 weeks)
    median 0.5296   range 0.1149 to 0.8406   interquartile range 0.4278 to 0.7238
    substantial agreement (kappa >= 0.60)  36.1%
    fair or poor (kappa < 0.40)            22.2%
    maximum  0.84058  rate per 100, causal MA-3, 7-week baseline, 3.0 SD, consensus >=2
    minimum  0.11494  A-cluster positivity (%), raw, 5-week baseline, 2.0 SD, consensus >=2
    the specification of record sits at 0.83500, ranking 2 of 72, above 94% of the curve, tied with 2 other specifications
      higher by +0.0056: rate per 100, causal MA-3, 7-week baseline, 3.0 SD, consensus >=2

    median kappa by measured quantity
                          median     min     max  count
measured_quantity                                      
A-cluster positivity (%)  0.5268  0.1149  0.8350     24
rate per 100              0.7171  0.2979  0.8406     24
weekly

### Cell 66 — A59: Alarm-window influence of the sparse closing week

**Pseudocode.**

1. The final week of the season is flagged by the sparse-week rule and it carries a
   participatory alarm that the sentinel series does not.
2. Recompute the full-season agreement with that week omitted, and report both values.
3. This is influence disclosure, not exclusion: the value of record is the one computed
   on all thirty-three weeks.


In [66]:
a, b = D_PART["any"], D_SENT["any"]
k_all = cohen_kappa_score(a, b)
m = np.r_[np.ones(N_WEEKS - 1, bool), False]
k_excl = cohen_kappa_score(a[m], b[m])
print(f"full season, all {N_WEEKS} weeks (value of record)  kappa {k_all:.5f}")
print(f"omitting the flagged closing week {WEEKS[-1]}, {int(m.sum())} weeks  "
      f"kappa {k_excl:.5f}")
print(f"the flagged week costs {k_excl-k_all:+.5f} of kappa; it carries a participatory "
      "alarm the sentinel series does not")
print("\nthe closing week is retained in every reported analysis; this block discloses its "
      "influence and does not remove it")

ALARM_COMPARISON = pd.DataFrame({
    "week": WEEKS,
    "participatory_n_detectors": D_PART["n_detectors"],
    "participatory_alarm": D_PART["any"],
    "participatory_alarm_two_of_six": D_PART["two"],
    "sentinel_n_detectors": D_SENT["n_detectors"],
    "sentinel_alarm": D_SENT["any"],
    "sentinel_alarm_two_of_six": D_SENT["two"],
    "phase": np.select([PRE, EPI, POST], ["pre-epidemic", "epidemic", "post-epidemic"],
                       default="unassigned"),
    "in_early_warning_window_19wk": EARLY.astype(int),
    "in_2025_calendar_portion_13wk": EARLY_2025.astype(int),
    "agree": (D_PART["any"] == D_SENT["any"]).astype(int)})
ALARM_COMPARISON.to_csv("alarm_comparison_by_week.csv", index=False)
print(f"\nweek-by-week alarm comparison written for all {N_WEEKS} weeks; "
      f"the two series agree in {int(ALARM_COMPARISON.agree.sum())} of {N_WEEKS}")
record("A59", kappa_all=float(k_all), kappa_excl=float(k_excl))

full season, all 33 weeks (value of record)  kappa 0.83500
omitting the flagged closing week 2026-W20, 32 weeks  kappa 0.91304
the flagged week costs +0.07804 of kappa; it carries a participatory alarm the sentinel series does not

the closing week is retained in every reported analysis; this block discloses its influence and does not remove it

week-by-week alarm comparison written for all 33 weeks; the two series agree in 31 of 33


### Cell 67 — A64: IPRW re-estimation of the primary incidence model

**Pseudocode.**

1. Inverse-probability-of-reporting weighting is a SENSITIVITY ANALYSIS, not the
   estimation method. The estimates of record are unweighted.
2. Attach each physician's stabilized reporting weight to every one of that
   physician's person-weeks and refit the incidence specification with those weights.
3. Compare each weighted coefficient with its unweighted counterpart, and express the
   shift as a percentage of that coefficient's OWN unweighted interval width, which is
   the scale on which a shift is or is not material.


In [67]:
P = pw.copy()
P["weight"] = P.participant_id.map(STAB_WEIGHTS).fillna(1.0)
wt = CoxTimeVaryingFitter().fit(
    P[["participant_id", "tstart", "tstop", "event"] + AG_COV + ["weight"]],
    id_col="participant_id", event_col="event", start_col="tstart", stop_col="tstop",
    weights_col="weight", show_progress=False)
ci = np.exp(AG_UNWEIGHTED.confidence_intervals_)
comp = pd.DataFrame({
    "HR_unweighted": np.exp(AG_UNWEIGHTED.params_[AG_COV]),
    "HR_IPRW": np.exp(wt.params_[AG_COV]),
    "unweighted_CI_width": (ci.iloc[:, 1] - ci.iloc[:, 0])[AG_COV].values})
comp["abs_shift"] = (comp.HR_IPRW - comp.HR_unweighted).abs()
comp["shift_pct_of_CI_width"] = 100 * comp.abs_shift / comp.unweighted_CI_width
print("weights are applied to a model whose estimates of record are unweighted")
print(comp.round(5).to_string())
big = comp.shift_pct_of_CI_width.idxmax()
print(f"\nlargest shift: {big} at {comp.shift_pct_of_CI_width.max():.1f}% of that "
      "coefficient's own unweighted interval width")
print("no coefficient moves by an amount comparable to its own uncertainty, so the "
      "unweighted estimates stand")
record("A64", largest_shift_pct=float(comp.shift_pct_of_CI_width.max()),
       largest_shift_term=str(big),
       shifts={k: float(v) for k, v in comp.shift_pct_of_CI_width.items()})

weights are applied to a model whose estimates of record are unweighted
                 HR_unweighted  HR_IPRW  unweighted_CI_width  abs_shift  shift_pct_of_CI_width
covariate                                                                                     
age10                  0.85128  0.83987              0.16133    0.01140                7.06893
school_kids_any        1.29168  1.27089              0.47865    0.02079                4.34345
ili_freq_ord4          1.30979  1.32108              0.37640    0.01129                2.99867
vax_protected          0.95408  0.93801              0.37592    0.01607                4.27453

largest shift: age10 at 7.1% of that coefficient's own unweighted interval width
no coefficient moves by an amount comparable to its own uncertainty, so the unweighted estimates stand


### Cell 68 — A69: Overdispersion sensitivity for the gradient

**Pseudocode.**

1. The reporting-intensity gradient of record is an offset Poisson fit with robust
   variance, which is consistent under overdispersion.
2. Refit the same specification as a negative-binomial model, which models the extra
   variance rather than correcting the standard errors for it.
3. Report both, plus the dispersion parameter and the formal overdispersion test. This
   block has no single scalar target; it is a comparison of two fits.


In [68]:
D, FULL = PHYS_LEVEL, GRADIENT_FORMULA
TERMS = ["weeks10", "age10", "school_kids_any", "ili_freq_ord4"]
po = smf.glm(FULL, data=D, family=sm.families.Poisson(),
             offset=np.log(D.at_risk)).fit(cov_type="HC0")
nbf = sm.NegativeBinomial(D.episodes, sm.add_constant(D[TERMS]),
                          offset=np.log(D.at_risk), loglike_method="nb2").fit(disp=0)
po_plain = smf.glm(FULL, data=D, family=sm.families.Poisson(),
                   offset=np.log(D.at_risk)).fit()
out = pd.DataFrame({
    "IRR_Poisson_robust": np.exp(po.params[TERMS]),
    "p_Poisson_robust": po.pvalues[TERMS],
    "IRR_negative_binomial": np.exp(nbf.params[TERMS]),
    "p_negative_binomial": nbf.pvalues[TERMS]}).round(5)
print(out.to_string())
alpha = nbf.params["alpha"]
lr = 2 * (nbf.llf - po_plain.llf)
print(f"\ndispersion parameter alpha  {alpha:.5f} (SE {nbf.bse['alpha']:.5f})")
print(f"overdispersion likelihood-ratio test  chi-square {lr:.4f}, "
      f"p = {0.5*stats.chi2.sf(lr,1):.2e} (one-sided, boundary parameter)")
print(f"Pearson dispersion of the Poisson fit  {po_plain.pearson_chi2/po_plain.df_resid:.4f}")
print("\nthe two families agree in direction and magnitude, so the gradient does not "
      "depend on how the extra variance is handled")
record("A69", irr_poisson=float(np.exp(po.params["weeks10"])),
       irr_nb=float(np.exp(nbf.params["weeks10"])), alpha=float(alpha), lr=float(lr))

                 IRR_Poisson_robust  p_Poisson_robust  IRR_negative_binomial  p_negative_binomial
weeks10                     0.80455           0.00069                0.82009              0.00317
age10                       0.85574           0.00913                0.85948              0.01018
school_kids_any             1.28233           0.03520                1.32452              0.01919
ili_freq_ord4               1.24463           0.00975                1.25401              0.01381

dispersion parameter alpha  0.25534 (SE 0.06775)
overdispersion likelihood-ratio test  chi-square 28.9896, p = 3.64e-08 (one-sided, boundary parameter)
Pearson dispersion of the Poisson fit  1.7023

the two families agree in direction and magnitude, so the gradient does not depend on how the extra variance is handled


### Cell 69 — A72: Risk-set sensitivity of the post-gap odds ratio

**Pseudocode.**

1. The post-gap coding comparison requires a preceding week in the physician's own
   record, so every physician's first observed week is outside the risk set. Several
   other risk sets are defensible.
2. Recompute the crude association across four risk sets: the one of record; all
   at-risk weeks including each first week; the risk set of record with the final
   season week removed; and all FILED weeks after the first observed week rather than
   only at-risk weeks.
3. Report the crude odds ratio in each, so the range is visible. The conditional
   estimate of record, which carries the physician random intercept, is fitted in the
   R notebook.


In [69]:
at_risk = pw[pw.at_risk_new_episode == 1]
not_first = ~pw.is_first_week
SETS = [("at-risk weeks after the first observed week (risk set of record)",
         at_risk[~at_risk.is_first_week]),
        ("all at-risk weeks, including each first week", at_risk),
        ("risk set of record, excluding the final season week",
         at_risk[(~at_risk.is_first_week) & (at_risk.week_idx < pw.week_idx.max())]),
        ("all filed weeks after the first observed week", pw[not_first])]
rows = []
for lab, sub in SETS:
    ct = pd.crosstab(sub.gap_before_derived, sub.new_episode)
    orv = (ct.loc[1, 1] * ct.loc[0, 0]) / (ct.loc[1, 0] * ct.loc[0, 1])
    se = np.sqrt(sum(1 / ct.values.flatten()))
    rows.append(dict(risk_set=lab, person_weeks=len(sub),
                     physicians=sub.participant_id.nunique(),
                     events=int(sub.new_episode.sum()), crude_OR=round(orv, 4),
                     lo=round(np.exp(np.log(orv) - 1.96*se), 3),
                     hi=round(np.exp(np.log(orv) + 1.96*se), 3)))
R = pd.DataFrame(rows)
print(R.to_string(index=False))
print(f"\ncrude odds ratio ranges {R.crude_OR.min():.3f} to {R.crude_OR.max():.3f} across "
      "the four risk sets")
print("the conditional estimate of record, with a per-physician random intercept, is "
      "fitted in the R notebook")
record("A72", or_range=[float(R.crude_OR.min()), float(R.crude_OR.max())],
       or_record=float(R.crude_OR.iloc[0]))

                                                        risk_set  person_weeks  physicians  events  crude_OR    lo    hi
at-risk weeks after the first observed week (risk set of record)          4378         232     449    1.7755 1.372 2.298
                    all at-risk weeks, including each first week          4626         248     497    1.6597 1.286 2.143
             risk set of record, excluding the final season week          4222         231     441    1.7453 1.343 2.269
                   all filed weeks after the first observed week          4481         233     449    1.8280 1.412 2.366

crude odds ratio ranges 1.660 to 1.828 across the four risk sets
the conditional estimate of record, with a per-physician random intercept, is fitted in the R notebook


### Cell 70 — A74: Bounding incidence over unfiled interior person-weeks

**Pseudocode.**

1. A physician's interior grid is every week from their first observed week to their
   last, whether filed or not. Unfiled interior weeks are weeks in which an episode
   could have occurred and gone unreported.
2. Enumerate those weeks and express them as a share of the interior grid.
3. Bound the season incidence deterministically across a stated sensitivity parameter
   k, the ratio of the episode rate in unfiled weeks to the observed rate: k below one
   means unfiled weeks were quieter, above one that they were busier.
4. Report the bound across a range of k so that the reader sees the whole band rather
   than one assumption.


In [70]:
first = pw.groupby("participant_id").week_idx.min()
last = pw.groupby("participant_id").week_idx.max()
interior = int((last - first + 1).sum())
unfiled = interior - len(pw)
print(f"interior grid (first to last observed week, per physician)  {interior:,} weeks")
print(f"filed person-weeks                                         {len(pw):,}")
print(f"unfiled interior person-weeks                              {unfiled} "
      f"({100*unfiled/interior:.4f}% of the interior grid)")
k_obs, n_ar = int(pw.new_episode.sum()), int(pw.at_risk_new_episode.sum())
observed_rate = k_obs / n_ar
rows = []
for kf in (0.5, 0.75, 1.0, 1.28, 1.46, 1.5, 2.0):
    extra = kf * unfiled * observed_rate
    rows.append(dict(k=kf, implied_extra_episodes=round(extra, 1),
                     season_incidence_per_100=round(100*(k_obs+extra)/(n_ar+unfiled), 4)))
print("\nbounding the season incidence per 100 at-risk person-weeks")
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nobserved rate at k = 1 reproduces the estimate of record, {100*observed_rate:.4f} "
      "per 100; the band over k from 0.5 to 2.0 is the deterministic bound")
record("A74", interior=interior, unfiled=unfiled, pct_unfiled=100*unfiled/interior,
       bounds=[r["season_incidence_per_100"] for r in rows])

interior grid (first to last observed week, per physician)  5,667 weeks
filed person-weeks                                         4,729
unfiled interior person-weeks                              938 (16.5520% of the interior grid)

bounding the season incidence per 100 at-risk person-weeks
   k  implied_extra_episodes  season_incidence_per_100
0.50                    50.4                    9.8380
0.75                    75.6                   10.2908
1.00                   100.8                   10.7436
1.28                   129.0                   11.2508
1.46                   147.1                   11.5768
1.50                   151.2                   11.6492
2.00                   201.6                   12.5548

observed rate at k = 1 reproduces the estimate of record, 10.7436 per 100; the band over k from 0.5 to 2.0 is the deterministic bound


### Cell 71 — A77: Cochran precision and prospective panel size

**Pseudocode.**

1. Under Cochran's formula the sample size for a target half-width on a proportion is
   the squared critical value times the proportion times its complement, over the
   squared half-width.
2. Invert it to state the half-width the achieved panel supports, using the effective
   sample rather than the nominal one, because person-weeks within a physician are
   correlated.
3. Report the independent-observation requirement for a five-percentage-point margin,
   and the number of reports that requirement implies once the design effect is applied.
4. Report the requirement under both the achieved and the planning design effect, and
   label which is which.


In [71]:
p_hat = float(wk.n_new_A.sum() / wk.n_new_episodes.sum())
z = stats.norm.ppf(0.975)
print(f"season positivity  {100*p_hat:.4f}%")
for lab, deff in [("achieved", DEFF_ACHIEVED), ("planning", DEFF_PLANNING)]:
    n_eff = len(pw) / deff
    hw = z * np.sqrt(p_hat * (1 - p_hat) / n_eff)
    print(f"{lab:9s} design effect {deff:.4f} -> effective sample {n_eff:.1f}; "
          f"half-width plus or minus {hw:.5f} ({100*hw:.2f} percentage points)")
n_ind = z ** 2 * p_hat * (1 - p_hat) / 0.05 ** 2
print(f"\nindependent observations for a 5-percentage-point margin  {int(np.ceil(n_ind))}")
for lab, deff in [("achieved", DEFF_ACHIEVED), ("planning", DEFF_PLANNING)]:
    print(f"  reports required under the {lab} design effect  "
          f"{int(np.ceil(n_ind*deff))}")
record("A77", p=p_hat, hw_achieved=float(z*np.sqrt(p_hat*(1-p_hat)/(len(pw)/DEFF_ACHIEVED))),
       hw_planning=float(z*np.sqrt(p_hat*(1-p_hat)/(len(pw)/DEFF_PLANNING))),
       n_independent=int(np.ceil(n_ind)),
       n_reports_planning=int(np.ceil(n_ind*DEFF_PLANNING)))

season positivity  18.3099%
achieved  design effect 1.9574 -> effective sample 2415.9; half-width plus or minus 0.01542 (1.54 percentage points)
planning  design effect 3.2947 -> effective sample 1435.3; half-width plus or minus 0.02001 (2.00 percentage points)

independent observations for a 5-percentage-point margin  230
  reports required under the achieved design effect  450
  reports required under the planning design effect  758


### Cell 72 — A78: Weeks required to establish the observed concordance

**Pseudocode.**

1. Under Fisher's z transformation the standard error of a transformed correlation is
   one over the square root of the sample size minus three.
2. Invert that at 80% power and a two-sided 5% level to state the number of weeks a
   season would need to establish a correlation of the observed size.
3. Report the requirement for the early-warning coefficient and for the full-season
   coefficient, since they differ in size and therefore in the season length they need.


In [72]:
z_a, z_b = stats.norm.ppf(0.975), stats.norm.ppf(0.80)
rows = []
for lab, r in [("early-warning window", stats.pearsonr(np.asarray(PART_SM)[EARLY],
                                                       np.asarray(SENT_SM)[EARLY])[0]),
               ("full season", stats.pearsonr(np.asarray(PART_SM), np.asarray(SENT_SM))[0]),
               ("pre-epidemic phase", stats.pearsonr(np.asarray(PART_SM)[PRE],
                                                     np.asarray(SENT_SM)[PRE])[0])]:
    zr = np.arctanh(r)
    n_req = ((z_a + z_b) / zr) ** 2 + 3
    rows.append(dict(coefficient=lab, r=round(r, 5), fisher_z=round(zr, 5),
                     weeks_required=int(np.ceil(n_req))))
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nthe season observed here is {N_WEEKS} weeks long")
record("A78", weeks_ew=rows[0]["weeks_required"], weeks_season=rows[1]["weeks_required"])

         coefficient       r  fisher_z  weeks_required
early-warning window 0.64196   0.76150              17
         full season 0.53804   0.60140              25
  pre-epidemic phase 0.88609   1.40342               7

the season observed here is 33 weeks long


### Cell 73 — A79: Events required for smaller true vaccine effects

**Pseudocode.**

1. Invert Schoenfeld's formula at 80% power to state how many events a study needs to
   detect a stated true effectiveness, at the exposed person-time share achieved here.
2. Report the requirement at several true effects and place the achieved A-cluster
   event count against them, which is the statement of what this season could and could
   not have detected.


In [73]:
Z = stats.norm.ppf(0.975) + stats.norm.ppf(0.80)
p_exp = PROTECTED_SHARE
n_A = int((episodes.cluster_letter == "A").sum())
rows = []
for ve in (0.50, 0.40, 0.30, 0.20):
    b = abs(np.log(1 - ve))
    ev = (Z / b) ** 2 / (p_exp * (1 - p_exp))
    rows.append(dict(true_VE_pct=int(100*ve), events_required=int(np.ceil(ev)),
                     achieved_A_cluster_events=n_A,
                     detectable=("yes" if np.ceil(ev) <= n_A else "no")))
print(f"exposed person-time share {p_exp:.4f}; A-cluster events achieved {n_A}")
print(pd.DataFrame(rows).to_string(index=False))
record("A79", requirements={r["true_VE_pct"]: r["events_required"] for r in rows},
       n_A=n_A)

exposed person-time share 0.3411; A-cluster events achieved 91
 true_VE_pct  events_required  achieved_A_cluster_events detectable
          50               73                         91        yes
          40              134                         91         no
          30              275                         91         no
          20              702                         91         no


### Cell 74 — A80: Reliability required to observe higher correlations

**Pseudocode.**

1. The observable correlation between two error-laden series is bounded by the
   geometric mean of their reliabilities.
2. Hold the sentinel reliability fixed and invert that relation to find the
   participatory reliability a target correlation would require.
3. Report the requirement for several targets beside the reliability actually achieved,
   which states how much measurement improvement each target would take.


In [74]:
rows = []
for target in (0.60, 0.70, 0.80, 0.90):
    need = target ** 2 / RELIABILITY_SENT
    rows.append(dict(target_correlation=target, participatory_reliability_required=round(need, 4),
                     achieved=round(RELIABILITY_PART, 4),
                     attainable=("yes" if need <= 1 else "no — exceeds 1")))
print(f"sentinel reliability held at {RELIABILITY_SENT:.5f}; participatory reliability "
      f"achieved {RELIABILITY_PART:.5f}")
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nthe current ceiling on the observable correlation is {CEILING:.5f}")
record("A80", required={r["target_correlation"]: r["participatory_reliability_required"]
                        for r in rows}, achieved=float(RELIABILITY_PART))

sentinel reliability held at 0.96197; participatory reliability achieved 0.29331
 target_correlation  participatory_reliability_required  achieved attainable
                0.6                              0.3742    0.2933        yes
                0.7                              0.5094    0.2933        yes
                0.8                              0.6653    0.2933        yes
                0.9                              0.8420    0.2933        yes

the current ceiling on the observable correlation is 0.53118
